In [1]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime -q

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from kaggle_secrets import UserSecretsClient

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 87.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


In [2]:
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("IBM_QUANTUM_TOKEN")

service = QiskitRuntimeService(
    token=token,
    instance="open-instance",  # same as in setup_ibm_backend
)

qiskit_runtime_service._discover_account:WARNING:2026-05-12 06:24:17,589: Loading account with the given token. A saved account will not be used.


In [3]:
def kl_divergence(p, q, epsilon=1e-10):
    """KL(P || Q) for discrete distributions p, q."""
    p = np.array(p, dtype=float) + epsilon
    q = np.array(q, dtype=float) + epsilon
    p = p / np.sum(p)
    q = q / np.sum(q)
    return float(np.sum(p * np.log(p / q)))


def counts_to_probs(counts, n_qubits):
    """Convert Qiskit counts dict to length-2^n probability vector."""
    total = sum(counts.values())
    n_states = 2 ** n_qubits
    probs = np.zeros(n_states, dtype=float)

    for bitstring, count in counts.items():
        idx = int(bitstring, 2)
        probs[idx] = count / total

    return probs

In [4]:
# Cell 2: noise model + simulators

def make_noise_model(error_rate_1q=0.02, factor_2q=2.0):
    """Simple depolarizing noise model."""
    noise_model = NoiseModel()
    err_1q = depolarizing_error(error_rate_1q, 1)
    err_2q = depolarizing_error(error_rate_1q * factor_2q, 2)

    noise_model.add_all_qubit_quantum_error(err_1q, ['h', 'x', 'u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(err_2q, ['cx'])
    return noise_model


def make_simulators(error_rate_1q=0.02, factor_2q=2.0):
    """Return (ideal_sim, noisy_sim, noise_model)."""
    noise_model = make_noise_model(error_rate_1q, factor_2q)
    sim_ideal = AerSimulator()
    sim_noisy = AerSimulator(noise_model=noise_model)
    return sim_ideal, sim_noisy, noise_model


# Build once and reuse
simulator_ideal, simulator_noisy, noise_model = make_simulators()
shots = 8192

print("Simulators ready.")

Simulators ready.


In [5]:




# Cell 3: simulation driver (no hardware)

def simulate_bug_detection(
    name,
    qc_correct,
    qc_buggy,
    n_qubits,
    sim_ideal,
    sim_noisy,
    shots=8192,
    detection_ratio_threshold=2.0,
):
    """
    Run ideal + noisy + buggy-noisy for a given circuit pair and return metrics.
    """
    # Ideal reference distribution
    job_ideal = sim_ideal.run(transpile(qc_correct, sim_ideal), shots=shots)
    counts_ideal = job_ideal.result().get_counts()
    probs_ideal = counts_to_probs(counts_ideal, n_qubits)

    # Hardware-like noise (correct circuit)
    job_noisy = sim_noisy.run(transpile(qc_correct, sim_noisy), shots=shots)
    counts_noisy = job_noisy.result().get_counts()
    probs_noisy = counts_to_probs(counts_noisy, n_qubits)

    # Buggy circuit under same noise model
    job_buggy = sim_noisy.run(transpile(qc_buggy, sim_noisy), shots=shots)
    counts_buggy = job_buggy.result().get_counts()
    probs_buggy = counts_to_probs(counts_buggy, n_qubits)

    # KL metrics
    kl_noise = kl_divergence(probs_noisy, probs_ideal)
    kl_bug = kl_divergence(probs_buggy, probs_ideal)
    ratio = kl_bug / kl_noise if kl_noise > 0 else np.inf
    threshold = kl_noise * 3.0

    min_kl = 1e-3      # ignore cases where both are ~0
    min_delta = 1e-3   # require a minimum absolute gap
    
    if kl_noise < min_kl and kl_bug < min_kl:
        verdict = "INCONCLUSIVE"
    elif abs(kl_bug - kl_noise) < min_delta:
        verdict = "INCONCLUSIVE"
    elif ratio > detection_ratio_threshold:
        verdict = "BUG DETECTED"
    else:
        verdict = "INCONCLUSIVE"

    print(f"\n{name.upper()} – SIMULATION")
    print("-" * 60)
    print(f"Qubits             : {n_qubits}")
    print(f"Shots              : {shots}")
    print(f"KL (noise vs ideal): {kl_noise:.4f}")
    print(f"KL (bug vs ideal)  : {kl_bug:.4f}")
    print(f"Detection ratio    : {ratio:.2f}x")
    print(f"Adaptive threshold : {threshold:.2f}")
    print(f"Verdict (>{detection_ratio_threshold:.1f}x): {verdict}")

    result_dict = {
        "name": name,
        "n_qubits": n_qubits,
        "shots": shots,
        "kl_noise": kl_noise,
        "kl_bug": kl_bug,
        "ratio": ratio,
        "threshold": threshold,
        "verdict": verdict,
        "probs_ideal": probs_ideal,
        "probs_noisy": probs_noisy,
        "probs_buggy": probs_buggy,
        "counts_ideal": counts_ideal,
        "counts_noisy": counts_noisy,
        "counts_buggy": counts_buggy,
    }
    return result_dict

In [6]:
def print_probability_distribution(probs, nqubits, title, topk=None, minprob=0.0, sortby='bitstring'):
    """
    Pretty-print a probability distribution as bitstring - probability.
    """
    dist = []
    for idx, p in enumerate(probs):
        if p < minprob:
            continue
        bitstring = format(idx, f'0{nqubits}b')
        dist.append((bitstring, p))
    
    if sortby == 'prob':
        dist.sort(key=lambda x: x[1], reverse=True)
    
    if topk is not None:
        dist = dist[:topk]
    
    if title:
        print(f"{title}")
        print("-" * 30)
    
    for bitstring, p in dist:
        print(f"{bitstring}: {p:.4f}")


In [7]:
def show_all_distributions(name, sim_result, hw_result=None, top_k=None, n_qubits=None):
    """
    Print probability distributions for:
      - Ideal simulation
      - Noisy correct (simulation)
      - Noisy buggy (simulation)
      - Optional: HW correct / HW buggy
    in a single aligned table in the terminal.
    """

    # ---- helper to find a key under multiple possible names ----
    def _get(d, *keys):
        for k in keys:
            if k in d:
                return d[k]
        raise KeyError(f"None of {keys} found in dict with keys {list(d.keys())}")

    # ---- determine n_qubits ----
    if n_qubits is None:
        if "nqubits" in sim_result:
            n_qubits = sim_result["nqubits"]
        elif "n_qubits" in sim_result:
            n_qubits = sim_result["n_qubits"]
        else:
            raise ValueError(
                "Please pass n_qubits explicitly to show_all_distributions()."
            )

    # ---- pull probability arrays from sim_result / hw_result ----
    # These names cover both your original notebook and the Phase_Kickback cell
    probs_ideal  = _get(sim_result, "probsideal",  "probs_ideal")
    probs_noisy  = _get(sim_result, "probsnoisy",  "probs_noisy")
    probs_buggy  = _get(sim_result, "probsbuggy",  "probs_buggy")

    categories = ["Ideal Sim", "Noisy Correct", "Noisy Buggy"]
    probs_list = [probs_ideal,  probs_noisy,      probs_buggy]

    if hw_result is not None:
        probs_hw_correct = _get(
            hw_result,
            "probshwcorrect", "probs_hw_correct", "probs_correct_hw"
        )
        probs_hw_buggy = _get(
            hw_result,
            "probshwbuggy", "probs_hw_buggy", "probs_buggy_hw"
        )
        categories += ["HW Correct", "HW Buggy"]
        probs_list += [probs_hw_correct, probs_hw_buggy]

    # ---- build table of top states across all categories ----
    min_prob = 1e-6
    all_states = set()
    top_dist = {}

    for cat, probs in zip(categories, probs_list):
        # list of (bitstring, prob) filtered and sorted
        dist = [
            (format(idx, f"0{n_qubits}b"), float(p))
            for idx, p in enumerate(probs)
            if p >= min_prob
        ]
        dist.sort(key=lambda x: x[1], reverse=True)
        if top_k is not None:
            dist = dist[:top_k]
        top_dist[cat] = {b: p for b, p in dist}
        all_states.update(top_dist[cat].keys())

    # sort states by probability in the ideal distribution (fallback 0)
    def ideal_prob(state):
        return top_dist["Ideal Sim"].get(state, 0.0)

    ordered_states = sorted(all_states, key=ideal_prob, reverse=True)
    if top_k is not None:
        ordered_states = ordered_states[:top_k]

    # ---- print table ----
    print(f"\n{name} – Probability Distributions")
    print("=" * 80)

    # header
    header = "State".ljust(8)
    for cat in categories:
        header += cat.ljust(16)
    print(header)
    print("-" * 80)

    # rows
    for state in ordered_states:
        row = state.ljust(8)
        for cat in categories:
            p = top_dist[cat].get(state, 0.0)
            row += f"{p:.4f}".ljust(16)
        print(row)

    print("-" * 80)
    print()

In [8]:
# Cell 4: IBM backend setup (hardware part is optional)

def setup_ibm_backend(min_num_qubits=3, instance="open-instance"):
    """Return (backend, pass_manager) for the least-busy device with >= min_num_qubits."""
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("IBM_QUANTUM_TOKEN")

    service = QiskitRuntimeService(token=token, instance=instance)

    backend = service.least_busy(
        simulator=False,
        operational=True,
        min_num_qubits=min_num_qubits,
    )

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=1,
        translation_method="translator",
    )

    print(f"\nSelected backend: {backend.name}")
    print(f"Qubits          : {backend.num_qubits}")
    print(f"Pending jobs    : {backend.status().pending_jobs}")
    return backend, pm


backend, pm = setup_ibm_backend(min_num_qubits=3)

qiskit_runtime_service._discover_account:WARNING:2026-05-12 06:24:19,236: Loading account with the given token. A saved account will not be used.



Selected backend: ibm_fez
Qubits          : 156
Pending jobs    : 0


In [9]:
def print_remaining_time(label=""):
    usage = service.usage()  # monthly / period usage for this instance
    used = float(usage.get("usage_consumed_seconds", 0.0))
    remaining = usage.get("usage_remaining_seconds")

    # Fallback if your client version doesn’t yet expose usage_remaining_seconds
    if remaining is None:
        limit = usage.get("usage_limit_seconds") or usage.get("usage_allocation_seconds")
        if limit is not None:
            remaining = float(limit) - used

    if remaining is None:
        print(f"[{label}] Used: {used:.2f} s (remaining: unknown)")
    else:
        print(f"[{label}] Remaining: {remaining:.2f} s | Used: {used:.2f} s")


In [10]:
def sampler_result_to_counts(result_entry):
    """
    Convert a single SamplerV2 pub_result entry to a dict of bitstring->count.
    """
    # Method 1: classical register 'c'
    try:
        return result_entry.data.c.get_counts()
    except AttributeError:
        pass

    # Method 2: first classical register name
    try:
        reg_name = list(result_entry.data.keys())[0]
        return result_entry.data[reg_name].get_counts()
    except Exception:
        pass

    # Method 3: build from bitstrings + counts if available
    counts = {}
    try:
        bitstrings = result_entry.data.get_bitstrings()
        freq = result_entry.data.get_counts()
        for b, c in zip(bitstrings, freq):
            counts[b] = c
    except Exception:
        raise RuntimeError("Cannot extract counts from SamplerV2 result entry.")

    return counts


def _get_usage_seconds():
    """Return (used_seconds, remaining_seconds) for this instance."""
    usage = service.usage()
    used = float(usage.get("usage_consumed_seconds", 0.0))
    remaining = usage.get("usage_remaining_seconds")

    if remaining is None:
        limit = usage.get("usage_limit_seconds") or usage.get("usage_allocation_seconds")
        remaining = float(limit) - used if limit is not None else None

    return used, remaining



def hardware_bug_detection(
    name,
    qc_correct,
    qc_buggy,
    n_qubits,
    backend,
    pass_manager,
    probs_ideal,
    shots=8192,
    ratio_threshold=1.5,
):
    """
    Run correct+buggy circuits on IBM hardware, compare to ideal probs, and compute KLs.
    Also print per-job usage (seconds this call took) and new remaining total.
    """
    # --- usage snapshot before ---
    used_before, rem_before = _get_usage_seconds()

    # Transpile to backend
    isa_correct = pass_manager.run(qc_correct)
    isa_buggy = pass_manager.run(qc_buggy)

    sampler = Sampler(mode=backend)
    job = sampler.run(pubs=[isa_correct, isa_buggy], shots=shots)
    result = job.result()

    counts_correct = sampler_result_to_counts(result[0])
    counts_buggy = sampler_result_to_counts(result[1])

    probs_correct = counts_to_probs(counts_correct, n_qubits)
    probs_buggy = counts_to_probs(counts_buggy, n_qubits)

    kl_noise = kl_divergence(probs_correct, probs_ideal)
    kl_bug = kl_divergence(probs_buggy, probs_ideal)
    ratio = kl_bug / kl_noise if kl_noise > 0 else np.inf

    min_kl = 1e-3
    min_delta = 1e-3
    
    if kl_noise < min_kl and kl_bug < min_kl:
        verdict = "INCONCLUSIVE"
    elif abs(kl_bug - kl_noise) < min_delta:
        verdict = "INCONCLUSIVE"
    elif ratio > ratio_threshold:
        verdict = "BUG DETECTED"
    else:
        verdict = "INCONCLUSIVE"

    # --- usage snapshot after ---
    used_after, rem_after = _get_usage_seconds()
    job_seconds = used_after - used_before

    print(f"\n{name.upper()} – REAL HARDWARE ({backend.name})")
    print("-" * 60)
    print(f"Qubits : {n_qubits}")
    print(f"Shots  : {shots}")
    print(f"KL (noise vs ideal): {kl_noise:.4f}")
    print(f"KL (bug vs ideal) : {kl_bug:.4f}")
    print(f"Detection ratio   : {ratio:.2f}x")
    print(f"Threshold         : {ratio_threshold:.2f}x")
    print(f"Verdict           : {verdict}")

    # New usage information
    print(f"Job usage (this call): {job_seconds:.2f} s")
    if rem_after is not None:
        print(f"Remaining total: {rem_after:.2f} s | Used total: {used_after:.2f} s")
    else:
        print(f"Used total: {used_after:.2f} s (remaining: unknown)")

    return {
        "name": name,
        "backend": backend.name,
        "n_qubits": n_qubits,
        "shots": shots,
        "kl_noise": kl_noise,
        "kl_bug": kl_bug,
        "ratio": ratio,
        "verdict": verdict,
        "probs_hw_correct": probs_correct,
        "probs_hw_buggy": probs_buggy,
        "counts_hw_correct": counts_correct,
        "counts_hw_buggy": counts_buggy,
        "job_seconds": job_seconds,
        "used_total_after": used_after,
        "remaining_total_after": rem_after,
    }

In [12]:
# GHZ circuits (3 qubits)
n_qubits = 3
# Correct GHZ |000> → (|000> + |111>)/√2
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_correct.measure([0, 1, 2], [0, 1, 2])

# Buggy GHZ (intentional error, e.g. missing last CX)
ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
# missing cx(1, 2) on purpose
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())

ghz_sim = simulate_bug_detection(
    name="GHZ_Test",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ_Test",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "GHZ_Test",
    sim_result=ghz_sim,
    hw_result=ghz_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
      ┌─┐ └───┘ ║ └╥┘
q_2: ─┤M├───────╫──╫─
      └╥┘       ║  ║ 
c: 3/══╩════════╩══╩═
       2        0  1 

GHZ_TEST – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8135
KL (bug vs ideal)  : 11.2243
Detection ratio    : 13.80x
Adaptive threshold : 2.44
Verdict (>2.0x): BUG DETECTED

GHZ_TEST – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 3
Shots  : 8192
KL (noise vs ideal): 3.2846
KL (bug vs ideal) : 11.4679
Detection ratio   : 3.49x
Threshold 

In [15]:
# Bell-state circuits (2 qubits) + mandatory printing

# Correct Bell |Φ+> = (|00> + |11>)/√2
bell_correct = QuantumCircuit(2, 2)
bell_correct.h(0)
bell_correct.cx(0, 1)
bell_correct.measure([0, 1], [0, 1])

# Buggy Bell (intentional error: wrong CX direction)
bell_buggy = QuantumCircuit(2, 2)
bell_buggy.h(0)
bell_buggy.x(0)  # bug: swapped control/target
bell_buggy.measure([0, 1], [0, 1])

# ----- MANDATORY: print circuits -----
print("Correct Bell circuit:")
print(bell_correct.draw())

print("\nBuggy Bell circuit:")
print(bell_buggy.draw())
# -------------------------------------

# Synchronized template code
n_qubits = 2

bell_sim = simulate_bug_detection(
    name="Bell_Test",
    qc_correct=bell_correct,
    qc_buggy=bell_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

bell_hw = hardware_bug_detection(
    name="Bell_Test",
    qc_correct=bell_correct,
    qc_buggy=bell_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=bell_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Bell_Test",
    sim_result=bell_sim,
    hw_result=bell_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Bell circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Buggy Bell circuit:
     ┌───┐┌───┐┌─┐
q_0: ┤ H ├┤ X ├┤M├
     └┬─┬┘└───┘└╥┘
q_1: ─┤M├───────╫─
      └╥┘       ║ 
c: 2/══╩════════╩═
       1        0 

BELL_TEST – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.3808
KL (bug vs ideal)  : 11.2426
Detection ratio    : 29.52x
Adaptive threshold : 1.14
Verdict (>2.0x): BUG DETECTED

BELL_TEST – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 2.6182
KL (bug vs ideal) : 10.3674
Detection ratio   : 3.96x
Threshold         : 1.50x
Verdict           : BUG DETECTED
Job usage (this call): 6.00 s
Remaining total: 269.00 s | Used total: 331.00 s

Bell_Test – Probability Distributions
State   Id

In [17]:
# GHZ circuits (3 qubits)
n_qubits = 3

# Correct GHZ |000> → (|000> + |111>)/√2
ghz_correct = QuantumCircuit(n_qubits, n_qubits)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_correct.measure([0, 1, 2], [0, 1, 2])

# Buggy GHZ (extra Z phase flip)
ghz_buggy = QuantumCircuit(n_qubits, n_qubits)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
ghz_buggy.cx(1, 2)
ghz_buggy.z(0)  # Phase flip error on qubit 0
ghz_buggy.measure([0, 1, 2], [0, 1, 2])

# ----- MANDATORY: print circuits -----
print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())
# -------------------------------------

ghz_sim = simulate_bug_detection(
    name="GHZ_Test",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ_Test",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "GHZ_Test",
    sim_result=ghz_sim,
    hw_result=ghz_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐     ┌───┐┌─┐      
q_0: ┤ H ├──■──┤ Z ├┤M├──────
     └───┘┌─┴─┐└───┘└╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

GHZ_TEST – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8928
KL (bug vs ideal)  : 0.9479
Detection ratio    : 1.06x
Adaptive threshold : 2.68
Verdict (>2.0x): INCONCLUSIVE

GHZ_TEST – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 3
Shots  : 8192
KL (noise vs ideal): 3

In [19]:
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_correct.measure([0, 1, 2], [0, 1, 2])


ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
ghz_buggy.cx(1, 2)
ghz_buggy.z(1)
ghz_buggy.barrier()
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())


# Example: GHZ – simulation + hardware + distributions
n_qubits = 3

# 1. Simulation-only bug detection
ghz_sim = simulate_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

# 2. Hardware bug detection
ghz_hw = hardware_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

# 3. Show all probability distributions
show_all_distributions(
    "GHZ",
    sim_result=ghz_sim,
    hw_result=ghz_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐                ░ ┌─┐      
q_0: ┤ H ├──■─────────────░─┤M├──────
     └───┘┌─┴─┐     ┌───┐ ░ └╥┘┌─┐   
q_1: ─────┤ X ├──■──┤ Z ├─░──╫─┤M├───
          └───┘┌─┴─┐└───┘ ░  ║ └╥┘┌─┐
q_2: ──────────┤ X ├──────░──╫──╫─┤M├
               └───┘      ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

GHZ – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8358
KL (bug vs ideal)  : 1.0284
Detection ratio    : 1.23x
Adaptive threshold : 2.51
Verdict (>2.0x): INCONCLUSIVE

GHZ – REAL HARDWARE (ibm_torino)
----------------------------------------------

In [13]:
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_correct.measure([0, 1, 2], [0, 1, 2])


ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
ghz_buggy.cx(1, 2)
ghz_buggy.z(2)
ghz_buggy.barrier()
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())


# Example: GHZ
ghz_sim = simulate_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

# Now print ALL probability distributions for GHZ
show_all_distributions("GHZ", sim_result=ghz_sim, hw_result=ghz_hw, top_k=8)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐                ░ ┌─┐      
q_0: ┤ H ├──■─────────────░─┤M├──────
     └───┘┌─┴─┐           ░ └╥┘┌─┐   
q_1: ─────┤ X ├──■────────░──╫─┤M├───
          └───┘┌─┴─┐┌───┐ ░  ║ └╥┘┌─┐
q_2: ──────────┤ X ├┤ Z ├─░──╫──╫─┤M├
               └───┘└───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

GHZ – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8839
KL (bug vs ideal)  : 1.1621
Detection ratio    : 1.31x
Adaptive threshold : 2.65
Verdict (>2.0x): NO BUG

GHZ – REAL HARDWARE (ibm_fez)
-------------------------------------------------------

In [14]:
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_buggy.barrier()
ghz_correct.measure([0, 1, 2], [0, 1, 2])


ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.z(0)
ghz_buggy.cx(0, 1)
ghz_buggy.cx(1, 2)
ghz_buggy.barrier()
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())


# Example: GHZ
ghz_sim = simulate_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

# Now print ALL probability distributions for GHZ
show_all_distributions("GHZ", sim_result=ghz_sim, hw_result=ghz_hw, top_k=8)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐┌───┐           ░ ┌─┐      
q_0: ┤ H ├┤ Z ├──■────────░─┤M├──────
     └───┘└───┘┌─┴─┐      ░ └╥┘┌─┐   
q_1: ──────────┤ X ├──■───░──╫─┤M├───
               └───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ───────────────┤ X ├─░──╫──╫─┤M├
                    └───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

GHZ – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8685
KL (bug vs ideal)  : 0.8572
Detection ratio    : 0.99x
Adaptive threshold : 2.61
Verdict (>2.0x): NO BUG

GHZ – REAL HARDWARE (ibm_fez)
-------------------------------------------------------

In [15]:
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_buggy.barrier()
ghz_correct.measure([0, 1, 2], [0, 1, 2])


ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
ghz_buggy.z(1)
ghz_buggy.cx(1, 2)
ghz_buggy.barrier()
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())


# Example: GHZ
ghz_sim = simulate_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

# Now print ALL probability distributions for GHZ
show_all_distributions("GHZ", sim_result=ghz_sim, hw_result=ghz_hw, top_k=8)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐                ░ ┌─┐      
q_0: ┤ H ├──■─────────────░─┤M├──────
     └───┘┌─┴─┐┌───┐      ░ └╥┘┌─┐   
q_1: ─────┤ X ├┤ Z ├──■───░──╫─┤M├───
          └───┘└───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ───────────────┤ X ├─░──╫──╫─┤M├
                    └───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

GHZ – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8935
KL (bug vs ideal)  : 0.9977
Detection ratio    : 1.12x
Adaptive threshold : 2.68
Verdict (>2.0x): NO BUG

GHZ – REAL HARDWARE (ibm_fez)
-------------------------------------------------------

In [16]:
ghz_correct = QuantumCircuit(3, 3)
ghz_correct.h(0)
ghz_correct.cx(0, 1)
ghz_correct.cx(1, 2)
ghz_buggy.barrier()
ghz_correct.measure([0, 1, 2], [0, 1, 2])


ghz_buggy = QuantumCircuit(3, 3)
ghz_buggy.h(0)
ghz_buggy.cx(0, 1)
ghz_buggy.cx(1, 2)
ghz_buggy.barrier()
ghz_buggy.measure([0, 1, 2], [0, 1, 2])


print("Correct GHZ circuit:")
print(ghz_correct.draw())

print("\nBuggy GHZ circuit:")
print(ghz_buggy.draw())


# Example: GHZ
ghz_sim = simulate_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_hw = hardware_bug_detection(
    name="GHZ",
    qc_correct=ghz_correct,
    qc_buggy=ghz_buggy,
    n_qubits=3,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_sim["probs_ideal"],
    shots=shots,
)

# Now print ALL probability distributions for GHZ
show_all_distributions("GHZ", sim_result=ghz_sim, hw_result=ghz_hw, top_k=8)

Correct GHZ circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

Buggy GHZ circuit:
     ┌───┐           ░ ┌─┐      
q_0: ┤ H ├──■────────░─┤M├──────
     └───┘┌─┴─┐      ░ └╥┘┌─┐   
q_1: ─────┤ X ├──■───░──╫─┤M├───
          └───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ──────────┤ X ├─░──╫──╫─┤M├
               └───┘ ░  ║  ║ └╥┘
c: 3/═══════════════════╩══╩══╩═
                        0  1  2 

GHZ – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.7373
KL (bug vs ideal)  : 0.7634
Detection ratio    : 1.04x
Adaptive threshold : 2.21
Verdict (>2.0x): NO BUG

GHZ – REAL HARDWARE (ibm_fez)
------------------------------------------------------------
Qubits             : 3
Shots           

In [24]:
# Number of qubits in the circuit (q0 = control, q1 = ancilla)
n_qubits = 2

# Correct phase kickback circuit (no phase flip on control)
pk_correct = QuantumCircuit(n_qubits, 1)  # 2 qubits, 1 classical bit
pk_correct.h(0)       # put control into |+>_c
pk_correct.h(1)       # put ancilla into |+>_a
pk_correct.cz(0, 1)   # controlled-Z: phase kickback from control to ancilla
pk_correct.h(1)       # rotate ancilla back to Z basis (X-basis measurement)
pk_correct.measure(1, 0)  # measure ancilla into c[0]

# Buggy phase kickback circuit (extra Z phase flip on control)
pk_buggy = QuantumCircuit(n_qubits, 1)
pk_buggy.h(0)         # |+>_c
pk_buggy.z(0)         # phase flip error on control
pk_buggy.h(1)         # |+>_a
pk_buggy.cz(0, 1)     # kickback now includes the extra -1 phase
pk_buggy.h(1)
pk_buggy.measure(1, 0)

print("Correct Phase Kickback circuit:")
print(pk_correct.draw())

print("\nBuggy Phase Kickback circuit (Z phase flip on control):")
print(pk_buggy.draw())

pk_sim = simulate_bug_detection(
    name="Phase_Kickback",
    qc_correct=pk_correct,
    qc_buggy=pk_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

pk_hw = hardware_bug_detection(
    name="Phase_Kickback",
    qc_correct=pk_correct,
    qc_buggy=pk_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=pk_sim["probs_ideal"],  # or pk_sim["probsideal"], depending on your dict
    shots=shots,
)

show_all_distributions(
    "Phase_Kickback",
    sim_result=pk_sim,
    hw_result=pk_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Phase Kickback circuit:
     ┌───┐           
q_0: ┤ H ├─■─────────
     ├───┤ │ ┌───┐┌─┐
q_1: ┤ H ├─■─┤ H ├┤M├
     └───┘   └───┘└╥┘
c: 1/══════════════╩═
                   0 

Buggy Phase Kickback circuit (Z phase flip on control):
     ┌───┐┌───┐           
q_0: ┤ H ├┤ Z ├─■─────────
     ├───┤└───┘ │ ┌───┐┌─┐
q_1: ┤ H ├──────■─┤ H ├┤M├
     └───┘        └───┘└╥┘
c: 1/═══════════════════╩═
                        0 

PHASE_KICKBACK – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0000
KL (bug vs ideal)  : 0.0000
Detection ratio    : 1.00x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

PHASE_KICKBACK – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 0.0007
KL (bug vs ideal) : 0.0006
Detection ratio   : 0.95x
Threshold         : 1.50x
Verdict           : INCONCLUSIVE
Job usage (this cal

In [28]:
# Number of qubits in the circuit (q0 = control, q1 = target)
n_qubits = 2

# Correct entangled phase-flip circuit (no extra Z on control)
pf_correct = QuantumCircuit(n_qubits, 2)  # 2 qubits, 2 classical bits
pf_correct.h(0)          # put q0 into |+> (start of Bell-like state)
pf_correct.cx(0, 1)      # entangle q0 and q1 -> |Φ+> ≈ (|00> + |11>)/√2
pf_correct.h(0)          # interference layer on control
pf_correct.h(1)          # interference layer on target
pf_correct.measure([0, 1], [0, 1])  # measure both qubits into c[0], c[1]

# Buggy entangled phase-flip circuit (extra Z phase flip on control)
pf_buggy = QuantumCircuit(n_qubits, 2)
pf_buggy.h(0)            # same start: create superposition on q0
pf_buggy.cx(0, 1)        # same entangling step
pf_buggy.z(0)            # extra Z phase flip on control (changes Bell phase)
pf_buggy.h(0)            # same interference layer
pf_buggy.h(1)
pf_buggy.measure([0, 1], [0, 1])

print("Correct entangled Phase-Flip circuit:")
print(pf_correct.draw())

print("\nBuggy entangled Phase-Flip circuit (extra Z on control q0):")
print(pf_buggy.draw())

pf_sim = simulate_bug_detection(
    name="Entangled_Phase_Flip",
    qc_correct=pf_correct,
    qc_buggy=pf_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

pf_hw = hardware_bug_detection(
    name="Entangled_Phase_Flip",
    qc_correct=pf_correct,
    qc_buggy=pf_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=pf_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Entangled_Phase_Flip",
    sim_result=pf_sim,
    hw_result=pf_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct entangled Phase-Flip circuit:
     ┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├──■──┤ H ├┤M├───
     └───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 

Buggy entangled Phase-Flip circuit (extra Z on control q0):
     ┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├┤ H ├┤M├
     └───┘┌─┴─┐├───┤└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ H ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 

ENTANGLED_PHASE_FLIP – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.8107
KL (bug vs ideal)  : 21.0380
Detection ratio    : 25.95x
Adaptive threshold : 2.43
Verdict (>2.0x): BUG DETECTED

ENTANGLED_PHASE_FLIP – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 2.8948
KL (bug vs ideal) : 18.6245
Detection ratio   : 6.43x
Thre

In [29]:
# Number of qubits in the circuit (x0, x1 = input qubits)
n_qubits = 2

# Correct Deutsch–Jozsa–style phase oracle circuit (no extra phase flip)
dj_correct = QuantumCircuit(n_qubits, 2)  # 2 qubits, 2 classical bits
dj_correct.h([0, 1])           # put both inputs into uniform superposition
dj_correct.cz(0, 1)            # oracle: flip phase only on |11>
dj_correct.h([0, 1])           # interference to read oracle type
dj_correct.measure([0, 1], [0, 1])

# Buggy Deutsch–Jozsa–style circuit (extra Z phase flip on qubit 0)
dj_buggy = QuantumCircuit(n_qubits, 2)
dj_buggy.h([0, 1])             # same superposition
dj_buggy.cz(0, 1)              # same oracle core
dj_buggy.z(0)                  # BUG: extra phase flip on states with q0 = 1
dj_buggy.h([0, 1])
dj_buggy.barrier()
dj_buggy.measure([0, 1], [0, 1])

print("Correct Deutsch–Jozsa-style oracle circuit:")
print(dj_correct.draw())

print("\nBuggy Deutsch–Jozsa-style oracle circuit (extra Z on q0):")
print(dj_buggy.draw())

dj_sim = simulate_bug_detection(
    name="Deutsch_Jozsa_Oracle",
    qc_correct=dj_correct,
    qc_buggy=dj_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

dj_hw = hardware_bug_detection(
    name="Deutsch_Jozsa_Oracle",
    qc_correct=dj_correct,
    qc_buggy=dj_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=dj_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Deutsch_Jozsa_Oracle",
    sim_result=dj_sim,
    hw_result=dj_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Deutsch–Jozsa-style oracle circuit:
     ┌───┐   ┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤M├───
     ├───┤ │ ├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├─╫─┤M├
     └───┘   └───┘ ║ └╥┘
c: 2/══════════════╩══╩═
                   0  1 

Buggy Deutsch–Jozsa-style oracle circuit (extra Z on q0):
     ┌───┐   ┌───┐┌───┐┌─┐
q_0: ┤ H ├─■─┤ Z ├┤ H ├┤M├
     ├───┤ │ ├───┤└┬─┬┘└╥┘
q_1: ┤ H ├─■─┤ H ├─┤M├──╫─
     └───┘   └───┘ └╥┘  ║ 
c: 2/═══════════════╩═══╩═
                    1   0 

DEUTSCH_JOZSA_ORACLE – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0006
KL (bug vs ideal)  : 0.0003
Detection ratio    : 0.55x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

DEUTSCH_JOZSA_ORACLE – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 0.0026
KL (bug vs ideal) : 0.0020
Detection ratio   : 0.78x
Threshold         : 1.50x
Verdi

In [11]:
# Number of qubits in the circuit (x0, x1 = input qubits)
n_qubits = 2

# Correct Deutsch–Jozsa–style phase oracle circuit (no extra phase flip)
dj_correct = QuantumCircuit(n_qubits, 2)  # 2 qubits, 2 classical bits
dj_correct.h([0, 1])           # put both inputs into uniform superposition
dj_correct.cz(0, 1)            # oracle: flip phase only on |11>
dj_correct.h([0, 1])           # interference to read oracle type
dj_correct.measure([0, 1], [0, 1])

# Buggy Deutsch–Jozsa–style circuit (extra Z phase flip on qubit 0)
dj_buggy = QuantumCircuit(n_qubits, 2)
dj_buggy.h([0, 1])             # same superposition
dj_buggy.cz(0, 1)              # same oracle core
dj_buggy.z(1)                  # BUG: extra phase flip
dj_buggy.h([0, 1])
dj_buggy.measure([0, 1], [0, 1])

print("Correct Deutsch–Jozsa-style oracle circuit:")
print(dj_correct.draw())

print("\nBuggy Deutsch–Jozsa-style oracle circuit (extra Z on q0):")
print(dj_buggy.draw())

dj_sim = simulate_bug_detection(
    name="Deutsch_Jozsa_Oracle",
    qc_correct=dj_correct,
    qc_buggy=dj_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

dj_hw = hardware_bug_detection(
    name="Deutsch_Jozsa_Oracle",
    qc_correct=dj_correct,
    qc_buggy=dj_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=dj_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Deutsch_Jozsa_Oracle",
    sim_result=dj_sim,
    hw_result=dj_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Deutsch–Jozsa-style oracle circuit:
     ┌───┐   ┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤M├───
     ├───┤ │ ├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├─╫─┤M├
     └───┘   └───┘ ║ └╥┘
c: 2/══════════════╩══╩═
                   0  1 

Buggy Deutsch–Jozsa-style oracle circuit (extra Z on q0):
     ┌───┐   ┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├─────┤M├───
     ├───┤ │ ├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ Z ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘ ║ └╥┘
c: 2/═══════════════════╩══╩═
                        0  1 

DEUTSCH_JOZSA_ORACLE – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0005
KL (bug vs ideal)  : 0.0005
Detection ratio    : 0.93x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

DEUTSCH_JOZSA_ORACLE – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 0.0023
KL (bug vs ideal) : 0.0074
Detection ratio   : 3.18x
Threshold 

In [31]:
# Number of qubits in the circuit (q0,q1 = search register)
n_qubits = 2

# Helper: Grover diffusion operator on 2 qubits
def grover_diffusion_2q(circ, q0=0, q1=1):
    circ.h([q0, q1])
    circ.x([q0, q1])
    circ.h(q1)
    circ.cx(q0, q1)
    circ.h(q1)
    circ.x([q0, q1])
    circ.h([q0, q1])

# Correct Grover circuit (oracle marks |11> only)
gr_correct = QuantumCircuit(n_qubits, 2)
gr_correct.h([0, 1])          # uniform superposition over 00,01,10,11

# Oracle: mark |11> by a phase flip
gr_correct.cz(0, 1)           # phase flip only on |11>

# One Grover diffusion step
grover_diffusion_2q(gr_correct)

gr_correct.measure([0, 1], [0, 1])

# Buggy Grover circuit (oracle effectively marks |11> AND adds extra phase via Z on q0)
gr_buggy = QuantumCircuit(n_qubits, 2)
gr_buggy.h([0, 1])

# Same base oracle + BUG: extra Z on q0 (affects |10> and |11>)
gr_buggy.cz(0, 1)             # intended mark on |11>
gr_buggy.z(0)                 # BUG: extra phase flip on states with q0 = 1

grover_diffusion_2q(gr_buggy)

gr_buggy.measure([0, 1], [0, 1])

print("Correct 2-qubit Grover circuit (oracle marks |11>):")
print(gr_correct.draw())

print("\nBuggy 2-qubit Grover circuit (oracle + extra Z on q0):")
print(gr_buggy.draw())

gr_sim = simulate_bug_detection(
    name="Grover_Oracle",
    qc_correct=gr_correct,
    qc_buggy=gr_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

gr_hw = hardware_bug_detection(
    name="Grover_Oracle",
    qc_correct=gr_correct,
    qc_buggy=gr_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=gr_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Grover_Oracle",
    sim_result=gr_sim,
    hw_result=gr_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct 2-qubit Grover circuit (oracle marks |11>):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy 2-qubit Grover circuit (oracle + extra Z on q0):
     ┌───┐   ┌───┐┌───┐┌───┐     ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ Z ├┤ H ├┤ X ├──■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤├───┤┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

GROVER_ORACLE – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots     

In [32]:
# Number of qubits in the circuit (q0, q1 = search register)
n_qubits = 2

# Helper: Grover diffusion operator on 2 qubits
def grover_diffusion_2q(circ, q0=0, q1=1):
    circ.h([q0, q1])
    circ.x([q0, q1])
    circ.h(q1)
    circ.cx(q0, q1)
    circ.h(q1)
    circ.x([q0, q1])
    circ.h([q0, q1])

# Correct Grover circuit (oracle marks |11> only)
gr_correct = QuantumCircuit(n_qubits, 2)  # 2 qubits, 2 classical bits
gr_correct.h([0, 1])          # uniform superposition over {00,01,10,11}

# Oracle: phase flip only on |11>
gr_correct.cz(0, 1)

# One Grover diffusion step
grover_diffusion_2q(gr_correct)

gr_correct.measure([0, 1], [0, 1])

# Buggy Grover circuit (oracle + extra Z phase flip on q0)
gr_buggy = QuantumCircuit(n_qubits, 2)
gr_buggy.h([0, 1])

# Same intended oracle, but with BUG: extra phase flip on all states with q0 = 1
gr_buggy.cz(0, 1)             # intended mark on |11>
gr_buggy.z(0)                 # BUG: also flips phase of |10> and |11>

grover_diffusion_2q(gr_buggy)

gr_buggy.measure([0, 1], [0, 1])

print("Correct 2-qubit Grover circuit (oracle marks |11>):")
print(gr_correct.draw())

print("\nBuggy 2-qubit Grover circuit (oracle + extra Z on q0):")
print(gr_buggy.draw())

gr_sim = simulate_bug_detection(
    name="Grover_Oracle",
    qc_correct=gr_correct,
    qc_buggy=gr_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

gr_hw = hardware_bug_detection(
    name="Grover_Oracle",
    qc_correct=gr_correct,
    qc_buggy=gr_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=gr_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Grover_Oracle",
    sim_result=gr_sim,
    hw_result=gr_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct 2-qubit Grover circuit (oracle marks |11>):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy 2-qubit Grover circuit (oracle + extra Z on q0):
     ┌───┐   ┌───┐┌───┐┌───┐     ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ Z ├┤ H ├┤ X ├──■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤├───┤┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

GROVER_ORACLE – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots     

In [33]:
from math import pi

# Number of qubits in the circuit (q0 = control, q1 = target eigenstate)
n_qubits = 2

# Correct 1-bit phase-estimation-style circuit (no extra phase flip on target)
pe_correct = QuantumCircuit(n_qubits, 2)  # 2 qubits, 2 classical bits

# Prepare eigenstate |1> on target (q1)
pe_correct.x(1)

# Put control into |+> (superposition)
pe_correct.h(0)

# Controlled phase gate: U = S = diag(1, i) applied if control is 1
pe_correct.cp(pi/2, 0, 1)

# Inverse "QFT" on single control qubit is just H
pe_correct.h(0)

# Measure both qubits (you mainly care about c[0], the control)
pe_correct.measure([0, 1], [0, 1])

# Buggy circuit (extra Z phase flip on the target eigenstate before controlled-U)
pe_buggy = QuantumCircuit(n_qubits, 2)

pe_buggy.x(1)          # prepare |1> on target
pe_buggy.h(0)          # put control into |+>

pe_buggy.z(1)          # BUG: extra phase flip on the eigenstate
pe_buggy.cp(pi/2, 0, 1)

pe_buggy.h(0)          # same final interference
pe_buggy.measure([0, 1], [0, 1])

print("Correct Phase_Estimation_1bit circuit:")
print(pe_correct.draw())

print("\nBuggy Phase_Estimation_1bit circuit (extra Z on target q1):")
print(pe_buggy.draw())

pe_sim = simulate_bug_detection(
    name="Phase_Estimation_1bit",
    qc_correct=pe_correct,
    qc_buggy=pe_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

pe_hw = hardware_bug_detection(
    name="Phase_Estimation_1bit",
    qc_correct=pe_correct,
    qc_buggy=pe_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=pe_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "Phase_Estimation_1bit",
    sim_result=pe_sim,
    hw_result=pe_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Phase_Estimation_1bit circuit:
     ┌───┐         ┌───┐┌─┐
q_0: ┤ H ├─■───────┤ H ├┤M├
     ├───┤ │P(π/2) └┬─┬┘└╥┘
q_1: ┤ X ├─■────────┤M├──╫─
     └───┘          └╥┘  ║ 
c: 2/════════════════╩═══╩═
                     1   0 

Buggy Phase_Estimation_1bit circuit (extra Z on target q1):
     ┌───┐              ┌───┐┌─┐
q_0: ┤ H ├──────■───────┤ H ├┤M├
     ├───┤┌───┐ │P(π/2) └┬─┬┘└╥┘
q_1: ┤ X ├┤ Z ├─■────────┤M├──╫─
     └───┘└───┘          └╥┘  ║ 
c: 2/═════════════════════╩═══╩═
                          1   0 

PHASE_ESTIMATION_1BIT – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 1.2716
KL (bug vs ideal)  : 1.1504
Detection ratio    : 0.90x
Adaptive threshold : 3.81
Verdict (>2.0x): INCONCLUSIVE

PHASE_ESTIMATION_1BIT – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 1.1756
KL (bug vs ideal) : 1.

In [34]:
# Number of qubits in the circuit (q0, q1, q2 form GHZ)
n_qubits = 3

# Correct GHZ phase-interference circuit (no extra Z)
ghz_phase_correct = QuantumCircuit(n_qubits, 3)  # 3 qubits, 3 classical bits

# Prepare GHZ: (|000> + |111>)/√2
ghz_phase_correct.h(0)
ghz_phase_correct.cx(0, 1)
ghz_phase_correct.cx(1, 2)

# Global Hadamard layer to turn phase into measurement amplitudes
ghz_phase_correct.h([0, 1, 2])

ghz_phase_correct.measure([0, 1, 2], [0, 1, 2])

# Buggy GHZ phase-interference circuit (extra Z phase flip on q1 before final H's)
ghz_phase_buggy = QuantumCircuit(n_qubits, 3)

# Same GHZ preparation
ghz_phase_buggy.h(0)
ghz_phase_buggy.cx(0, 1)
ghz_phase_buggy.cx(1, 2)

ghz_phase_buggy.z(1)        # BUG: extra phase flip on the middle qubit

# Same global Hadamard layer
ghz_phase_buggy.h([0, 1, 2])

ghz_phase_buggy.measure([0, 1, 2], [0, 1, 2])

print("Correct GHZ phase-interference circuit:")
print(ghz_phase_correct.draw())

print("\nBuggy GHZ phase-interference circuit (extra Z on q1):")
print(ghz_phase_buggy.draw())

ghz_phase_sim = simulate_bug_detection(
    name="GHZ_Phase_Test",
    qc_correct=ghz_phase_correct,
    qc_buggy=ghz_phase_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ghz_phase_hw = hardware_bug_detection(
    name="GHZ_Phase_Test",
    qc_correct=ghz_phase_correct,
    qc_buggy=ghz_phase_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ghz_phase_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "GHZ_Phase_Test",
    sim_result=ghz_phase_sim,
    hw_result=ghz_phase_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct GHZ phase-interference circuit:
     ┌───┐     ┌───┐     ┌─┐      
q_0: ┤ H ├──■──┤ H ├─────┤M├──────
     └───┘┌─┴─┐└───┘┌───┐└╥┘┌─┐   
q_1: ─────┤ X ├──■──┤ H ├─╫─┤M├───
          └───┘┌─┴─┐├───┤ ║ └╥┘┌─┐
q_2: ──────────┤ X ├┤ H ├─╫──╫─┤M├
               └───┘└───┘ ║  ║ └╥┘
c: 3/═════════════════════╩══╩══╩═
                          0  1  2 

Buggy GHZ phase-interference circuit (extra Z on q1):
     ┌───┐     ┌───┐     ┌─┐        
q_0: ┤ H ├──■──┤ H ├─────┤M├────────
     └───┘┌─┴─┐└───┘┌───┐└╥┘┌───┐┌─┐
q_1: ─────┤ X ├──■──┤ Z ├─╫─┤ H ├┤M├
          └───┘┌─┴─┐├───┤ ║ └┬─┬┘└╥┘
q_2: ──────────┤ X ├┤ H ├─╫──┤M├──╫─
               └───┘└───┘ ║  └╥┘  ║ 
c: 3/═════════════════════╩═══╩═══╩═
                          0   2   1 

GHZ_PHASE_TEST – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 1.3467
KL (bug vs ideal)  : 19.6399
Detection ratio    : 14.58x
Adaptive threshold : 4.04
Verdict

In [13]:
# Number of qubits (all equivalent |+> states)
n_qubits = 3

# Correct: uniform superposition (|++++...> effectively)
uniform_correct = QuantumCircuit(n_qubits, n_qubits)
for i in range(n_qubits):
    uniform_correct.h(i)
uniform_correct.measure(range(n_qubits), range(n_qubits))

# Buggy: bit flip X on q1 (any qubit works)
uniform_buggy = QuantumCircuit(n_qubits, n_qubits)
for i in range(n_qubits):
    uniform_buggy.h(i)
uniform_buggy.x(1)  # X flips |+> to |->, but Z-measure still ~0.5/0.5 per bit
uniform_buggy.barrier()

uniform_buggy.measure(range(n_qubits), range(n_qubits))


print("Correct uniform superposition circuit:")
print(uniform_correct.draw())

print("\nBuggy uniform superposition:")
print(uniform_buggy.draw())
# Run as before
uniform_sim = simulate_bug_detection(name="UniformBitFlipTest", 
                                     qc_correct=uniform_correct, 
                                     qc_buggy=uniform_buggy, 
                                     n_qubits=n_qubits, 
                                     sim_ideal=simulator_ideal, 
                                     sim_noisy=simulator_noisy, 
                                     shots=shots)
uniform_hw = hardware_bug_detection(name="UniformBitFlipTest", 
                                    qc_correct=uniform_correct, 
                                    qc_buggy=uniform_buggy, 
                                    n_qubits=n_qubits, 
                                    backend=backend, 
                                    pass_manager=pm, 
                                    probs_ideal=uniform_sim["probs_ideal"], 
                                    shots=shots)
show_all_distributions("UniformBitFlipTest", 
                       sim_result=uniform_sim, 
                       hw_result=uniform_hw, 
                       top_k=8, 
                       n_qubits=n_qubits)

Correct uniform superposition circuit:
     ┌───┐┌─┐      
q_0: ┤ H ├┤M├──────
     ├───┤└╥┘┌─┐   
q_1: ┤ H ├─╫─┤M├───
     ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├─╫──╫─┤M├
     └───┘ ║  ║ └╥┘
c: 3/══════╩══╩══╩═
           0  1  2 

Buggy uniform superposition:
     ┌───┐      ░ ┌─┐      
q_0: ┤ H ├──────░─┤M├──────
     ├───┤┌───┐ ░ └╥┘┌─┐   
q_1: ┤ H ├┤ X ├─░──╫─┤M├───
     ├───┤└───┘ ░  ║ └╥┘┌─┐
q_2: ┤ H ├──────░──╫──╫─┤M├
     └───┘      ░  ║  ║ └╥┘
c: 3/══════════════╩══╩══╩═
                   0  1  2 

UNIFORMBITFLIPTEST – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.0008
KL (bug vs ideal)  : 0.0014
Detection ratio    : 1.82x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

UNIFORMBITFLIPTEST – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 3
Shots  : 8192
KL (noise vs ideal): 0.0043
KL (bug vs ideal) : 0.0055
Detection ratio   : 1.27x
T

In [15]:
n_qubits = 2

# Correct Hadamard Test circuit (interference on ancilla)
ht_correct = QuantumCircuit(n_qubits, 1)  # 2 qubits, 1 classical bit (ancilla measure)
ht_correct.h(0)                           # Superposition on ancilla
ht_correct.cx(0, 1)                       # Controlled-NOT (simple U = X effectively)
ht_correct.h(0)                           # Interference
ht_correct.measure(0, 0)                  # Measure ancilla

# Buggy: bit flip X on data qubit 1 before CX (symmetrizes to ~50/50)
ht_buggy = QuantumCircuit(n_qubits, 1)
ht_buggy.h(0)
ht_buggy.x(1)                             # BUG: X bit flip on data - flips basis, minimal dist change
ht_buggy.cx(0, 1)
ht_buggy.h(0)
ht_buggy.measure(0, 0)

print("Correct Hadamard Test circuit:")
print(ht_correct.draw())

print("\nBuggy Hadamard Test circuit (extra X bit flip on q1):")
print(ht_buggy.draw())

# Run in your framework
ht_sim = simulate_bug_detection(
    name="HadamardTest_BitFlip",
    qc_correct=ht_correct,
    qc_buggy=ht_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

ht_hw = hardware_bug_detection(
    name="HadamardTest_BitFlip",
    qc_correct=ht_correct,
    qc_buggy=ht_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=ht_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "HadamardTest_BitFlip",
    sim_result=ht_sim,
    hw_result=ht_hw,
    top_k=8,
    n_qubits=1,  # Effective measured qubits
)

Correct Hadamard Test circuit:
     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ H ├┤M├
     └───┘┌─┴─┐└───┘└╥┘
q_1: ─────┤ X ├──────╫─
          └───┘      ║ 
c: 1/════════════════╩═
                     0 

Buggy Hadamard Test circuit (extra X bit flip on q1):
     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ H ├┤M├
     ├───┤┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ X ├──────╫─
     └───┘└───┘      ║ 
c: 1/════════════════╩═
                     0 

HADAMARDTEST_BITFLIP – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0000
KL (bug vs ideal)  : 0.0001
Detection ratio    : 1.87x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

HADAMARDTEST_BITFLIP – REAL HARDWARE (ibm_torino)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 0.0005
KL (bug vs ideal) : 0.0005
Detection ratio   : 1.02x
Threshold         : 1.50x
Verdict           : INCONCLUSIVE
Job usage (this c

In [16]:
# Repetition Code Syndrome (separate block)

# Number of qubits (3-qubit code)
n_qubits_rep = 3

# Correct Repetition Code
rep_correct = QuantumCircuit(n_qubits_rep, 2)
rep_correct.cx(0, 1)  # Encode |0_L> = |000>
rep_correct.cx(0, 2)
rep_correct.cx(0, 1)  # Syndrome s0 = q0 XOR q1
rep_correct.cx(1, 2)  # Syndrome s1 = q1 XOR q2
rep_correct.measure([1, 2], [0, 1])

# Buggy: X bit flip on q1
rep_buggy = QuantumCircuit(n_qubits_rep, 2)
rep_buggy.cx(0, 1)
rep_buggy.cx(0, 2)
rep_buggy.x(1)        # BUG: bit flip (syndrome shifts, but noise masks)
rep_buggy.cx(0, 1)
rep_buggy.cx(1, 2)
rep_buggy.measure([1, 2], [0, 1])

print("\nCorrect Repetition Code Syndrome:")
print(rep_correct.draw())

print("\nBuggy Repetition Code Syndrome (X on q1):")
print(rep_buggy.draw())

# Run tests
rep_sim = simulate_bug_detection(
    name="RepCode_BitFlip",
    qc_correct=rep_correct,
    qc_buggy=rep_buggy,
    n_qubits=n_qubits_rep,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

rep_hw = hardware_bug_detection(
    name="RepCode_BitFlip",
    qc_correct=rep_correct,
    qc_buggy=rep_buggy,
    n_qubits=n_qubits_rep,
    backend=backend,
    pass_manager=pm,
    probs_ideal=rep_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "RepCode_BitFlip",
    sim_result=rep_sim,
    hw_result=rep_hw,
    top_k=8,
    n_qubits=2,  # Syndrome bits
)


Correct Repetition Code Syndrome:
                               
q_0: ──■────■────■─────────────
     ┌─┴─┐  │  ┌─┴─┐     ┌─┐   
q_1: ┤ X ├──┼──┤ X ├──■──┤M├───
     └───┘┌─┴─┐└───┘┌─┴─┐└╥┘┌─┐
q_2: ─────┤ X ├─────┤ X ├─╫─┤M├
          └───┘     └───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 

Buggy Repetition Code Syndrome (X on q1):
                                    
q_0: ──■────■─────────■─────────────
     ┌─┴─┐  │  ┌───┐┌─┴─┐     ┌─┐   
q_1: ┤ X ├──┼──┤ X ├┤ X ├──■──┤M├───
     └───┘┌─┴─┐└───┘└───┘┌─┴─┐└╥┘┌─┐
q_2: ─────┤ X ├──────────┤ X ├─╫─┤M├
          └───┘          └───┘ ║ └╥┘
c: 2/══════════════════════════╩══╩═
                               0  1 

REPCODE_BITFLIP – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8728
KL (bug vs ideal)  : 22.2477
Detection ratio    : 25.49x
Adaptive threshold : 2.62
Verdict (>2.0x): BUG DETECTED

REPCODE_BITFLIP – R

In [19]:
# 5-qubit Steane Code Syndrome Extraction ([[5,1,3]] perfect code for bit-flip correction)
# Measures 4 syndrome bits for X errors; single bit flip undetectable under high noise
n_qubits_steane = 5
n_syndromes = 4

# Correct Steane syndrome circuit (|0_L> logical, even parity checks)
steane_correct = QuantumCircuit(n_qubits_steane, n_syndromes)

# Simplified Steane encoding (data q0, ancillas q1-4 init |0>)
# Parity checks: s0=q0^q1, s1=q0^q2, s2=q0^q3, s3=q0^q4 ^q1^q2^q3^q4 (weight-4)
steane_correct.cx(0, 1)  # Check 1
steane_correct.cx(0, 2)
steane_correct.cx(0, 3)
steane_correct.cx(0, 4)

# Weight-4 check using ancillas (chain XORs)
steane_correct.cx(1, 2)  # Propagate for s3
steane_correct.cx(3, 4)
steane_correct.barrier()
# Measure syndromes (ancillas hold parity)
steane_correct.measure([1,2,3,4], [0,1,2,3])

# Buggy: single X bit flip on data qubit q0 (logical error, syndrome non-zero but noise-masked)
steane_buggy = QuantumCircuit(n_qubits_steane, n_syndromes)
steane_buggy.cx(0, 1)
steane_buggy.cx(0, 2)
steane_buggy.cx(0, 3)
steane_buggy.cx(0, 4)
steane_buggy.x(0)        # BUG: X bit flip on logical q0 → syndrome pattern, but depolarizing noise ~uniform 16 outcomes
steane_buggy.cx(1, 2)
steane_buggy.cx(3, 4)
steane_buggy.barrier()
steane_buggy.measure([1,2,3,4], [0,1,2,3])

print("Correct Steane Code Syndrome (5 qubits, 4 syndromes):")
print(steane_correct.draw(fold=-1))

print("\nBuggy Steane Code Syndrome (X bit flip on logical q0):")
print(steane_buggy.draw(fold=-1))

# Run tests (expects INCONCLUSIVE on noisy/HW due to 1/16 → ~0.06 shift overwhelmed by noise)
steane_sim = simulate_bug_detection(
    name="SteaneCode_BitFlip",
    qc_correct=steane_correct,
    qc_buggy=steane_buggy,
    n_qubits=n_qubits_steane,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,  # 8192 → good stats for 16 outcomes
)

steane_hw = hardware_bug_detection(
    name="SteaneCode_BitFlip",
    qc_correct=steane_correct,
    qc_buggy=steane_buggy,
    n_qubits=n_qubits_steane,
    backend=backend,  # Needs ≥5 qubits (torino ok)
    pass_manager=pm,
    probs_ideal=steane_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "SteaneCode_BitFlip",
    sim_result=steane_sim,
    hw_result=steane_hw,
    top_k=8,
    n_qubits=n_syndromes,  # 4 syndrome bits (16 states)
)

Correct Steane Code Syndrome (5 qubits, 4 syndromes):
                                    ░             
q_0: ──■────■────■─────────■────────░─────────────
     ┌─┴─┐  │    │         │        ░ ┌─┐         
q_1: ┤ X ├──┼────┼────■────┼────────░─┤M├─────────
     └───┘┌─┴─┐  │  ┌─┴─┐  │        ░ └╥┘┌─┐      
q_2: ─────┤ X ├──┼──┤ X ├──┼────────░──╫─┤M├──────
          └───┘┌─┴─┐└───┘  │        ░  ║ └╥┘┌─┐   
q_3: ──────────┤ X ├───────┼────■───░──╫──╫─┤M├───
               └───┘     ┌─┴─┐┌─┴─┐ ░  ║  ║ └╥┘┌─┐
q_4: ────────────────────┤ X ├┤ X ├─░──╫──╫──╫─┤M├
                         └───┘└───┘ ░  ║  ║  ║ └╥┘
c: 4/══════════════════════════════════╩══╩══╩══╩═
                                       0  1  2  3 

Buggy Steane Code Syndrome (X bit flip on logical q0):
                              ┌───┐ ░             
q_0: ──■────■────■─────────■──┤ X ├─░─────────────
     ┌─┴─┐  │    │         │  └───┘ ░ ┌─┐         
q_1: ┤ X ├──┼────┼────■────┼────────░─┤M├─────────
     └───┘┌─┴─┐  │  ┌─┴

In [22]:
# Number of qubits in the circuit (q0 initiator, q1-q2 chain)
n_qubits = 3

# Correct GHZ with explicit mid-circuit point
mid_correct = QuantumCircuit(n_qubits, n_qubits)
mid_correct.h(0)
mid_correct.cx(0, 1)
mid_correct.cx(1, 2)
mid_correct.barrier()                    # MID-CIRCUIT: after full encoding
mid_correct.measure([0, 1, 2], [0, 1, 2]) # Explicit clbits match qubits

# Buggy: X bit flip MID-circuit (between CXs, before barrier)
mid_buggy = QuantumCircuit(n_qubits, n_qubits)
mid_buggy.h(0)
mid_buggy.cx(0, 1)
mid_buggy.x(1)                           # BUG: mid-circuit X on q1 (propagates!)
mid_buggy.cx(1, 2)
mid_buggy.barrier()
mid_buggy.measure([0, 1, 2], [0, 1, 2])

print("Correct Mid-Circuit GHZ:")
print(mid_correct.draw())

print("\nBuggy Mid-Circuit GHZ (X bit flip on q1 mid-circuit):")
print(mid_buggy.draw())

mid_sim = simulate_bug_detection(
    name="MidCircuit_BitFlip",
    qc_correct=mid_correct,
    qc_buggy=mid_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

mid_hw = hardware_bug_detection(
    name="MidCircuit_BitFlip",
    qc_correct=mid_correct,
    qc_buggy=mid_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=mid_sim["probs_ideal"],
    shots=shots,
)

show_all_distributions(
    "MidCircuit_BitFlip",
    sim_result=mid_sim,
    hw_result=mid_hw,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Mid-Circuit GHZ:
     ┌───┐           ░ ┌─┐      
q_0: ┤ H ├──■────────░─┤M├──────
     └───┘┌─┴─┐      ░ └╥┘┌─┐   
q_1: ─────┤ X ├──■───░──╫─┤M├───
          └───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ──────────┤ X ├─░──╫──╫─┤M├
               └───┘ ░  ║  ║ └╥┘
c: 3/═══════════════════╩══╩══╩═
                        0  1  2 

Buggy Mid-Circuit GHZ (X bit flip on q1 mid-circuit):
     ┌───┐                ░ ┌─┐      
q_0: ┤ H ├──■─────────────░─┤M├──────
     └───┘┌─┴─┐┌───┐      ░ └╥┘┌─┐   
q_1: ─────┤ X ├┤ X ├──■───░──╫─┤M├───
          └───┘└───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ───────────────┤ X ├─░──╫──╫─┤M├
                    └───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

MIDCIRCUIT_BITFLIP – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8511
KL (bug vs ideal)  : 21.2065
Detection ratio    : 24.92x
Adaptive threshold : 2.55
Verdict (>2.0x): BUG DETECT

In [25]:
# 1. Mid-X Before Final CX (partial propagation)

n_qubits = 3

stealth1_correct = QuantumCircuit(n_qubits, n_qubits)
stealth1_correct.h(0)
stealth1_correct.cx(0, 1)
stealth1_correct.cx(1, 2)
stealth1_correct.measure([0,1,2],[0,1,2])

stealth1_buggy = QuantumCircuit(n_qubits, n_qubits)
stealth1_buggy.h(0)
stealth1_buggy.cx(0, 1)
stealth1_buggy.x(1)                  # MID-X before final CX
stealth1_buggy.cx(1, 2)
stealth1_buggy.barrier()
stealth1_buggy.measure([0,1,2],[0,1,2])

print("\n1. Mid-X Before Final CX:")
print(stealth1_correct.draw())
print(stealth1_buggy.draw())

stealth1_sim = simulate_bug_detection(name="Stealth1_PreFinalCX", qc_correct=stealth1_correct, qc_buggy=stealth1_buggy, n_qubits=n_qubits, sim_ideal=simulator_ideal, sim_noisy=simulator_noisy, shots=shots)
stealth1_hw = hardware_bug_detection(name="Stealth1_PreFinalCX", qc_correct=stealth1_correct, qc_buggy=stealth1_buggy, n_qubits=n_qubits, backend=backend, pass_manager=pm, probs_ideal=stealth1_sim["probs_ideal"], shots=shots)
show_all_distributions("Stealth1_PreFinalCX", sim_result=stealth1_sim, hw_result=stealth1_hw, top_k=8, n_qubits=n_qubits)


1. Mid-X Before Final CX:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 
     ┌───┐                ░ ┌─┐      
q_0: ┤ H ├──■─────────────░─┤M├──────
     └───┘┌─┴─┐┌───┐      ░ └╥┘┌─┐   
q_1: ─────┤ X ├┤ X ├──■───░──╫─┤M├───
          └───┘└───┘┌─┴─┐ ░  ║ └╥┘┌─┐
q_2: ───────────────┤ X ├─░──╫──╫─┤M├
                    └───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

STEALTH1_PREFINALCX – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.8279
KL (bug vs ideal)  : 21.1397
Detection ratio    : 25.53x
Adaptive threshold : 2.48
Verdict (>2.0x): BUG DETECTED

STEALTH1_PREFINALCX – REAL HARDWARE (ibm_torino)
--------------------------

In [26]:
# 2. Double Mid-X (cancellation)

cancel_correct = QuantumCircuit(n_qubits, n_qubits)
cancel_correct.h(0)
cancel_correct.cx(0, 1)
cancel_correct.barrier()
cancel_correct.cx(1, 2)
cancel_correct.measure([0,1,2],[0,1,2])

cancel_buggy = QuantumCircuit(n_qubits, n_qubits)
cancel_buggy.h(0)
cancel_buggy.cx(0, 1)
cancel_buggy.x(1)                    # MID-X1
cancel_buggy.x(0)                    # MID-X2 cancels
cancel_buggy.barrier()
cancel_buggy.cx(1, 2)
cancel_buggy.measure([0,1,2],[0,1,2])

print("\n2. Double Mid-X (Cancellation):")
print(cancel_correct.draw())
print(cancel_buggy.draw())

cancel_sim = simulate_bug_detection(name="Cancel_DoubleX", qc_correct=cancel_correct, qc_buggy=cancel_buggy, n_qubits=n_qubits, sim_ideal=simulator_ideal, sim_noisy=simulator_noisy, shots=shots)
cancel_hw = hardware_bug_detection(name="Cancel_DoubleX", qc_correct=cancel_correct, qc_buggy=cancel_buggy, n_qubits=n_qubits, backend=backend, pass_manager=pm, probs_ideal=cancel_sim["probs_ideal"], shots=shots)
show_all_distributions("Cancel_DoubleX", sim_result=cancel_sim, hw_result=cancel_hw, top_k=8, n_qubits=n_qubits)


2. Double Mid-X (Cancellation):
     ┌───┐      ░      ┌─┐      
q_0: ┤ H ├──■───░──────┤M├──────
     └───┘┌─┴─┐ ░      └╥┘┌─┐   
q_1: ─────┤ X ├─░───■───╫─┤M├───
          └───┘ ░ ┌─┴─┐ ║ └╥┘┌─┐
q_2: ───────────░─┤ X ├─╫──╫─┤M├
                ░ └───┘ ║  ║ └╥┘
c: 3/═══════════════════╩══╩══╩═
                        0  1  2 
     ┌───┐     ┌───┐ ░      ┌─┐      
q_0: ┤ H ├──■──┤ X ├─░──────┤M├──────
     └───┘┌─┴─┐├───┤ ░      └╥┘┌─┐   
q_1: ─────┤ X ├┤ X ├─░───■───╫─┤M├───
          └───┘└───┘ ░ ┌─┴─┐ ║ └╥┘┌─┐
q_2: ────────────────░─┤ X ├─╫──╫─┤M├
                     ░ └───┘ ║  ║ └╥┘
c: 3/════════════════════════╩══╩══╩═
                             0  1  2 

CANCEL_DOUBLEX – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.9113
KL (bug vs ideal)  : 1.2308
Detection ratio    : 1.35x
Adaptive threshold : 2.73
Verdict (>2.0x): INCONCLUSIVE

CANCEL_DOUBLEX – REAL HARDWARE (ibm_torino)
-----

In [27]:
# 3. Mid-X on Idle Qubit (local flip)

idle_correct = QuantumCircuit(n_qubits, n_qubits)
idle_correct.h(0)
idle_correct.cx(0, 1)
idle_correct.barrier()
idle_correct.cx(1, 2)
idle_correct.measure([0,1,2],[0,1,2])

idle_buggy = QuantumCircuit(n_qubits, n_qubits)
idle_buggy.h(0)
idle_buggy.cx(0, 1)
idle_buggy.barrier()
idle_buggy.x(2)                      # MID-X on unused q2
idle_buggy.cx(1, 2)
idle_buggy.measure([0,1,2],[0,1,2])

print("\n3. Mid-X on Idle Qubit:")
print(idle_correct.draw())
print(idle_buggy.draw())

idle_sim = simulate_bug_detection(name="Idle_MidX", qc_correct=idle_correct, qc_buggy=idle_buggy, n_qubits=n_qubits, sim_ideal=simulator_ideal, sim_noisy=simulator_noisy, shots=shots)
idle_hw = hardware_bug_detection(name="Idle_MidX", qc_correct=idle_correct, qc_buggy=idle_buggy, n_qubits=n_qubits, backend=backend, pass_manager=pm, probs_ideal=idle_sim["probs_ideal"], shots=shots)
show_all_distributions("Idle_MidX", sim_result=idle_sim, hw_result=idle_hw, top_k=8, n_qubits=n_qubits)


3. Mid-X on Idle Qubit:
     ┌───┐      ░      ┌─┐      
q_0: ┤ H ├──■───░──────┤M├──────
     └───┘┌─┴─┐ ░      └╥┘┌─┐   
q_1: ─────┤ X ├─░───■───╫─┤M├───
          └───┘ ░ ┌─┴─┐ ║ └╥┘┌─┐
q_2: ───────────░─┤ X ├─╫──╫─┤M├
                ░ └───┘ ║  ║ └╥┘
c: 3/═══════════════════╩══╩══╩═
                        0  1  2 
     ┌───┐      ░      ┌─┐           
q_0: ┤ H ├──■───░──────┤M├───────────
     └───┘┌─┴─┐ ░      └╥┘     ┌─┐   
q_1: ─────┤ X ├─░───────╫───■──┤M├───
          └───┘ ░ ┌───┐ ║ ┌─┴─┐└╥┘┌─┐
q_2: ───────────░─┤ X ├─╫─┤ X ├─╫─┤M├
                ░ └───┘ ║ └───┘ ║ └╥┘
c: 3/═══════════════════╩═══════╩══╩═
                        0       1  2 

IDLE_MIDX – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 0.9433
KL (bug vs ideal)  : 21.5531
Detection ratio    : 22.85x
Adaptive threshold : 2.83
Verdict (>2.0x): BUG DETECTED

IDLE_MIDX – REAL HARDWARE (ibm_torino)
---------------------

In [16]:
# Number of qubits in the circuit (q0 = control, q1 = ancilla)
n_qubits = 2

# Correct phase kickback circuit (no phase flip on control)
pk_correct = QuantumCircuit(n_qubits, 1)  # 2 qubits, 1 classical bit
pk_correct.h(0)       # put control into |+>_c
pk_correct.h(1) # put ancilla into |+>_a
pk_correct.cz(0, 1)   # controlled-Z: phase kickback from control to ancilla
       # rotate ancilla back to Z basis (X-basis measurement)
pk_correct.measure(1, 0)  # measure ancilla into c[0]

# Buggy phase kickback circuit (extra Z phase flip on control)
pk_buggy = QuantumCircuit(n_qubits, 1)
pk_buggy.h(0)         # |+>_c
        # phase flip error on control
pk_buggy.h(1) 
pk_buggy.z(1)# |+>_a
pk_buggy.cz(0, 1)     # kickback now includes the extra -1 phase
pk_buggy.measure(1, 0)

print("Correct Phase Kickback circuit:")
print(pk_correct.draw())

print("\nBuggy Phase Kickback circuit (Z phase flip on control):")
print(pk_buggy.draw())

pk_sim = simulate_bug_detection(
    name="Phase_Kickback",
    qc_correct=pk_correct,
    qc_buggy=pk_buggy,
    n_qubits=n_qubits,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

# pk_hw = hardware_bug_detection(
#     name="Phase_Kickback",
#     qc_correct=pk_correct,
#     qc_buggy=pk_buggy,
#     n_qubits=n_qubits,
#     backend=backend,
#     pass_manager=pm,
#     probs_ideal=pk_sim["probs_ideal"],  # or pk_sim["probsideal"], depending on your dict
#     shots=shots,
# )

show_all_distributions(
    "Phase_Kickback",
    sim_result=pk_sim,
    top_k=8,
    n_qubits=n_qubits,
)

Correct Phase Kickback circuit:
     ┌───┐      
q_0: ┤ H ├─■────
     ├───┤ │ ┌─┐
q_1: ┤ H ├─■─┤M├
     └───┘   └╥┘
c: 1/═════════╩═
              0 

Buggy Phase Kickback circuit (Z phase flip on control):
     ┌───┐           
q_0: ┤ H ├──────■────
     ├───┤┌───┐ │ ┌─┐
q_1: ┤ H ├┤ Z ├─■─┤M├
     └───┘└───┘   └╥┘
c: 1/══════════════╩═
                   0 

PHASE_KICKBACK – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0001
KL (bug vs ideal)  : 0.0000
Detection ratio    : 0.00x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

Phase_Kickback – Probability Distributions
State   Ideal Sim       Noisy Correct   Noisy Buggy     
--------------------------------------------------------------------------------
01      0.5017          0.4951          0.5020          
00      0.4983          0.5049          0.4980          
----------------------------------------------------------------

In [63]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    if j==0:
        qft_buggy.z(1)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)


print("="*80)
print("="*80)


qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    if j==0:
        qft_buggy.z(1)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     ├───┤ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ┤ Z ├─■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
     └───┘          │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
              

In [65]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    if j==1:
        qft_buggy.z(1)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)





print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    if j==1:
        qft_buggy.z(1)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                              ░ ┌─┐      
q_0: ┤ H ├─■────────■────────────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├┤ Z ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘└───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■──────────────────■───────┤ H ├

In [66]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    if j==1:
        qft_buggy.z(1)
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)






print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    if j==1:
        qft_buggy.z(1)
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                              ░ ┌─┐      
q_0: ┤ H ├─■────────■────────────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ Z ├┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘└───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■──────────────────■───────┤ H ├

In [67]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==0 and k==n_qubits_qft-1:
            qft_buggy.z(2)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==0 and k==n_qubits_qft-1:
            qft_buggy.z(2)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) ├───┤ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■───────┤ Z ├─■───────┤ H ├─X──░──╫──╫─┤M├
              

In [68]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==1 and k==n_qubits_qft-1:
            qft_buggy.z(2)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==1 and k==n_qubits_qft-1:
            qft_buggy.z(2)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                              ░ ┌─┐      
q_0: ┤ H ├─■────────■────────────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐                    │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■──────────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ Z ├┤ H ├

In [69]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    qft_buggy.z(2)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    qft_buggy.z(2)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐                                              ░ ┌─┐      
q_0: ┤ H ├─■────────■────────────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐                    │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■──────────────────┼──░──╫─┤M├───
     ┌───┐          │P(π/4) ├───┤ │P(π/2) ┌───┐┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ┤ Z ├──────────■───────┤ Z ├─■───────┤ H ├┤ Z ├

In [17]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    if j==0:
        qft_buggy.x(0)
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    if j==0:
        qft_buggy.x(0)
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐┌───┐                                         ░ ┌─┐      
q_0: ┤ X ├┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘└───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ───────────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                         │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ────────────────────■─────────────■───────┤ H ├

In [15]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==0 and k==1:
            qft_buggy.x(0)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
        if j==0 and k==1:
            qft_buggy.x(0)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit:
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit:
     ┌───┐         ┌───┐                           ░ ┌─┐      
q_0: ┤ H ├─■───────┤ X ├─■──────────────────────X──░─┤M├──────
     └───┘ │P(π/2) ├───┤ │                      │  ░ └╥┘┌─┐   
q_1: ──────■───────┤ H ├─┼────────■─────────────┼──░──╫─┤M├───
                   └───┘ │P(π/4)  │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ────────────────────■────────■───────┤ H ├─X──░──╫──╫─┤M├
              

In [ ]:
n_qubits_qft = 3

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit:")
print(qft_correct.draw())

print("\nBuggy QFT circuit:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)




print("="*80)
print("="*80)

qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_correct.h(i)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)

qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
for i in range(n_qubits_qft):
    qft_buggy.h(i)
for i in range(n_qubits_qft // 2):
    qft_buggy.swap(i, n_qubits_qft - 1 - i)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)


print("Correct QFT circuit + H gate at the end for all qubits:")
print(qft_correct.draw())

print("\nBuggy QFT circuit + H gate at the end for all qubits:")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

In [14]:
n_qubits_phase = 3

phase_correct = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ from q0 to q1,q2
phase_correct.cz(0, 1)
phase_correct.cz(0, 2)

# Layer 2: H on q0; Z on q1,q2
phase_correct.h(0)
phase_correct.z(1)
phase_correct.z(2)

# Layer 3: CX from q0 to q1,q2
phase_correct.cx(0, 1)
phase_correct.cx(0, 2)

# Layer 4: Z on all qubits
phase_correct.z(0)
phase_correct.z(1)
phase_correct.z(2)

phase_correct.barrier()
for q in range(n_qubits_phase):
    phase_correct.measure(q, q)

phase_buggy = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ unchanged
phase_buggy.cz(0, 1)
phase_buggy.cz(0, 2)

# Layer 2: H on q0; S replaces Z on q1,q2
phase_buggy.h(0)
phase_buggy.s(1)
phase_buggy.s(2)

# Layer 3: CX unchanged
phase_buggy.cx(0, 1)
phase_buggy.cx(0, 2)

# Layer 4: S replaces Z on all qubits
phase_buggy.s(0)
phase_buggy.s(1)
phase_buggy.s(2)

phase_buggy.barrier()
for q in range(n_qubits_phase):
    phase_buggy.measure(q, q)

print("Correct Phase-Entanglement circuit:")
print(phase_correct.draw())

print("\nBuggy Phase-Entanglement circuit (Z→S):")
print(phase_buggy.draw())

phase_sim = simulate_bug_detection(
    name="PhaseEntanglement",
    qc_correct=phase_correct,
    qc_buggy=phase_buggy,
    n_qubits=n_qubits_phase,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "PhaseEntanglement",
    sim_result=phase_sim,
    top_k=8,
    n_qubits=n_qubits_phase,
)

Correct Phase-Entanglement circuit:
           ┌───┐          ┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├──■────■──┤ Z ├─░─┤M├──────
      │  │ ├───┤┌─┴─┐  │  ├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ Z ├┤ X ├──┼──┤ Z ├─░──╫─┤M├───
         │ ├───┤└───┘┌─┴─┐├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ Z ├─────┤ X ├┤ Z ├─░──╫──╫─┤M├
           └───┘     └───┘└───┘ ░  ║  ║ └╥┘
c: 3/══════════════════════════════╩══╩══╩═
                                   0  1  2 

Buggy Phase-Entanglement circuit (Z→S):
           ┌───┐          ┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├──■────■──┤ S ├─░─┤M├──────
      │  │ ├───┤┌─┴─┐  │  ├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ S ├┤ X ├──┼──┤ S ├─░──╫─┤M├───
         │ ├───┤└───┘┌─┴─┐├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ S ├─────┤ X ├┤ S ├─░──╫──╫─┤M├
           └───┘     └───┘└───┘ ░  ║  ║ └╥┘
c: 3/══════════════════════════════╩══╩══╩═
                                   0  1  2 

PHASEENTANGLEMENT – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots          

In [15]:
n_qubits_phase = 3

phase_correct = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ from q0 to q1,q2
phase_correct.cz(0, 1)
phase_correct.cz(0, 2)

# Layer 2: H on q0; Z on q1,q2
phase_correct.h(0)
phase_correct.z(1)
phase_correct.z(2)

# Layer 3: CX from q0 to q1,q2
phase_correct.cx(0, 1)
phase_correct.cx(0, 2)

# Layer 4: Z on all qubits
phase_correct.z(0)
phase_correct.z(1)
phase_correct.z(2)

phase_correct.barrier()
for q in range(n_qubits_phase):
    phase_correct.measure(q, q)

phase_buggy = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ unchanged
phase_buggy.cz(0, 1)
phase_buggy.cz(0, 2)

# Layer 2: H on q0; S+T replaces Z on q1,q2 (S=π/2 + T=π/4 → closer to π)
phase_buggy.h(0)
phase_buggy.s(1); phase_buggy.t(1)
phase_buggy.s(2); phase_buggy.t(2)

# Layer 3: CX unchanged
phase_buggy.cx(0, 1)
phase_buggy.cx(0, 2)

# Layer 4: S+T replaces Z on all qubits
phase_buggy.s(0); phase_buggy.t(0)
phase_buggy.s(1); phase_buggy.t(1)
phase_buggy.s(2); phase_buggy.t(2)

phase_buggy.barrier()
for q in range(n_qubits_phase):
    phase_buggy.measure(q, q)

print("Correct Phase-Entanglement circuit:")
print(phase_correct.draw())

print("\nBuggy Phase-Entanglement circuit (Z→S+T):")
print(phase_buggy.draw())

phase_sim = simulate_bug_detection(
    name="PhaseEntanglement",
    qc_correct=phase_correct,
    qc_buggy=phase_buggy,
    n_qubits=n_qubits_phase,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "PhaseEntanglement",
    sim_result=phase_sim,
    top_k=8,
    n_qubits=n_qubits_phase,
)

Correct Phase-Entanglement circuit:
           ┌───┐          ┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├──■────■──┤ Z ├─░─┤M├──────
      │  │ ├───┤┌─┴─┐  │  ├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ Z ├┤ X ├──┼──┤ Z ├─░──╫─┤M├───
         │ ├───┤└───┘┌─┴─┐├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ Z ├─────┤ X ├┤ Z ├─░──╫──╫─┤M├
           └───┘     └───┘└───┘ ░  ║  ║ └╥┘
c: 3/══════════════════════════════╩══╩══╩═
                                   0  1  2 

Buggy Phase-Entanglement circuit (Z→S+T):
           ┌───┐               ┌───┐┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├───────■────■──┤ S ├┤ T ├─░─┤M├──────
      │  │ ├───┤┌───┐┌─┴─┐  │  ├───┤├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ S ├┤ T ├┤ X ├──┼──┤ S ├┤ T ├─░──╫─┤M├───
         │ ├───┤├───┤└───┘┌─┴─┐├───┤├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ S ├┤ T ├─────┤ X ├┤ S ├┤ T ├─░──╫──╫─┤M├
           └───┘└───┘     └───┘└───┘└───┘ ░  ║  ║ └╥┘
c: 3/════════════════════════════════════════╩══╩══╩═
                                             0  1  2 

PHASEENTANGLEMENT – SIMULATION
-------

In [16]:
n_qubits_phase = 3

phase_correct = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ from q0 to q1,q2
phase_correct.cz(0, 1)
phase_correct.cz(0, 2)

# Layer 2: H on q0; Z on q1,q2
phase_correct.h(0)
phase_correct.z(1)
phase_correct.z(2)

# Layer 3: CX from q0 to q1,q2
phase_correct.cx(0, 1)
phase_correct.cx(0, 2)

# Layer 4: Z on all qubits
phase_correct.z(0)
phase_correct.z(1)
phase_correct.z(2)

phase_correct.barrier()
for q in range(n_qubits_phase):
    phase_correct.measure(q, q)

phase_buggy = QuantumCircuit(n_qubits_phase, n_qubits_phase)

# Layer 1: CZ unchanged
phase_buggy.cz(0, 1)
phase_buggy.cz(0, 2)

# Layer 2: H on q0; T replaces Z on q1,q2 (π/4 phase only)
phase_buggy.h(0)
phase_buggy.t(1)
phase_buggy.t(2)

# Layer 3: CX unchanged
phase_buggy.cx(0, 1)
phase_buggy.cx(0, 2)

# Layer 4: T replaces Z on all qubits (π/4 phase only)
phase_buggy.t(0)
phase_buggy.t(1)
phase_buggy.t(2)

phase_buggy.barrier()
for q in range(n_qubits_phase):
    phase_buggy.measure(q, q)

print("Correct Phase-Entanglement circuit:")
print(phase_correct.draw())

print("\nBuggy Phase-Entanglement circuit (Z→T only):")
print(phase_buggy.draw())

phase_sim = simulate_bug_detection(
    name="PhaseEntanglement",
    qc_correct=phase_correct,
    qc_buggy=phase_buggy,
    n_qubits=n_qubits_phase,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "PhaseEntanglement",
    sim_result=phase_sim,
    top_k=8,
    n_qubits=n_qubits_phase,
)

Correct Phase-Entanglement circuit:
           ┌───┐          ┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├──■────■──┤ Z ├─░─┤M├──────
      │  │ ├───┤┌─┴─┐  │  ├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ Z ├┤ X ├──┼──┤ Z ├─░──╫─┤M├───
         │ ├───┤└───┘┌─┴─┐├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ Z ├─────┤ X ├┤ Z ├─░──╫──╫─┤M├
           └───┘     └───┘└───┘ ░  ║  ║ └╥┘
c: 3/══════════════════════════════╩══╩══╩═
                                   0  1  2 

Buggy Phase-Entanglement circuit (Z→T only):
           ┌───┐          ┌───┐ ░ ┌─┐      
q_0: ─■──■─┤ H ├──■────■──┤ T ├─░─┤M├──────
      │  │ ├───┤┌─┴─┐  │  ├───┤ ░ └╥┘┌─┐   
q_1: ─■──┼─┤ T ├┤ X ├──┼──┤ T ├─░──╫─┤M├───
         │ ├───┤└───┘┌─┴─┐├───┤ ░  ║ └╥┘┌─┐
q_2: ────■─┤ T ├─────┤ X ├┤ T ├─░──╫──╫─┤M├
           └───┘     └───┘└───┘ ░  ║  ║ └╥┘
c: 3/══════════════════════════════╩══╩══╩═
                                   0  1  2 

PHASEENTANGLEMENT – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots     

In [13]:
n_qubits_y = 3

# CORRECT circuit: 3-qubit Y-gate circuit (5 layers)
y_correct = QuantumCircuit(n_qubits_y, n_qubits_y)

# Layer 1: Y on q0 (90° rotation around Y-axis)
y_correct.y(0)

# Layer 2: H on q1, Y on q2
y_correct.h(1)
y_correct.y(2)

# Layer 3: CX q0→q1, CZ q1→q2
y_correct.cx(0, 1)
y_correct.cz(1, 2)

# Layer 4: Y on all qubits
y_correct.y(0)
y_correct.y(1)
y_correct.y(2)

# Layer 5: SWAP q0 and q2
y_correct.swap(0, 2)

y_correct.barrier()
for q in range(n_qubits_y):
    y_correct.measure(q, q)

# BUGGY circuit: All Y gates replaced by T gates (weaker phase)
y_buggy = QuantumCircuit(n_qubits_y, n_qubits_y)

# Layer 1: T replaces Y on q0
y_buggy.t(0)

# Layer 2: H on q1, T replaces Y on q2
y_buggy.h(1)
y_buggy.t(2)

# Layer 3: CX q0→q1, CZ q1→q2 (unchanged)
y_buggy.cx(0, 1)
y_buggy.cz(1, 2)

# Layer 4: T replaces Y on all qubits
y_buggy.t(0)
y_buggy.t(1)
y_buggy.t(2)

# Layer 5: SWAP q0 and q2 (unchanged)
y_buggy.swap(0, 2)

y_buggy.barrier()
for q in range(n_qubits_y):
    y_buggy.measure(q, q)

print("Correct Y-Gate circuit:")
print(y_correct.draw())

print("\nBuggy Y-Gate circuit (Y→T only):")
print(y_buggy.draw())

y_sim = simulate_bug_detection(
    name="YGateCircuit",
    qc_correct=y_correct,
    qc_buggy=y_buggy,
    n_qubits=n_qubits_y,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "YGateCircuit",
    sim_result=y_sim,
    top_k=8,
    n_qubits=n_qubits_y,
)

Correct Y-Gate circuit:
     ┌───┐     ┌───┐         ░ ┌─┐      
q_0: ┤ Y ├──■──┤ Y ├──────X──░─┤M├──────
     ├───┤┌─┴─┐└───┘┌───┐ │  ░ └╥┘┌─┐   
q_1: ┤ H ├┤ X ├──■──┤ Y ├─┼──░──╫─┤M├───
     ├───┤└───┘  │  ├───┤ │  ░  ║ └╥┘┌─┐
q_2: ┤ Y ├───────■──┤ Y ├─X──░──╫──╫─┤M├
     └───┘          └───┘    ░  ║  ║ └╥┘
c: 3/═══════════════════════════╩══╩══╩═
                                0  1  2 

Buggy Y-Gate circuit (Y→T only):
     ┌───┐     ┌───┐         ░ ┌─┐      
q_0: ┤ T ├──■──┤ T ├──────X──░─┤M├──────
     ├───┤┌─┴─┐└───┘┌───┐ │  ░ └╥┘┌─┐   
q_1: ┤ H ├┤ X ├──■──┤ T ├─┼──░──╫─┤M├───
     ├───┤└───┘  │  ├───┤ │  ░  ║ └╥┘┌─┐
q_2: ┤ T ├───────■──┤ T ├─X──░──╫──╫─┤M├
     └───┘          └───┘    ░  ║  ║ └╥┘
c: 3/═══════════════════════════╩══╩══╩═
                                0  1  2 

YGATECIRCUIT – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 1.3882
KL (bug vs ideal)  : 1.4524
Detection r

In [14]:
n_qubits_yz = 3

# CORRECT circuit: 3-qubit Y-gate circuit (5 layers)
yz_correct = QuantumCircuit(n_qubits_yz, n_qubits_yz)

# Layer 1: Y on q0 (90° rotation around Y-axis)
yz_correct.y(0)

# Layer 2: H on q1, Y on q2
yz_correct.h(1)
yz_correct.y(2)

# Layer 3: CX q0→q1, CZ q1→q2
yz_correct.cx(0, 1)
yz_correct.cz(1, 2)

# Layer 4: Y on all qubits
yz_correct.y(0)
yz_correct.y(1)
yz_correct.y(2)

# Layer 5: SWAP q0 and q2
yz_correct.swap(0, 2)

yz_correct.barrier()
for q in range(n_qubits_yz):
    yz_correct.measure(q, q)

# BUGGY circuit: All Y gates replaced by Z gates (phase flip only)
yz_buggy = QuantumCircuit(n_qubits_yz, n_qubits_yz)

# Layer 1: Z replaces Y on q0
yz_buggy.z(0)

# Layer 2: H on q1, Z replaces Y on q2
yz_buggy.h(1)
yz_buggy.z(2)

# Layer 3: CX q0→q1, CZ q1→q2 (unchanged)
yz_buggy.cx(0, 1)
yz_buggy.cz(1, 2)

# Layer 4: Z replaces Y on all qubits
yz_buggy.z(0)
yz_buggy.z(1)
yz_buggy.z(2)

# Layer 5: SWAP q0 and q2 (unchanged)
yz_buggy.swap(0, 2)

yz_buggy.barrier()
for q in range(n_qubits_yz):
    yz_buggy.measure(q, q)

print("Correct Y-Gate circuit:")
print(yz_correct.draw())

print("\nBuggy Y-Gate circuit (Y→Z replacement):")
print(yz_buggy.draw())

yz_sim = simulate_bug_detection(
    name="YGateCircuit",
    qc_correct=yz_correct,
    qc_buggy=yz_buggy,
    n_qubits=n_qubits_yz,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "YGateCircuit",
    sim_result=yz_sim,
    top_k=8,
    n_qubits=n_qubits_yz,
)

Correct Y-Gate circuit:
     ┌───┐     ┌───┐         ░ ┌─┐      
q_0: ┤ Y ├──■──┤ Y ├──────X──░─┤M├──────
     ├───┤┌─┴─┐└───┘┌───┐ │  ░ └╥┘┌─┐   
q_1: ┤ H ├┤ X ├──■──┤ Y ├─┼──░──╫─┤M├───
     ├───┤└───┘  │  ├───┤ │  ░  ║ └╥┘┌─┐
q_2: ┤ Y ├───────■──┤ Y ├─X──░──╫──╫─┤M├
     └───┘          └───┘    ░  ║  ║ └╥┘
c: 3/═══════════════════════════╩══╩══╩═
                                0  1  2 

Buggy Y-Gate circuit (Y→Z replacement):
     ┌───┐     ┌───┐         ░ ┌─┐      
q_0: ┤ Z ├──■──┤ Z ├──────X──░─┤M├──────
     ├───┤┌─┴─┐└───┘┌───┐ │  ░ └╥┘┌─┐   
q_1: ┤ H ├┤ X ├──■──┤ Z ├─┼──░──╫─┤M├───
     ├───┤└───┘  │  ├───┤ │  ░  ║ └╥┘┌─┐
q_2: ┤ Z ├───────■──┤ Z ├─X──░──╫──╫─┤M├
     └───┘          └───┘    ░  ║  ║ └╥┘
c: 3/═══════════════════════════╩══╩══╩═
                                0  1  2 

YGATECIRCUIT – SIMULATION
------------------------------------------------------------
Qubits             : 3
Shots              : 8192
KL (noise vs ideal): 1.3936
KL (bug vs ideal)  : 1.3879
Dete

In [24]:
n_qubits_qft = 3

# CORRECT QFT circuit (3 qubits)
qft_correct = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_correct.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_correct.cp(angle, k, j)
for i in range(n_qubits_qft // 2):
    qft_correct.swap(i, n_qubits_qft - 1 - i)
qft_correct.barrier()
for q in range(n_qubits_qft):
    qft_correct.measure(q, q)  # Proper ordering

# BUGGY QFT circuit: MISSING SWAP gates (bit-reversal layer removed)
qft_buggy = QuantumCircuit(n_qubits_qft, n_qubits_qft)
for j in range(n_qubits_qft):
    qft_buggy.h(j)
    for k in range(j + 1, n_qubits_qft):
        angle = np.pi / (2 ** (k - j))
        qft_buggy.cp(angle, k, j)
# BUG: No swap gates (missing bit-reversal)
qft_buggy.barrier()
for q in range(n_qubits_qft):
    qft_buggy.measure(q, q)

print("Correct QFT circuit (3 qubits):")
print(qft_correct.draw())

print("\nBuggy QFT circuit (missing SWAP gates):")
print(qft_buggy.draw())

qft_sim = simulate_bug_detection(
    name="QFT",
    qc_correct=qft_correct,
    qc_buggy=qft_buggy,
    n_qubits=n_qubits_qft,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "QFT",
    sim_result=qft_sim,
    top_k=8,
    n_qubits=n_qubits_qft,
)

Correct QFT circuit (3 qubits):
     ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────X──░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               │  ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────┼──░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ │  ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─X──░──╫──╫─┤M├
                                          └───┘    ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 

Buggy QFT circuit (missing SWAP gates):
     ┌───┐                                      ░ ┌─┐      
q_0: ┤ H ├─■────────■───────────────────────────░─┤M├──────
     └───┘ │P(π/2)  │       ┌───┐               ░ └╥┘┌─┐   
q_1: ──────■────────┼───────┤ H ├─■─────────────░──╫─┤M├───
                    │P(π/4) └───┘ │P(π/2) ┌───┐ ░  ║ └╥┘┌─┐
q_2: ───────────────■─────────────■───────┤ H ├─░──╫──╫─┤M├


In [25]:
from math import pi

n_qubits_pe = 4  # 1 control + 3 target qubits for phase estimation

# CORRECT Phase Estimation circuit
pe_correct = QuantumCircuit(n_qubits_pe, n_qubits_pe)

# Prepare eigenstate |1> on target qubits (q1,q2,q3)
pe_correct.x(1)
pe_correct.x(2)
pe_correct.x(3)

# Put control qubit q0 into superposition
pe_correct.h(0)

# Controlled-U operations (increasing powers)
pe_correct.cp(pi/2, 1, 0)   # U^{1} controlled on q0→q1
pe_correct.cp(pi/4, 2, 0)   # U^{2} controlled on q0→q2  
pe_correct.cp(pi/8, 3, 0)   # U^{4} controlled on q0→q3

# QFT on control qubit (simplified for single qubit, full for multi)
pe_correct.h(0)

# Bit-reversal swap (essential for QFT output ordering)
pe_correct.swap(0, 3)

pe_correct.barrier()
for q in range(n_qubits_pe):
    pe_correct.measure(q, q)

# BUGGY Phase Estimation: MISSING SWAP gate
pe_buggy = QuantumCircuit(n_qubits_pe, n_qubits_pe)

# Same preparation
pe_buggy.x(1)
pe_buggy.x(2)
pe_buggy.x(3)
pe_buggy.h(0)

# Same controlled-U operations
pe_buggy.cp(pi/2, 1, 0)
pe_buggy.cp(pi/4, 2, 0)
pe_buggy.cp(pi/8, 3, 0)

pe_buggy.h(0)

# BUG: No swap gate (missing bit-reversal)
pe_buggy.barrier()
for q in range(n_qubits_pe):
    pe_buggy.measure(q, q)

print("Correct Phase-Estimation circuit:")
print(pe_correct.draw())

print("\nBuggy Phase-Estimation circuit (missing SWAP):")
print(pe_buggy.draw())

pe_sim = simulate_bug_detection(
    name="PhaseEstimation",
    qc_correct=pe_correct,
    qc_buggy=pe_buggy,
    n_qubits=n_qubits_pe,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "PhaseEstimation",
    sim_result=pe_sim,
    top_k=12,
    n_qubits=n_qubits_pe,
)

Correct Phase-Estimation circuit:
     ┌───┐                           ┌───┐    ░ ┌─┐         
q_0: ┤ H ├─■────────■────────■───────┤ H ├─X──░─┤M├─────────
     ├───┤ │P(π/2)  │        │       └───┘ │  ░ └╥┘┌─┐      
q_1: ┤ X ├─■────────┼────────┼─────────────┼──░──╫─┤M├──────
     ├───┤          │P(π/4)  │             │  ░  ║ └╥┘┌─┐   
q_2: ┤ X ├──────────■────────┼─────────────┼──░──╫──╫─┤M├───
     ├───┤                   │P(π/8)       │  ░  ║  ║ └╥┘┌─┐
q_3: ┤ X ├───────────────────■─────────────X──░──╫──╫──╫─┤M├
     └───┘                                    ░  ║  ║  ║ └╥┘
c: 4/════════════════════════════════════════════╩══╩══╩══╩═
                                                 0  1  2  3 

Buggy Phase-Estimation circuit (missing SWAP):
     ┌───┐                           ┌───┐ ░ ┌─┐         
q_0: ┤ H ├─■────────■────────■───────┤ H ├─░─┤M├─────────
     ├───┤ │P(π/2)  │        │       └───┘ ░ └╥┘┌─┐      
q_1: ┤ X ├─■────────┼────────┼─────────────░──╫─┤M├──────
     ├───┤     

In [11]:
n_qubits_iswap = 2

# CORRECT circuit: iSWAP gate + measurement
iswap_correct = QuantumCircuit(n_qubits_iswap, n_qubits_iswap)

# Prepare initial state |10> 
iswap_correct.x(0)

# iSWAP gate between q0 and q1
iswap_correct.iswap(0, 1)

iswap_correct.barrier()
for q in range(n_qubits_iswap):
    iswap_correct.measure(q, q)

# BUGGY circuit: iSWAP replaced by plain SWAP
iswap_buggy = QuantumCircuit(n_qubits_iswap, n_qubits_iswap)

# Same initial state |10>
iswap_buggy.x(0)

# BUG: SWAP instead of iSWAP (no phase coupling)
iswap_buggy.swap(0, 1)

iswap_buggy.barrier()
for q in range(n_qubits_iswap):
    iswap_buggy.measure(q, q)

print("Correct iSWAP circuit (2 qubits):")
print(iswap_correct.draw())

print("\nBuggy circuit (iSWAP → SWAP):")
print(iswap_buggy.draw())

iswap_sim = simulate_bug_detection(
    name="iSWAP_2q",
    qc_correct=iswap_correct,
    qc_buggy=iswap_buggy,
    n_qubits=n_qubits_iswap,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "iSWAP_2q",
    sim_result=iswap_sim,
    top_k=4,
    n_qubits=n_qubits_iswap,
)

Correct iSWAP circuit (2 qubits):
     ┌───┐┌────────┐ ░ ┌─┐   
q_0: ┤ X ├┤0       ├─░─┤M├───
     └───┘│  Iswap │ ░ └╥┘┌─┐
q_1: ─────┤1       ├─░──╫─┤M├
          └────────┘ ░  ║ └╥┘
c: 2/═══════════════════╩══╩═
                        0  1 

Buggy circuit (iSWAP → SWAP):
     ┌───┐    ░ ┌─┐   
q_0: ┤ X ├─X──░─┤M├───
     └───┘ │  ░ └╥┘┌─┐
q_1: ──────X──░──╫─┤M├
              ░  ║ └╥┘
c: 2/════════════╩══╩═
                 0  1 

ISWAP_2Q – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 1.6740
KL (bug vs ideal)  : 0.1993
Detection ratio    : 0.12x
Adaptive threshold : 5.02
Verdict (>2.0x): INCONCLUSIVE

iSWAP_2q – Probability Distributions
State   Ideal Sim       Noisy Correct   Noisy Buggy     
--------------------------------------------------------------------------------
10      1.0000          0.9099          0.9886          
00      0.0000          0.0344          0.0114          
01

In [12]:
from math import pi

n_qubits_pe = 3  # 1 control + 2 target qubits for phase estimation

# CORRECT Phase Estimation circuit
pe_correct = QuantumCircuit(n_qubits_pe, n_qubits_pe)

# Prepare eigenstate |1> on target qubits (q1,q2)
pe_correct.x(1)
pe_correct.x(2)

# Put control qubit q0 into superposition
pe_correct.h(0)

# Controlled-U operations (increasing powers)
pe_correct.cp(pi/2, 1, 0)   # U^{1} controlled on q0→q1
pe_correct.cp(pi/4, 2, 0)   # U^{2} controlled on q0→q2  

# QFT on control qubit (simplified for single qubit)
pe_correct.h(0)

# Bit-reversal swap (essential for QFT output ordering)
pe_correct.swap(0, 2)

pe_correct.barrier()
for q in range(n_qubits_pe):
    pe_correct.measure(q, q)

# BUGGY Phase Estimation: MISSING SWAP gate
pe_buggy = QuantumCircuit(n_qubits_pe, n_qubits_pe)

# Same preparation
pe_buggy.x(1)
pe_buggy.x(2)
pe_buggy.h(0)

# Same controlled-U operations
pe_buggy.cp(pi/2, 1, 0)
pe_buggy.cp(pi/4, 2, 0)

pe_buggy.h(0)

# BUG: No swap gate (missing bit-reversal)
pe_buggy.barrier()
for q in range(n_qubits_pe):
    pe_buggy.measure(q, q)

print("Correct Phase-Estimation circuit:")
print(pe_correct.draw())

print("\nBuggy Phase-Estimation circuit (missing SWAP):")
print(pe_buggy.draw())

pe_sim = simulate_bug_detection(
    name="PhaseEstimation",
    qc_correct=pe_correct,
    qc_buggy=pe_buggy,
    n_qubits=n_qubits_pe,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "PhaseEstimation",
    sim_result=pe_sim,
    top_k=8,
    n_qubits=n_qubits_pe,
)

Correct Phase-Estimation circuit:
     ┌───┐                  ┌───┐    ░ ┌─┐      
q_0: ┤ H ├─■────────■───────┤ H ├─X──░─┤M├──────
     ├───┤ │P(π/2)  │       └───┘ │  ░ └╥┘┌─┐   
q_1: ┤ X ├─■────────┼─────────────┼──░──╫─┤M├───
     ├───┤          │P(π/4)       │  ░  ║ └╥┘┌─┐
q_2: ┤ X ├──────────■─────────────X──░──╫──╫─┤M├
     └───┘                           ░  ║  ║ └╥┘
c: 3/═══════════════════════════════════╩══╩══╩═
                                        0  1  2 

Buggy Phase-Estimation circuit (missing SWAP):
     ┌───┐                  ┌───┐ ░ ┌─┐      
q_0: ┤ H ├─■────────■───────┤ H ├─░─┤M├──────
     ├───┤ │P(π/2)  │       └───┘ ░ └╥┘┌─┐   
q_1: ┤ X ├─■────────┼─────────────░──╫─┤M├───
     ├───┤          │P(π/4)       ░  ║ └╥┘┌─┐
q_2: ┤ X ├──────────■─────────────░──╫──╫─┤M├
     └───┘                        ░  ║  ║ └╥┘
c: 3/════════════════════════════════╩══╩══╩═
                                     0  1  2 

PHASEESTIMATION – SIMULATION
---------------------------------

In [13]:
n_qubits_ipea = 3

# CORRECT iPEA (iterative Phase Estimation Algorithm) circuit
ipea_correct = QuantumCircuit(n_qubits_ipea, n_qubits_ipea)

# Iterative phase estimation: q0=control/ancilla, q1-q2=register
ipea_correct.h(0)  # Control superposition

# Controlled-U^2^k operations iteratively
ipea_correct.cp(pi/2, 1, 0)      # First iteration: U^1
ipea_correct.h(0); ipea_correct.measure(0, 0); ipea_correct.reset(0)
ipea_correct.h(0)
ipea_correct.cp(pi/4, 2, 0)      # Second iteration: U^2  
ipea_correct.h(0); 
ipea_correct.measure(0, 1); 
ipea_correct.reset(0)

# Final register measurement
ipea_correct.measure([1,2], [2,0])  # Adaptive bit ordering

ipea_correct.barrier()

# BUGGY iPEA: Missing reset gates after measurements
ipea_buggy = QuantumCircuit(n_qubits_ipea, n_qubits_ipea)

ipea_buggy.h(0)
ipea_buggy.cp(pi/2, 1, 0)
ipea_buggy.h(0); ipea_buggy.measure(0, 0)  # BUG: No reset(0)
ipea_buggy.h(0)
ipea_buggy.cp(pi/4, 2, 0)
ipea_buggy.h(0); ipea_buggy.measure(0, 1)  # Collapsed state carries over

ipea_buggy.measure([1,2], [2,0])

ipea_buggy.barrier()

print("Correct iPEA circuit:")
print(ipea_correct.draw())

print("\nBuggy iPEA circuit (missing reset gates):")
print(ipea_buggy.draw())

ipea_sim = simulate_bug_detection(
    name="iPEA",
    qc_correct=ipea_correct,
    qc_buggy=ipea_buggy,
    n_qubits=n_qubits_ipea,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "iPEA",
    sim_result=ipea_sim,
    top_k=8,
    n_qubits=n_qubits_ipea,
)

Correct iPEA circuit:
     ┌───┐         ┌───┐┌─┐     ┌───┐         ┌───┐┌─┐      ░ 
q_0: ┤ H ├─■───────┤ H ├┤M├─|0>─┤ H ├─■───────┤ H ├┤M├─|0>──░─
     └───┘ │P(π/2) └┬─┬┘└╥┘     └───┘ │       └───┘└╥┘      ░ 
q_1: ──────■────────┤M├──╫────────────┼─────────────╫───────░─
                    └╥┘  ║            │P(π/4)  ┌─┐  ║       ░ 
q_2: ────────────────╫───╫────────────■────────┤M├──╫───────░─
                     ║   ║                     └╥┘  ║       ░ 
c: 3/════════════════╩═══╩══════════════════════╩═══╩═════════
                     2   0                      0   1         

Buggy iPEA circuit (missing reset gates):
     ┌───┐         ┌───┐┌─┐┌───┐         ┌───┐┌─┐ ░ 
q_0: ┤ H ├─■───────┤ H ├┤M├┤ H ├─■───────┤ H ├┤M├─░─
     └───┘ │P(π/2) └┬─┬┘└╥┘└───┘ │       └───┘└╥┘ ░ 
q_1: ──────■────────┤M├──╫───────┼─────────────╫──░─
                    └╥┘  ║       │P(π/4)  ┌─┐  ║  ░ 
q_2: ────────────────╫───╫───────■────────┤M├──╫──░─
                     ║   ║                └╥┘  ║  

In [18]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate

# ── Custom Unitary: e^(iπ/4) · Rx(π/2) ─────────────────────────
# This gate has a global phase e^(iπ/4) baked in
theta = np.pi / 2
phi   = np.pi / 4

U_matrix = np.array([
    [np.cos(theta/2),       -1j * np.sin(theta/2)],
    [-1j * np.sin(theta/2),  np.cos(theta/2)     ]
])

# Same gate used in BOTH circuits — global phase e^(iπ/4) included
U_global = UnitaryGate(np.exp(1j * phi) * U_matrix, label='e^iφ·U')

# ── CORRECT Circuit ──────────────────────────────────────────────
# q0: H only  → |+⟩ superposition
# q1: H → e^(iφ)U applied DIRECTLY (not controlled)
# e^(iφ) is on ENTIRE state → global → invisible → no bug

gphase_correct = QuantumCircuit(2, 2)

gphase_correct.h(0)                      # q0 = |+⟩
gphase_correct.h(1)                      # q1 = |+⟩
gphase_correct.append(U_global, [1])     # e^(iφ)U direct → global phase invisible

gphase_correct.barrier()
gphase_correct.measure(0, 0)
gphase_correct.measure(1, 1)

# ── BUGGY Circuit (GPHASE) ───────────────────────────────────────
# q0: H only  → |+⟩ superposition  (control)
# q1: H → ctrl-[e^(iφ)U]           (target)
# e^(iφ) attaches ONLY to |1⟩ branch → becomes LOCAL → observable → BUG

gphase_buggy = QuantumCircuit(2, 2)

gphase_buggy.h(0)                              # q0 = |+⟩ superposition
gphase_buggy.h(1)                              # q1 = |+⟩
gphase_buggy.append(U_global.control(1), [0, 1])  # ctrl-[e^iφ·U] → phase becomes LOCAL

gphase_buggy.barrier()
gphase_buggy.measure(0, 0)
gphase_buggy.measure(1, 1)

# ── Print circuits ───────────────────────────────────────────────
print("── CORRECT: e^iφ·U direct (global phase → invisible) ──")
print(gphase_correct.draw())

print("\n── BUGGY GPHASE: ctrl-[e^iφ·U] (global phase → local) ──")
print(gphase_buggy.draw())

# ── Run detection ────────────────────────────────────────────────
gphase_sim = simulate_bug_detection(
    name="GPHASE",
    qc_correct=gphase_correct,
    qc_buggy=gphase_buggy,
    n_qubits=2,
    sim_ideal=simulator_ideal,
    sim_noisy=simulator_noisy,
    shots=shots,
)

show_all_distributions(
    "GPHASE",
    sim_result=gphase_sim,
    top_k=4,
    n_qubits=2,
)

── CORRECT: e^iφ·U direct (global phase → invisible) ──
     ┌───┐           ░ ┌─┐   
q_0: ┤ H ├───────────░─┤M├───
     ├───┤┌────────┐ ░ └╥┘┌─┐
q_1: ┤ H ├┤ e^iφ·U ├─░──╫─┤M├
     └───┘└────────┘ ░  ║ └╥┘
c: 2/═══════════════════╩══╩═
                        0  1 

── BUGGY GPHASE: ctrl-[e^iφ·U] (global phase → local) ──
     ┌───┐           ░ ┌─┐   
q_0: ┤ H ├────■──────░─┤M├───
     ├───┤┌───┴────┐ ░ └╥┘┌─┐
q_1: ┤ H ├┤ e^iφ·U ├─░──╫─┤M├
     └───┘└────────┘ ░  ║ └╥┘
c: 2/═══════════════════╩══╩═
                        0  1 

GPHASE – SIMULATION
------------------------------------------------------------
Qubits             : 2
Shots              : 8192
KL (noise vs ideal): 0.0006
KL (bug vs ideal)  : 0.0002
Detection ratio    : 0.25x
Adaptive threshold : 0.00
Verdict (>2.0x): INCONCLUSIVE

GPHASE – Probability Distributions
State   Ideal Sim       Noisy Correct   Noisy Buggy     
--------------------------------------------------------------------------------
01      0.2550        

In [23]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

def run(qc):
    t = transpile(qc, sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()
    total = sum(counts.values())
    return {k: round(v / total, 4) for k, v in sorted(counts.items())}

# ─────────────────────────────────────────────
# CIRCUIT 1a: Z-Phase HIDDEN
# ─────────────────────────────────────────────
t1a = QuantumCircuit(1, 1, name="Z-Phase HIDDEN")
t1a.h(0)
t1a.z(0)
t1a.measure(0, 0)

# ─────────────────────────────────────────────
# CIRCUIT 1b: Z-Phase REVEALED
# ─────────────────────────────────────────────
t1b = QuantumCircuit(1, 1, name="Z-Phase REVEALED")
t1b.h(0)
t1b.z(0)
t1b.h(0)           # <-- basis rotation
t1b.measure(0, 0)

# ─────────────────────────────────────────────
# CIRCUIT 2: Y-Phase via S†H
# ─────────────────────────────────────────────
t2 = QuantumCircuit(1, 1, name="Y-Phase via SdgH")
t2.h(0)
t2.sdg(0)          # creates |−i>
t2.sdg(0)          # S† part of basis rotation
t2.h(0)            # H part of basis rotation
t2.measure(0, 0)

# ─────────────────────────────────────────────
# CIRCUIT 3: Hadamard Test (θ = π/2)
# ─────────────────────────────────────────────
t3 = QuantumCircuit(2, 1, name="Hadamard Test")
t3.h(0)
t3.cp(np.pi / 2, 0, 1)
t3.h(0)
t3.measure(0, 0)

# ─────────────────────────────────────────────
# CIRCUIT 4: QPE (θ = π/2)
# ─────────────────────────────────────────────
n_count = 3
t4 = QuantumCircuit(n_count + 1, n_count, name="QPE")
t4.x(n_count)
for q in range(n_count):
    t4.h(q)
for k in range(n_count):
    for _ in range(2 ** k):
        t4.cp(np.pi / 2, k, n_count)
t4.barrier()
for j in range(n_count // 2):
    t4.swap(j, n_count - 1 - j)
for j in range(n_count):
    for k in range(j):
        t4.cp(-np.pi / 2 ** (j - k), k, j)
    t4.h(j)
t4.barrier()
t4.measure(range(n_count), range(n_count))

# ─────────────────────────────────────────────
# DRAW CIRCUITS (text mode — no extra libs)
# ─────────────────────────────────────────────
for qc in [t1a, t1b, t2, t3, t4]:
    print("\n" + "=" * 55)
    print(f"  CIRCUIT: {qc.name}")
    print("=" * 55)
    print(qc.draw(output="text", fold=-1))

# ─────────────────────────────────────────────
# SIMULATE & PRINT RESULTS
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("SIMULATION RESULTS")
print("=" * 55)

print("\n[1a] Z-Phase HIDDEN (no rotation):")
print("     ", run(t1a), " ← ~50/50, phase invisible")

print("\n[1b] Z-Phase REVEALED (H before measure):")
print("     ", run(t1b), " ← P(1)≈1.0, phase visible")

print("\n[2]  Y-Phase via S†H:")
print("     ", run(t2), " ← P(1)≈1.0, Y-phase visible")

print("\n[3]  Hadamard Test — P(0) = (1 + cos θ) / 2:")
for lbl, theta in [("π/4", np.pi/4), ("π/2", np.pi/2), ("π", np.pi)]:
    qc = QuantumCircuit(2, 1)
    qc.h(0); qc.cp(theta, 0, 1); qc.h(0); qc.measure(0, 0)
    p0 = run(qc).get("0", 0)
    print(f"     θ={lbl:4s} | P(0)={p0:.4f} | expected={(1+np.cos(theta))/2:.4f}")

print("\n[4]  QPE (θ=π/2) — expected |010> = 0.2500:")
for bs, p in sorted(run(t4).items(), key=lambda x: x[1], reverse=True)[:4]:
    print(f"     |{bs}>  P={p:.4f}  θ_est={int(bs,2)/2**n_count:.4f}")


  CIRCUIT: Z-Phase HIDDEN
     ┌───┐┌───┐┌─┐
  q: ┤ H ├┤ Z ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 

  CIRCUIT: Z-Phase REVEALED
     ┌───┐┌───┐┌───┐┌─┐
  q: ┤ H ├┤ Z ├┤ H ├┤M├
     └───┘└───┘└───┘└╥┘
c: 1/════════════════╩═
                     0 

  CIRCUIT: Y-Phase via SdgH
     ┌───┐┌─────┐┌─────┐┌───┐┌─┐
  q: ┤ H ├┤ Sdg ├┤ Sdg ├┤ H ├┤M├
     └───┘└─────┘└─────┘└───┘└╥┘
c: 1/═════════════════════════╩═
                              0 

  CIRCUIT: Hadamard Test
     ┌───┐         ┌───┐┌─┐
q_0: ┤ H ├─■───────┤ H ├┤M├
     └───┘ │P(π/2) └───┘└╥┘
q_1: ──────■─────────────╫─
                         ║ 
c: 1/════════════════════╩═
                         0 

  CIRCUIT: QPE
     ┌───┐                                                                ░    ┌───┐                                         ░ ┌─┐      
q_0: ┤ H ├─■──────────────────────────────────────────────────────────────░──X─┤ H ├─■──────────────■────────────────────────░─┤M├──────
     ├───┤ │           

In [3]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Dummy unitary U: adds phase e^{iθ} to |1>
# -------------------------------------------------
def make_U(theta):
    """
    Single-qubit unitary U such that:
        U|0> = |0>
        U|1> = e^{iθ} |1>
    Implemented as the phase gate P(θ).
    """
    U = QuantumCircuit(1, name="U(θ)")
    U.p(theta, 0)              # Phase gate P(θ) [web:87][web:89]
    return U.to_gate(label="U(θ)")


theta_true = 3* np.pi / 4  # choose any angle you like (here: 60 degrees)
U_gate = make_U(theta_true)

# -------------------------------------------------
# 2) Hadamard test circuit for this U
#    ancilla = q0, data qubit = q1
# -------------------------------------------------
qc = QuantumCircuit(2, 1)
# Data qubit in eigenstate |1>
qc.x(1)

# Hadamard on ancilla
qc.h(0)

# Controlled-U acting on data, controlled by ancilla
qc.append(U_gate.control(1), [0, 1])

# Second Hadamard on ancilla (interference)
qc.h(0)

# Measure ancilla only
qc.measure(0, 0)

print("Hadamard-test circuit:")
print(qc.draw(output="text", fold=-1))

# -------------------------------------------------
# 3) Run the circuit and collect ancilla statistics
# -------------------------------------------------
t_qc = transpile(qc, sim)
result = sim.run(t_qc, shots=SHOTS).result()
counts = result.get_counts()

n0 = counts.get("0", 0)
n1 = counts.get("1", 0)
p0 = n0 / SHOTS
p1 = n1 / SHOTS

print("\nAncilla measurement counts:", counts)
print(f"P(0) = {p0:.4f}")
print(f"P(1) = {p1:.4f}")

# -------------------------------------------------
# 4) Recover the phase θ from P(0)
#    For eigenstate |1>, U|1> = e^{iθ}|1>  ⇒  P(0) = (1 + cos θ)/2
# -------------------------------------------------
theta_est = np.arccos(2 * p0 - 1)

print("\nPhase estimation from Hadamard test:")
print(f"  θ_true  = {theta_true:.6f} rad  ({np.degrees(theta_true):.2f}°)")
print(f"  θ_est   = {theta_est:.6f} rad  ({np.degrees(theta_est):.2f}°)")

Hadamard-test circuit:
     ┌───┐        ┌───┐┌─┐
q_0: ┤ H ├───■────┤ H ├┤M├
     ├───┤┌──┴───┐└───┘└╥┘
q_1: ┤ X ├┤ U(θ) ├──────╫─
     └───┘└──────┘      ║ 
c: 1/═══════════════════╩═
                        0 

Ancilla measurement counts: {'0': 1194, '1': 6998}
P(0) = 0.1458
P(1) = 0.8542

Phase estimation from Hadamard test:
  θ_true  = 2.356194 rad  (135.00°)
  θ_est   = 2.358161 rad  (135.11°)


In [11]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Define U_block: 2-qubit unitary adding phase e^{iθ} to |11>
#    U = diag(1, 1, 1, e^{iθ}) implemented by a controlled-Phase gate
# -------------------------------------------------
def make_U_block(theta):
    """
    Two-qubit unitary U such that:
        U|00> = |00>
        U|01> = |01>
        U|10> = |10>
        U|11> = e^{iθ} |11>
    Implemented as controlled-Phase P(θ).
    """
    U = QuantumCircuit(2, name="U_block")
    U.cp(theta, 0, 1)   # control=q0, target=q1 [web:87]
    return U

theta_true = 4* np.pi / 5          # choose any angle you like (here: 60 degrees)
U_block   = make_U_block(theta_true)
U_gate    = U_block.to_gate(label="U")     # convert subcircuit to a Gate [web:99]
U_ctrl    = U_gate.control(1)              # add 1 control qubit (ancilla) [web:98]

# -------------------------------------------------
# 2) Build Hadamard-test circuit around this U
#    ancilla = q0, data qubits = q1,q2
#    Data prepared in |11>, which is an eigenstate of U with eigenvalue e^{iθ}
# -------------------------------------------------
qc = QuantumCircuit(1 + 2, 1)   # 1 ancilla + 2 data, 1 classical bit
anc = 0
d0, d1 = 1, 2

# Prepare data register in |11>
qc.x(d0)
qc.x(d1)

# Hadamard on ancilla
qc.h(anc)

# Controlled-U acting on data qubits (control is ancilla)
qc.append(U_ctrl, [anc, d0, d1])

# Second Hadamard on ancilla (interference)
qc.h(anc)

# Measure ancilla only
qc.measure(anc, 0)

print("Hadamard-test circuit with U_block as a controlled subcircuit:\n")
print(qc.draw(output="text", fold=-1))

# -------------------------------------------------
# 3) Run and collect ancilla statistics
# -------------------------------------------------
t_qc   = transpile(qc, sim)
result = sim.run(t_qc, shots=SHOTS).result()
counts = result.get_counts()

n0 = counts.get("0", 0)
n1 = counts.get("1", 0)
p0 = n0 / SHOTS
p1 = n1 / SHOTS

print("\nAncilla measurement counts:", counts)
print(f"P(0) = {p0:.4f}")
print(f"P(1) = {p1:.4f}")

# For eigenstate |ψ> with U|ψ> = e^{iθ}|ψ>, Hadamard test gives:
#     P(0) = (1 + cos θ) / 2   [web:73][web:20]
# so we can estimate θ from P(0)
theta_est = np.arccos(2 * p0 - 1)

print("\nPhase estimation from Hadamard test:")
print(f"  θ_true  = {theta_true:.6f} rad  ({np.degrees(theta_true):.2f}°)")
print(f"  θ_est   = {theta_est:.6f} rad  ({np.degrees(theta_est):.2f}°)")

Hadamard-test circuit with U_block as a controlled subcircuit:

     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     ├───┤┌─┴──┐└───┘└╥┘
q_1: ┤ X ├┤0   ├──────╫─
     ├───┤│  U │      ║ 
q_2: ┤ X ├┤1   ├──────╫─
     └───┘└────┘      ║ 
c: 1/═════════════════╩═
                      0 

Ancilla measurement counts: {'0': 750, '1': 7442}
P(0) = 0.0916
P(1) = 0.9084

Phase estimation from Hadamard test:
  θ_true  = 2.513274 rad  (144.00°)
  θ_est   = 2.526803 rad  (144.78°)


In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Define a 3-qubit entangling unitary U_block
#    (replace this with your real circuit in practice)
# -------------------------------------------------
def make_U_block(theta):
    """
    3-qubit entangling unitary U.
    Phase comes from Rz(theta) on qubit 1,
    sandwiched by CNOT layers.
    """
    U = QuantumCircuit(3, name="U_block")
    U.h(0)
    U.cx(0, 1)
    U.rz(theta, 1)
    U.cx(1, 2)
    return U

theta_true = 4 * np.pi / 5    # some non-trivial phase
U_block    = make_U_block(theta_true)
U_gate     = U_block.to_gate(label="U")
U_ctrl     = U_gate.control(1)   # controlled-U

print("U_block (3-qubit entangling circuit):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Helper: build Hadamard-test circuit for a given
#    state-preparation routine prep_psi(qc, data_qubits)
# -------------------------------------------------
def hadamard_test_for_state(prep_psi, description):
    """
    prep_psi(qc, data) should prepare |psi> on 'data' qubits.
    Returns P(0), P(1), Re(<psi|U|psi>) and an effective angle.
    """
    n_data = 3
    anc = 0
    data = [1, 2, 3]

    qc = QuantumCircuit(1 + n_data, 1)

    # (A) Prepare |psi> on data qubits
    prep_psi(qc, data)

    # (B) Hadamard on ancilla
    qc.h(anc)

    # (C) Controlled-U on data
    qc.append(U_ctrl, [anc] + data)

    # (D) Hadamard on ancilla
    qc.h(anc)

    # (E) Measure ancilla
    qc.measure(anc, 0)

    print(f"\nHadamard-test circuit for state: {description}")
    print(qc.draw(output="text", fold=-1))

    # Run
    t_qc   = transpile(qc, sim)
    result = sim.run(t_qc, shots=SHOTS).result()
    counts = result.get_counts()

    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    p0 = n0 / SHOTS
    p1 = n1 / SHOTS

    # Re(<psi|U|psi>) = P(0) - P(1) = 2P(0) - 1  (standard Hadamard test) [web:73]
    re_expect = 2 * p0 - 1
    # If |psi> ≈ eigenstate, you can interpret this as cos(theta_eff)
    arg = max(-1.0, min(1.0, re_expect))
    theta_eff = np.arccos(arg)

    print("Ancilla counts:", counts)
    print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
    print(f"Re(<psi|U|psi>) ≈ {re_expect:.4f}")
    print(f"θ_eff (from arccos(Re)) ≈ {theta_eff:.6f} rad  ({np.degrees(theta_eff):.2f}°)")
    return p0, p1, re_expect, theta_eff


# -------------------------------------------------
# 3) Define several example |psi> preparations
#    You can replace these with any states you want.
# -------------------------------------------------
def prep_000(qc, data):
    # |000>
    pass

def prep_111(qc, data):
    # |111>
    for q in data:
        qc.x(q)

def prep_GHZ(qc, data):
    # (|000> + |111>) / sqrt(2)
    a, b, c = data
    qc.h(a)
    qc.cx(a, b)
    qc.cx(b, c)

def prep_plus_plus_plus(qc, data):
    # |+++> = H⊗H⊗H |000>
    for q in data:
        qc.h(q)

# -------------------------------------------------
# 4) Run Hadamard test for different |psi>
# -------------------------------------------------
hadamard_test_for_state(prep_000,        "|000>")
hadamard_test_for_state(prep_111,        "|111>")


U_block (3-qubit entangling circuit):
     ┌───┐                      
q_0: ┤ H ├──■───────────────────
     └───┘┌─┴─┐┌──────────┐     
q_1: ─────┤ X ├┤ Rz(4π/5) ├──■──
          └───┘└──────────┘┌─┴─┐
q_2: ──────────────────────┤ X ├
                           └───┘

Hadamard-test circuit for state: |000>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     └───┘┌─┴──┐└───┘└╥┘
q_1: ─────┤0   ├──────╫─
          │    │      ║ 
q_2: ─────┤1 U ├──────╫─
          │    │      ║ 
q_3: ─────┤2   ├──────╫─
          └────┘      ║ 
c: 1/═════════════════╩═
                      0 
Ancilla counts: {'1': 3259, '0': 4933}
P(0) = 0.6022, P(1) = 0.3978
Re(<psi|U|psi>) ≈ 0.2043
θ_eff (from arccos(Re)) ≈ 1.365001 rad  (78.21°)

Hadamard-test circuit for state: |111>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     ├───┤┌─┴──┐└───┘└╥┘
q_1: ┤ X ├┤0   ├──────╫─
     ├───┤│    │      ║ 
q_2: ┤ X ├┤1 U ├──────╫─
     ├───┤│    │      ║ 
q_3: ┤ X ├┤2   ├──────╫─
     └───┘└────┘      ║ 
c: 1/════

(0.4991455078125,
 0.5008544921875,
 -0.001708984375,
 np.float64(1.5725053120017822))

In [4]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Define a 3-qubit entangling unitary U_block
# -------------------------------------------------
def make_U_block(theta):
    """
    3-qubit entangling unitary U.
    Phase comes from Rz(theta) on qubit 1,
    sandwiched by CNOT layers.
    """
    U = QuantumCircuit(3, name="U_block")
    U.h(0)
    U.cx(0, 1)
    U.rz(theta, 1)
    U.cx(1, 2)
    return U

theta_true = 4 * np.pi / 5    # some non-trivial phase
U_block    = make_U_block(theta_true)
U_gate     = U_block.to_gate(label="U")
U_ctrl     = U_gate.control(1)   # controlled-U

print("U_block (3-qubit entangling circuit):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Hadamard-test helper for a given |phi>
# -------------------------------------------------
def hadamard_test_for_state(prep_phi, description):
    """
    prep_phi(qc, data) should prepare |phi> on 'data' qubits.
    Returns P(0), P(1), Re(<phi|U|phi>) and an effective angle.
    """
    n_data = 3
    anc = 0
    data = [1, 2, 3]

    qc = QuantumCircuit(1 + n_data, 1)

    # (A) Prepare |phi> on data qubits
    prep_phi(qc, data)

    # (B) Hadamard on ancilla
    qc.h(anc)

    # (C) Controlled-U on data
    qc.append(U_ctrl, [anc] + data)

    # (D) Hadamard on ancilla
    qc.h(anc)

    # (E) Measure ancilla
    qc.measure(anc, 0)

    print(f"\nHadamard-test circuit for state: {description}")
    print(qc.draw(output="text", fold=-1))

    # Run
    t_qc   = transpile(qc, sim)
    result = sim.run(t_qc, shots=SHOTS).result()
    counts = result.get_counts()

    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    p0 = n0 / SHOTS
    p1 = n1 / SHOTS

    # Standard Hadamard test: Re(<phi|U|phi>) = P(0) - P(1) = 2P(0) - 1
    re_expect = 2 * p0 - 1
    # If |phi> ≈ eigenstate, interpret as cos(theta_eff)
    arg = max(-1.0, min(1.0, re_expect))
    theta_eff = np.arccos(arg)

    print("Ancilla counts:", counts)
    print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
    print(f"Re(<phi|U|phi>) ≈ {re_expect:.4f}")
    print(f"θ_eff (from arccos(Re)) ≈ {theta_eff:.6f} rad  ({np.degrees(theta_eff):.2f}°)")
    return p0, p1, re_expect, theta_eff


# -------------------------------------------------
# 3) Generic basis-state preparation: |b2 b1 b0>
# -------------------------------------------------
def make_prep_basis(bitstring):
    """
    Returns a function prep_phi(qc, data) that prepares the
    computational basis state |bitstring> on 'data' qubits.
    bitstring like '000', '101', '111', etc.
    """
    bits = list(bitstring)

    def prep_phi(qc, data):
        # data[0] ↔ bits[0], data[1] ↔ bits[1], data[2] ↔ bits[2]
        for q_idx, b in zip(data, bits):
            if b == '1':
                qc.x(q_idx)

    return prep_phi


# -------------------------------------------------
# 4) Scan all 3-qubit computational basis states
#    |000>, |001>, ..., |111>
# -------------------------------------------------
for i in range(8):
    bitstring = format(i, "03b")   # '000' .. '111'
    prep_phi = make_prep_basis(bitstring)
    desc = f"|{bitstring}>"
    hadamard_test_for_state(prep_phi, desc)

U_block (3-qubit entangling circuit):
     ┌───┐                      
q_0: ┤ H ├──■───────────────────
     └───┘┌─┴─┐┌──────────┐     
q_1: ─────┤ X ├┤ Rz(4π/5) ├──■──
          └───┘└──────────┘┌─┴─┐
q_2: ──────────────────────┤ X ├
                           └───┘

Hadamard-test circuit for state: |000>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     └───┘┌─┴──┐└───┘└╥┘
q_1: ─────┤0   ├──────╫─
          │    │      ║ 
q_2: ─────┤1 U ├──────╫─
          │    │      ║ 
q_3: ─────┤2   ├──────╫─
          └────┘      ║ 
c: 1/═════════════════╩═
                      0 
Ancilla counts: {'1': 3122, '0': 5070}
P(0) = 0.6189, P(1) = 0.3811
Re(<phi|U|phi>) ≈ 0.2378
θ_eff (from arccos(Re)) ≈ 1.330703 rad  (76.24°)

Hadamard-test circuit for state: |001>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     └───┘┌─┴──┐└───┘└╥┘
q_1: ─────┤0   ├──────╫─
          │    │      ║ 
q_2: ─────┤1 U ├──────╫─
     ┌───┐│    │      ║ 
q_3: ┤ X ├┤2   ├──────╫─
     └───┘└────┘      ║ 
c: 1/════

In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Define a 3-qubit entangling unitary U_block
#    NO explicit phase gates, only H + CNOT
#    Example: GHZ-style entangler
# -------------------------------------------------
def make_U_block():
    """
    3-qubit entangling unitary U with NO explicit Z-phase gates.
    Starting from |000>, this prepares (|000> + |111>)/sqrt(2),
    i.e. equal real amplitudes, no complex relative phase.
    """
    U = QuantumCircuit(3, name="U_block")
    U.h(0)          # create superposition on q0
    U.cx(0, 1)      # entangle q0-q1
    U.cx(1, 2)      # entangle q1-q2
    return U

U_block = make_U_block()
U_gate  = U_block.to_gate(label="U")
U_ctrl  = U_gate.control(1)   # controlled-U

print("U_block (3-qubit entangling circuit, no Rz phase):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Hadamard-test helper for a given |phi>
# -------------------------------------------------
def hadamard_test_for_state(prep_phi, description):
    """
    prep_phi(qc, data) should prepare |phi> on 'data' qubits.
    Returns P(0), P(1), Re(<phi|U|phi>) and an effective angle.
    """
    n_data = 3
    anc = 0
    data = [1, 2, 3]

    qc = QuantumCircuit(1 + n_data, 1)

    # (A) Prepare |phi> on data qubits
    prep_phi(qc, data)

    # (B) Hadamard on ancilla
    qc.h(anc)

    # (C) Controlled-U on data
    qc.append(U_ctrl, [anc] + data)

    # (D) Hadamard on ancilla
    qc.h(anc)

    # (E) Measure ancilla
    qc.measure(anc, 0)

    print(f"\nHadamard-test circuit for state: {description}")
    print(qc.draw(output="text", fold=-1))

    # Run
    t_qc   = transpile(qc, sim)
    result = sim.run(t_qc, shots=SHOTS).result()
    counts = result.get_counts()

    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    p0 = n0 / SHOTS
    p1 = n1 / SHOTS

    # Standard Hadamard test: Re(<phi|U|phi>) = P(0) - P(1) = 2P(0) - 1 [web:73][web:125]
    re_expect = 2 * p0 - 1
    # If |phi> ≈ eigenstate, interpret as cos(theta_eff)
    arg = max(-1.0, min(1.0, re_expect))
    theta_eff = np.arccos(arg)

    print("Ancilla counts:", counts)
    print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
    print(f"Re(<phi|U|phi>) ≈ {re_expect:.4f}")
    print(f"θ_eff (from arccos(Re)) ≈ {theta_eff:.6f} rad  ({np.degrees(theta_eff):.2f}°)")
    return p0, p1, re_expect, theta_eff


# -------------------------------------------------
# 3) Generic basis-state preparation: |b2 b1 b0>
# -------------------------------------------------
def make_prep_basis(bitstring):
    """
    Returns a function prep_phi(qc, data) that prepares the
    computational basis state |bitstring> on 'data' qubits.
    bitstring like '000', '101', '111', etc.
    """
    bits = list(bitstring)

    def prep_phi(qc, data):
        # data[0] ↔ bits[0], data[1] ↔ bits[1], data[2] ↔ bits[2]
        for q_idx, b in zip(data, bits):
            if b == '1':
                qc.x(q_idx)

    return prep_phi


# -------------------------------------------------
# 4) Scan all 3-qubit computational basis states
#    |000>, |001>, ..., |111>
# -------------------------------------------------
for i in range(8):
    bitstring = format(i, "03b")   # '000' .. '111'
    prep_phi = make_prep_basis(bitstring)
    desc = f"|{bitstring}>"
    hadamard_test_for_state(prep_phi, desc)

U_block (3-qubit entangling circuit, no Rz phase):
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘

Hadamard-test circuit for state: |000>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     └───┘┌─┴──┐└───┘└╥┘
q_1: ─────┤0   ├──────╫─
          │    │      ║ 
q_2: ─────┤1 U ├──────╫─
          │    │      ║ 
q_3: ─────┤2   ├──────╫─
          └────┘      ║ 
c: 1/═════════════════╩═
                      0 
Ancilla counts: {'1': 1194, '0': 6998}
P(0) = 0.8542, P(1) = 0.1458
Re(<phi|U|phi>) ≈ 0.7085
θ_eff (from arccos(Re)) ≈ 0.783431 rad  (44.89°)

Hadamard-test circuit for state: |001>
     ┌───┐      ┌───┐┌─┐
q_0: ┤ H ├──■───┤ H ├┤M├
     └───┘┌─┴──┐└───┘└╥┘
q_1: ─────┤0   ├──────╫─
          │    │      ║ 
q_2: ─────┤1 U ├──────╫─
     ┌───┐│    │      ║ 
q_3: ┤ X ├┤2   ├──────╫─
     └───┘└────┘      ║ 
c: 1/═════════════════╩═
                      0 
Ancilla counts: {'1': 1154, '0

In [7]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192
TOL = 3 / np.sqrt(SHOTS)   # ~3σ threshold for |<Y>|

# 1) 3-qubit entangling U with NO explicit phase gates
def make_U_block_no_phase():
    U = QuantumCircuit(3, name="U_block")
    U.h(0)
    U.cx(0, 1)
    U.cx(1, 2)
    return U

U_block = make_U_block_no_phase()
print("U_block (no explicit phase gates):")
print(U_block.draw(output="text", fold=-1))

# 2) Prepare input basis state |b2 b1 b0>, then apply U_block
def make_input_circuit(bitstring):
    qc = QuantumCircuit(3)
    for q, b in enumerate(bitstring):
        if b == "1":
            qc.x(q)
    qc.compose(U_block, inplace=True)
    return qc

# 3) Run a given circuit, measure one qubit in Y basis, return <Y_q>
def run_and_expectation_Y(qc_quantum, target_qubit):
    """
    qc_quantum: QuantumCircuit with only qubits (no classical reg, no measure)
    target_qubit: index (0,1,2)
    """
    n = qc_quantum.num_qubits

    # New circuit with n qubits + n classical bits
    qc = QuantumCircuit(n, n)

    # Copy quantum part
    qc.compose(qc_quantum, inplace=True)

    # Y-basis on target_qubit: Sdg then H, then Z-measure
    qc.sdg(target_qubit)
    qc.h(target_qubit)

    # Measure all qubits in Z
    qc.measure(range(n), range(n))

    # Simulate
    t = transpile(qc, sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    # Marginal on target_qubit: P(bit_i = 0/1)
    p0 = 0.0
    for bitstr, c in counts.items():
        # bitstr order: c_{n-1}...c_0
        bit = bitstr[-1 - target_qubit]
        if bit == "0":
            p0 += c
    p0 /= SHOTS
    p1 = 1 - p0

    # Expectation value of Y on that qubit
    return p0 - p1

# 4) Scan all basis states and compute <Y_i> per qubit
for i in range(8):
    bitstring = format(i, "03b")  # '000'..'111'
    qc_in = make_input_circuit(bitstring)
    print(f"\nInput state |{bitstring}>:")

    for q in range(3):
        expY = run_and_expectation_Y(qc_in, q)
        flag = "PHASE DETECTED (local)" if abs(expY) > TOL else "no relative phase (locally)"
        print(f"  qubit {q}: <Y> ≈ {expY:.4f}  → {flag}")

U_block (no explicit phase gates):
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘

Input state |000>:
  qubit 0: <Y> ≈ 0.0054  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0032  → no relative phase (locally)
  qubit 2: <Y> ≈ 0.0059  → no relative phase (locally)

Input state |001>:
  qubit 0: <Y> ≈ 0.0076  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0063  → no relative phase (locally)
  qubit 2: <Y> ≈ -0.0154  → no relative phase (locally)

Input state |010>:
  qubit 0: <Y> ≈ -0.0059  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0281  → no relative phase (locally)
  qubit 2: <Y> ≈ -0.0115  → no relative phase (locally)

Input state |011>:
  qubit 0: <Y> ≈ -0.0088  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0071  → no relative phase (locally)
  qubit 2: <Y> ≈ 0.0088  → no relative phase (locally)

Input state |100>:
  qubit 0: <Y> ≈ 0.0046  → no relative phase (locally)

In [8]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192
TOL = 3 / np.sqrt(SHOTS)   # ~3σ threshold for |<Y>|

# -------------------------------------------------
# 1) Your 3-qubit entangling unitary U_block with Rz phase
# -------------------------------------------------
def make_U_block(theta):
    """
    3-qubit entangling unitary U.
    Phase comes from Rz(theta) on qubit 1,
    sandwiched by CNOT layers.
    """
    U = QuantumCircuit(3, name="U_block")
    U.h(0)
    U.cx(0, 1)
    U.rz(theta, 1)
    U.cx(1, 2)
    return U

theta_true = 4 * np.pi / 5    # some non-trivial phase
U_block    = make_U_block(theta_true)

print("U_block (3-qubit entangling circuit with Rz phase):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Prepare input basis state |b2 b1 b0>, then apply U_block
# -------------------------------------------------
def make_input_circuit(bitstring):
    qc = QuantumCircuit(3)
    # prepare |b2 b1 b0> on q0,q1,q2 (your earlier convention)
    for q, b in enumerate(bitstring):
        if b == "1":
            qc.x(q)
    qc.compose(U_block, inplace=True)
    return qc


# -------------------------------------------------
# 3) Measure Y on a chosen qubit via basis rotation + Z-measure
# -------------------------------------------------
def run_and_expectation_Y(qc_quantum, target_qubit):
    """
    qc_quantum: QuantumCircuit with only qubits (no classical reg, no measure)
    target_qubit: index (0,1,2)
    Returns <Y_target_qubit> estimated from measurement statistics.
    """
    n = qc_quantum.num_qubits

    # New circuit with n qubits + n classical bits
    qc = QuantumCircuit(n, n)

    # Copy quantum part
    qc.compose(qc_quantum, inplace=True)

    # Y-basis on target_qubit: Sdg then H, then Z-measure [web:39]
    qc.sdg(target_qubit)
    qc.h(target_qubit)

    # Measure all qubits in Z
    qc.measure(range(n), range(n))

    # Simulate
    t = transpile(qc, sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    # Marginal on target_qubit: P(bit_i = 0/1)
    p0 = 0.0
    for bitstr, c in counts.items():
        # bitstr order: c_{n-1}...c_0
        bit = bitstr[-1 - target_qubit]
        if bit == "0":
            p0 += c
    p0 /= SHOTS
    p1 = 1 - p0

    # Expectation value of Y on that qubit
    return p0 - p1


# -------------------------------------------------
# 4) Scan all 3-qubit computational basis states
#    |000>, |001>, ..., |111>, and compute <Y_i>
# -------------------------------------------------
for i in range(8):
    bitstring = format(i, "03b")   # '000' .. '111'
    qc_in = make_input_circuit(bitstring)
    print(f"\nInput state |{bitstring}>:")

    for q in range(3):
        expY = run_and_expectation_Y(qc_in, q)
        flag = "PHASE DETECTED (local)" if abs(expY) > TOL else "no relative phase (locally)"
        print(f"  qubit {q}: <Y> ≈ {expY:.4f}  → {flag}")

U_block (3-qubit entangling circuit with Rz phase):
     ┌───┐                      
q_0: ┤ H ├──■───────────────────
     └───┘┌─┴─┐┌──────────┐     
q_1: ─────┤ X ├┤ Rz(4π/5) ├──■──
          └───┘└──────────┘┌─┴─┐
q_2: ──────────────────────┤ X ├
                           └───┘

Input state |000>:
  qubit 0: <Y> ≈ -0.0066  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0027  → no relative phase (locally)
  qubit 2: <Y> ≈ -0.0073  → no relative phase (locally)

Input state |001>:
  qubit 0: <Y> ≈ -0.0178  → no relative phase (locally)
  qubit 1: <Y> ≈ -0.0037  → no relative phase (locally)
  qubit 2: <Y> ≈ -0.0085  → no relative phase (locally)

Input state |010>:
  qubit 0: <Y> ≈ -0.0061  → no relative phase (locally)
  qubit 1: <Y> ≈ -0.0232  → no relative phase (locally)
  qubit 2: <Y> ≈ 0.0012  → no relative phase (locally)

Input state |011>:
  qubit 0: <Y> ≈ 0.0073  → no relative phase (locally)
  qubit 1: <Y> ≈ 0.0078  → no relative phase (locally)
  qubit 2: <Y> ≈ 0.0015  

In [13]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Your 3-qubit entangling unitary U_block with Rz phase
# -------------------------------------------------
def make_U_block(theta):
    """
    3-qubit entangling unitary U.
    Phase comes from Rz(theta) on qubit 1,
    sandwiched by CNOT layers.
    """
    U = QuantumCircuit(3, name="U_block")
    U.h(0)
    U.cx(0, 1)
    U.rz(theta, 1)
    U.cx(1, 2)
    return U

theta_true = 4 * np.pi / 5    # some non-trivial phase
U_block    = make_U_block(theta_true)

print("U_block (3-qubit entangling circuit with Rz phase):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Prepare |000> and apply U_block
# -------------------------------------------------
def make_psi000_circuit():
    qc = QuantumCircuit(3)
    # start in |000>, just apply U_block
    qc.compose(U_block, inplace=True)
    return qc


# -------------------------------------------------
# 3) Expectation of a 3-qubit Pauli string P ∈ {XXX, YYY}
#    using basis rotations + Z measurement (measurement-only)
# -------------------------------------------------
def measure_pauli_string(qc_state, pauli_string):
    """
    qc_state: QuantumCircuit that prepares the 3-qubit state (no classical bits, no measure)
    pauli_string: string like "XXX" or "YYY" (length 3)
    Returns <P> estimated from SHOTS.
    """
    n = qc_state.num_qubits
    assert n == 3
    assert len(pauli_string) == n

    # New circuit with n qubits + n classical bits
    qc = QuantumCircuit(n, n)
    qc.compose(qc_state, inplace=True)

    # For each qubit, add basis rotation to diagonalize the Pauli:
    # X: H;  Y: Sdg then H;  Z: nothing (we only use X,Y here)
    for q, P in enumerate(pauli_string):
        if P == "X":
            qc.h(q)
        elif P == "Y":
            qc.sdg(q)
            qc.h(q)
        elif P == "Z":
            pass
        else:
            raise ValueError("Unsupported Pauli: " + P)

    # Measure all qubits in Z basis
    qc.measure(range(n), range(n))

    # Simulate
    t = transpile(qc, sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    # For each bitstring result, eigenvalue = product of eigenvalues on each qubit
    # After basis rotation, Z outcome '0' -> +1, '1' -> -1, so eigenvalue = (-1)^{# of 1s}
    exp_val = 0.0
    for bitstr, c in counts.items():
        # bitstr order: c2 c1 c0
        parity = bitstr.count("1")
        eigenvalue = (-1) ** parity
        exp_val += eigenvalue * c

    exp_val /= SHOTS
    return exp_val


# -------------------------------------------------
# 4) Run for |000>, measure <XXX> and <YYY>, estimate relative phase φ
# -------------------------------------------------
qc_psi = make_psi000_circuit()

exp_xxx = measure_pauli_string(qc_psi, "XXX")
exp_yyy = measure_pauli_string(qc_psi, "YYY")

print("\nMeasured Pauli expectations for |ψ> = U|000> :")
print(f"<XXX> ≈ {exp_xxx:.4f}")
print(f"<YYY> ≈ {exp_yyy:.4f}")

# For GHZ-type state (|000> + e^{iφ}|111>)/√2, theory gives:
#   <XXX> = cos φ,  <YYY> = -sin φ  [web:148]
# so we can estimate φ as:
phi_est = np.arctan2(-exp_yyy, exp_xxx)

print("\nEstimated relative phase between |000> and |111| components:")
print(f"  φ_true  = {theta_true:.6f} rad  ({np.degrees(theta_true):.2f}°)")
print(f"  φ_est   = {phi_est:.6f} rad  ({np.degrees(phi_est):.2f}°)")

U_block (3-qubit entangling circuit with Rz phase):
     ┌───┐                      
q_0: ┤ H ├──■───────────────────
     └───┘┌─┴─┐┌──────────┐     
q_1: ─────┤ X ├┤ Rz(4π/5) ├──■──
          └───┘└──────────┘┌─┴─┐
q_2: ──────────────────────┤ X ├
                           └───┘

Measured Pauli expectations for |ψ> = U|000> :
<XXX> ≈ -0.8105
<YYY> ≈ -0.5825

Estimated relative phase between |000> and |111| components:
  φ_true  = 2.513274 rad  (144.00°)
  φ_est   = 2.518443 rad  (144.30°)


In [14]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) New 3-qubit entangling circuit with NO Rz phase
#    Standard GHZ: (|000> + |111>)/√2, all amplitudes real and ≥ 0
# -------------------------------------------------
def make_U_block_no_phase():
    """
    3-qubit entangling unitary with no complex relative phase.
    H on qubit 0, then CNOT(0->1), CNOT(0->2).
    """
    U = QuantumCircuit(3, name="U_block_no_phase")
    U.h(0)
    U.cx(0, 1)
    U.cx(0, 2)
    return U

U_block_no_phase = make_U_block_no_phase()

print("U_block_no_phase (3-qubit entangling circuit, no complex relative phase):")
print(U_block_no_phase.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Prepare |000> and apply U_block_no_phase
# -------------------------------------------------
def make_psi000_no_phase():
    qc = QuantumCircuit(3)
    qc.compose(U_block_no_phase, inplace=True)
    return qc


# -------------------------------------------------
# 3) Same Pauli-string measurement routine as before
# -------------------------------------------------
def measure_pauli_string(qc_state, pauli_string):
    """
    qc_state: circuit preparing the 3-qubit state (no classical bits, no measure)
    pauli_string: string like "XXX" or "YYY"
    Returns expectation value <P>.
    """
    n = qc_state.num_qubits
    assert n == 3
    assert len(pauli_string) == n

    qc = QuantumCircuit(n, n)
    qc.compose(qc_state, inplace=True)

    for q, P in enumerate(pauli_string):
        if P == "X":
            qc.h(q)
        elif P == "Y":
            qc.sdg(q)
            qc.h(q)
        elif P == "Z":
            pass
        else:
            raise ValueError("Unsupported Pauli: " + P)

    qc.measure(range(n), range(n))

    t = transpile(qc, sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    exp_val = 0.0
    for bitstr, c in counts.items():
        # Z-basis measurement: '0' -> +1, '1' -> -1
        parity = bitstr.count("1")
        eigenvalue = (-1) ** parity
        exp_val += eigenvalue * c

    exp_val /= SHOTS
    return exp_val


# -------------------------------------------------
# 4) Run for |000>, measure <XXX> and <YYY>, estimate phase
# -------------------------------------------------
qc_psi_no_phase = make_psi000_no_phase()

exp_xxx = measure_pauli_string(qc_psi_no_phase, "XXX")
exp_yyy = measure_pauli_string(qc_psi_no_phase, "YYY")

print("\nMeasured Pauli expectations for |ψ_no_phase> = U_block_no_phase|000> :")
print(f"<XXX> ≈ {exp_xxx:.4f}")
print(f"<YYY> ≈ {exp_yyy:.4f}")

# For GHZ-type (|000> + e^{iφ}|111>)/√2, theory:
#   <XXX> = cos φ,  <YYY> = -sin φ  [web:148][web:162]
phi_est = np.arctan2(-exp_yyy, exp_xxx)

print("\nEstimated relative phase between |000> and |111| components:")
print(f"  φ_est ≈ {phi_est:.6f} rad  ({np.degrees(phi_est):.2f}°)")

U_block_no_phase (3-qubit entangling circuit, no complex relative phase):
     ┌───┐          
q_0: ┤ H ├──■────■──
     └───┘┌─┴─┐  │  
q_1: ─────┤ X ├──┼──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘

Measured Pauli expectations for |ψ_no_phase> = U_block_no_phase|000> :
<XXX> ≈ 1.0000
<YYY> ≈ -0.0117

Estimated relative phase between |000> and |111| components:
  φ_est ≈ 0.011718 rad  (0.67°)


In [15]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) Different entangling circuit: 2-qubit Bell, no complex phase
#     |00> --H0--> (|00>+|10>)/√2 --CX0->1--> (|00>+|11>)/√2
# -------------------------------------------------
def make_bell_no_phase():
    U = QuantumCircuit(2, name="Bell_no_phase")
    U.h(0)
    U.cx(0, 1)
    return U

U_bell = make_bell_no_phase()
print(U_bell.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Prepare |00> and apply Bell circuit
# -------------------------------------------------
def prep_state_bell_no_phase():
    qc = QuantumCircuit(2)
    qc.compose(U_bell, inplace=True)
    return qc


# -------------------------------------------------
# 3) Generic 2-qubit Pauli-string measurement (X/Y/Z)
# -------------------------------------------------
def measure_pauli_string_2q(qc_state, pauli_string):
    """
    qc_state: circuit that prepares a 2-qubit state (no classical bits)
    pauli_string: e.g. "XX", "XY", "XZ", "YY", etc.
    Returns expectation value <P>.
    """
    assert qc_state.num_qubits == 2
    assert len(pauli_string) == 2

    qc = QuantumCircuit(2, 2)
    qc.compose(qc_state, inplace=True)

    # Basis rotations: X -> H; Y -> Sdg+H; Z -> I
    for q, P in enumerate(pauli_string):
        if P == "X":
            qc.h(q)
        elif P == "Y":
            qc.sdg(q)
            qc.h(q)
        elif P == "Z":
            pass
        else:
            raise ValueError("Unsupported Pauli " + P)

    qc.measure([0, 1], [0, 1])

    t = transpile(qc, backend=sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    exp_val = 0.0
    for bitstr, c in counts.items():
        # bitstr like '10' (c1 c0). '0' -> +1, '1' -> -1
        parity = bitstr.count("1")
        eigenvalue = (-1) ** parity
        exp_val += eigenvalue * c

    exp_val /= SHOTS
    return exp_val


# -------------------------------------------------
# 4) Measure <XX> and <XY>, estimate phase φ
# -------------------------------------------------
qc_psi = prep_state_bell_no_phase()

exp_xx = measure_pauli_string_2q(qc_psi, "XX")
exp_xy = measure_pauli_string_2q(qc_psi, "XY")

print("\nMeasured correlators for |ψ> = Bell_no_phase |00> :")
print(f"<XX> ≈ {exp_xx:.4f}")
print(f"<XY> ≈ {exp_xy:.4f}")

# For (|00> + e^{iφ}|11>)/√2 :
#   <XX> = cos φ,  <XY> = sin φ   [web:148]
phi_est = np.arctan2(exp_xy, exp_xx)

print("\nEstimated relative phase between |00> and |111> components:")
print(f"  φ_est ≈ {phi_est:.6f} rad  ({np.degrees(phi_est):.2f}°)")

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Measured correlators for |ψ> = Bell_no_phase |00> :
<XX> ≈ 1.0000
<XY> ≈ -0.0015

Estimated relative phase between |00> and |111> components:
  φ_est ≈ -0.001465 rad  (-0.08°)


In [16]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) 3-qubit circuits with support on all 8 states
# -------------------------------------------------
def make_cluster_no_phase():
    """
    |000> --H⊗3--> uniform over 8 states,
            --CZ(0,1), CZ(1,2)--> 3-qubit cluster state with only ± real amplitudes.
    """
    qc = QuantumCircuit(3, name="Cluster_no_phase")
    qc.h(0); qc.h(1); qc.h(2)
    qc.cz(0, 1)
    qc.cz(1, 2)
    return qc

def make_cluster_with_phase():
    """
    Same as above, but add S on qubit 1
    to introduce a complex phase i on all basis states with q1 = 1.
    """
    qc = make_cluster_no_phase()
    qc.s(1)   # true complex relative phase
    qc.name = "Cluster_with_phase"
    return qc


# -------------------------------------------------
# 2) Generic 3-qubit Pauli-string measurement
# -------------------------------------------------
def measure_pauli_string_3q(qc_state, pauli_string):
    """
    qc_state: circuit preparing a 3-qubit state (no classical bits, no measure).
    pauli_string: length-3 string in {I,X,Y,Z}, e.g. 'XXY', 'YXI'.
    Returns <P> estimated from SHOTS, using basis rotations + Z-measurements.
    """
    n = qc_state.num_qubits
    assert n == 3
    assert len(pauli_string) == 3

    qc = QuantumCircuit(3, 3)
    qc.compose(qc_state, inplace=True)

    # Basis rotations: X -> H, Y -> Sdg+H, Z/I -> identity
    for q, P in enumerate(pauli_string):
        if P == 'X':
            qc.h(q)
        elif P == 'Y':
            qc.sdg(q)
            qc.h(q)
        elif P in ['Z', 'I']:
            pass
        else:
            raise ValueError("Unsupported Pauli " + P)

    qc.measure(range(3), range(3))

    t = transpile(qc, backend=sim)
    counts = sim.run(t, shots=SHOTS).result().get_counts()

    exp_val = 0.0
    for bitstr, c in counts.items():
        # bitstr is c2 c1 c0; '0' -> +1, '1' -> -1
        parity = bitstr.count("1")
        eigenvalue = (-1)**parity
        exp_val += eigenvalue * c

    return exp_val / SHOTS


# -------------------------------------------------
# 3) Compare circuits with and without complex phase
#    using odd-Y Pauli strings
# -------------------------------------------------
oddY_strings = ["XXY", "XYX", "YXX"]  # each has exactly one Y

for label, make_qc in [
    ("NO_COMPLEX_PHASE", make_cluster_no_phase),
    ("WITH_COMPLEX_PHASE", make_cluster_with_phase),
]:
    qc_state = make_qc()
    print(f"\nCircuit: {label}")
    print(qc_state.draw(output="text", fold=-1))

    for P in oddY_strings:
        ev = measure_pauli_string_3q(qc_state, P)
        print(f"  <{P}> ≈ {ev:.4f}")


Circuit: NO_COMPLEX_PHASE
     ┌───┐      
q_0: ┤ H ├─■────
     ├───┤ │    
q_1: ┤ H ├─■──■─
     ├───┤    │ 
q_2: ┤ H ├────■─
     └───┘      
  <XXY> ≈ 0.0012
  <XYX> ≈ -0.0078
  <YXX> ≈ 0.0100

Circuit: WITH_COMPLEX_PHASE
     ┌───┐           
q_0: ┤ H ├─■─────────
     ├───┤ │    ┌───┐
q_1: ┤ H ├─■──■─┤ S ├
     ├───┤    │ └───┘
q_2: ┤ H ├────■──────
     └───┘           
  <XXY> ≈ -0.0051
  <XYX> ≈ -0.0222
  <YXX> ≈ -0.0125


In [27]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
SHOTS = 8192

# -------------------------------------------------
# 1) NEW U_block: multiple qubit phase operations
#    Rz on every qubit at different angles + T gate
#    All 8 basis states populated with complex phases
# -------------------------------------------------
def make_U_block():
    U = QuantumCircuit(3, name="U_multi_phase")

    # Layer 1: superposition
    U.h(0); U.h(1); U.h(2)

    # Layer 2: individual Rz on each qubit (different angles)
    U.rz(np.pi / 3, 0)
    U.rz(np.pi / 5, 1)
    U.rz(np.pi / 7, 2)

    # Layer 3: entangling
    U.cx(0, 1)
    U.cx(1, 2)
    U.cx(2, 0)

    # Layer 4: T-gate phase on q1
    U.t(1)

    # Layer 5: more Rz phases
    U.rz(np.pi / 4, 0)
    U.rz(np.pi / 6, 2)

    # Layer 6: second entangling round
    U.cx(0, 1)
    U.cx(1, 2)

    return U

U_block = make_U_block()
U_gate  = U_block.to_gate(label="U_multi_phase")
U_ctrl  = U_gate.control(1)

print("U_block (multiple qubit phase operations):")
print(U_block.draw(output="text", fold=-1))


# -------------------------------------------------
# 2) Hadamard-test helper — EXACTLY YOUR ORIGINAL
# -------------------------------------------------
def hadamard_test_for_state(prep_phi, description):
    n_data = 3
    anc    = 0
    data   = [1, 2, 3]

    qc = QuantumCircuit(1 + n_data, 1)

    # (A) Prepare |phi>
    prep_phi(qc, data)

    # (B) Hadamard on ancilla
    qc.h(anc)

    # (C) Controlled-U
    qc.append(U_ctrl, [anc] + data)

    # (D) Hadamard on ancilla
    qc.h(anc)

    # (E) Measure ancilla
    qc.measure(anc, 0)

    print(f"\nHadamard-test circuit for state: {description}")
    print(qc.draw(output="text", fold=-1))

    t_qc   = transpile(qc, sim)
    result = sim.run(t_qc, shots=SHOTS).result()
    counts = result.get_counts()

    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    p0 = n0 / SHOTS
    p1 = n1 / SHOTS

    re_expect = 2 * p0 - 1
    arg       = max(-1.0, min(1.0, re_expect))
    theta_eff = np.arccos(arg)

    print("Ancilla counts:", counts)
    print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
    print(f"Re(<phi|U|phi>) ≈ {re_expect:.4f}")
    print(f"θ_eff (from arccos(Re)) ≈ {theta_eff:.6f} rad  ({np.degrees(theta_eff):.2f}°)")
    return p0, p1, re_expect, theta_eff


# -------------------------------------------------
# 3) Basis-state preparation — EXACTLY YOUR ORIGINAL
# -------------------------------------------------
def make_prep_basis(bitstring):
    bits = list(bitstring)
    def prep_phi(qc, data):
        for q_idx, b in zip(data, bits):
            if b == '1':
                qc.x(q_idx)
    return prep_phi


# -------------------------------------------------
# 4) Scan all 8 basis states — EXACTLY YOUR ORIGINAL
# -------------------------------------------------
for i in range(8):
    bitstring = format(i, "03b")
    prep_phi  = make_prep_basis(bitstring)
    desc      = f"|{bitstring}>"
    hadamard_test_for_state(prep_phi, desc)

U_block (multiple qubit phase operations):
     ┌───┐┌─────────┐          ┌───┐┌─────────┐          
q_0: ┤ H ├┤ Rz(π/3) ├──■───────┤ X ├┤ Rz(π/4) ├──■───────
     ├───┤├─────────┤┌─┴─┐     └─┬─┘└──┬───┬──┘┌─┴─┐     
q_1: ┤ H ├┤ Rz(π/5) ├┤ X ├──■────┼─────┤ T ├───┤ X ├──■──
     ├───┤├─────────┤└───┘┌─┴─┐  │  ┌──┴───┴──┐└───┘┌─┴─┐
q_2: ┤ H ├┤ Rz(π/7) ├─────┤ X ├──■──┤ Rz(π/6) ├─────┤ X ├
     └───┘└─────────┘     └───┘     └─────────┘     └───┘

Hadamard-test circuit for state: |000>
     ┌───┐                  ┌───┐┌─┐
q_0: ┤ H ├────────■─────────┤ H ├┤M├
     └───┘┌───────┴────────┐└───┘└╥┘
q_1: ─────┤0               ├──────╫─
          │                │      ║ 
q_2: ─────┤1 U_multi_phase ├──────╫─
          │                │      ║ 
q_3: ─────┤2               ├──────╫─
          └────────────────┘      ║ 
c: 1/═════════════════════════════╩═
                                  0 
Ancilla counts: {'1': 4272, '0': 3920}
P(0) = 0.4785, P(1) = 0.5215
Re(<phi|U|phi>) ≈ -0.0430
θ_eff (fro

## 13.03.2026

In [14]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from math import pi
import numpy as np

def detect_relative_phase_by_measurement(circuit: QuantumCircuit, shots=8192, atol=1e-3):
    """
    Generic method: Takes ANY circuit, modifies it for measurement,
    and detects if relative phase exists.
    
    Returns: dict with detection result and measurement data
    """
    sim = AerSimulator()
    n = circuit.num_qubits
    
    # --- Step 1: Z-basis measurement (reference: magnitudes only) ---
    qc_z = circuit.copy()
    qc_z.measure_all()
    
    t_qc = transpile(qc_z, sim)
    counts_z = sim.run(t_qc, shots=shots).result().get_counts()
    
    # Convert to probabilities
    total = sum(counts_z.values())
    probs_z = {k: v/total for k, v in sorted(counts_z.items())}
    
    # --- Step 2: X-basis measurement (detects phase via interference) ---
    qc_x = circuit.copy()
    qc_x.h(range(n))  # Rotate to X-basis
    qc_x.measure_all()
    
    t_qc = transpile(qc_x, sim)
    counts_x = sim.run(t_qc, shots=shots).result().get_counts()
    
    probs_x = {k: v/total for k, v in sorted(counts_x.items())}
    
    # --- Step 3: Theoretical "no-phase" X distribution for comparison ---
    # Get statevector to compute what X-basis would look like WITHOUT phase
    sv = Statevector(circuit)
    amplitudes = sv.data
    
    # Create "no-phase" version: same magnitudes, all phases set to 0
    no_phase_amplitudes = np.abs(amplitudes)
    no_phase_amplitudes = no_phase_amplitudes / np.linalg.norm(no_phase_amplitudes)
    sv_no_phase = Statevector(no_phase_amplitudes)
    
    # Apply H and get probabilities for no-phase state
    qc_no_phase = QuantumCircuit(n)
    qc_no_phase.initialize(sv_no_phase)
    qc_no_phase.h(range(n))
    sv_no_phase_x = Statevector(qc_no_phase)
    probs_x_no_phase = sv_no_phase_x.probabilities_dict()
    
    # --- Step 4: Compare actual X-measurement vs no-phase X-measurement ---
    # Calculate statistical distance
    keys = set(probs_x.keys()) | set(probs_x_no_phase.keys())
    distance = sum(abs(probs_x.get(k, 0) - probs_x_no_phase.get(k, 0)) for k in keys) / 2
    
    has_relative_phase = distance > atol
    
    return {
        'has_relative_phase': has_relative_phase,
        'distance': distance,
        'Z_basis_probs': probs_z,
        'X_basis_probs': probs_x,
        'X_basis_if_no_phase': probs_x_no_phase,
        'circuit_z': qc_z,
        'circuit_x': qc_x
    }


# ─────────────────────────────────────────────
# TEST CASE 1: Circuit WITH relative phase
# ─────────────────────────────────────────────
print("="*60)
print("TEST 1: Circuit WITH relative phase")
print("="*60)

qc_phase = QuantumCircuit(2)
qc_phase.h(0)
qc_phase.cx(0, 1)
qc_phase.rz(pi/2, 1)  # Inject relative phase on |11>

print("Input circuit:")
print(qc_phase.draw())

result = detect_relative_phase_by_measurement(qc_phase)

print(f"\nRelative phase detected: {result['has_relative_phase']}")
print(f"Statistical distance: {result['distance']:.4f}")

print("\nZ-basis (computational) measurement:")
for state, prob in sorted(result['Z_basis_probs'].items()):
    print(f"  |{state}⟩: {prob:.4f}")

print("\nX-basis measurement (actual):")
for state, prob in sorted(result['X_basis_probs'].items()):
    print(f"  |{state}⟩: {prob:.4f}")

print("\nX-basis measurement (if NO relative phase):")
for state, prob in sorted(result['X_basis_if_no_phase'].items()):
    print(f"  |{state}⟩: {prob:.4f}")


# ─────────────────────────────────────────────
# TEST CASE 2: Circuit WITHOUT relative phase
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("TEST 2: Circuit WITHOUT relative phase")
print("="*60)

qc_no_phase = QuantumCircuit(2)
qc_no_phase.h(0)
qc_no_phase.cx(0, 1)  # Pure GHZ, no phase

print("Input circuit:")
print(qc_no_phase.draw())

result2 = detect_relative_phase_by_measurement(qc_no_phase)

print(f"\nRelative phase detected: {result2['has_relative_phase']}")
print(f"Statistical distance: {result2['distance']:.4f}")

print("\nX-basis measurement (actual):")
for state, prob in sorted(result2['X_basis_probs'].items()):
    print(f"  |{state}⟩: {prob:.4f}")

print("\nX-basis measurement (if NO relative phase):")
for state, prob in sorted(result2['X_basis_if_no_phase'].items()):
    print(f"  |{state}⟩: {prob:.4f}")

TEST 1: Circuit WITH relative phase
Input circuit:
     ┌───┐                
q_0: ┤ H ├──■─────────────
     └───┘┌─┴─┐┌─────────┐
q_1: ─────┤ X ├┤ Rz(π/2) ├
          └───┘└─────────┘

Relative phase detected: True
Statistical distance: 0.5054

Z-basis (computational) measurement:
  |00⟩: 0.4949
  |11⟩: 0.5051

X-basis measurement (actual):
  |00⟩: 0.2452
  |01⟩: 0.2561
  |10⟩: 0.2493
  |11⟩: 0.2494

X-basis measurement (if NO relative phase):
  |00⟩: 0.5000
  |11⟩: 0.5000

TEST 2: Circuit WITHOUT relative phase
Input circuit:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

Relative phase detected: True
Statistical distance: 0.0033

X-basis measurement (actual):
  |00⟩: 0.5033
  |11⟩: 0.4967

X-basis measurement (if NO relative phase):
  |00⟩: 0.5000
  |11⟩: 0.5000


In [17]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import pi

sim = AerSimulator()

def measure(circuit, basis):
    qc = circuit.copy()
    if basis == 'x':
        qc.h(range(qc.num_qubits))
    qc.measure_all()
    counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
    return {k: round(v/4096, 3) for k, v in sorted(counts.items())}

# WITH relative phase
qc_phase = QuantumCircuit(3)
qc_phase.h([0,1,2])
qc_phase.crz(pi/2, 0, 1)
qc_phase.crz(pi/3, 1, 2)
qc_phase.crz(pi/4, 0, 2)

# WITHOUT relative phase
qc_no = QuantumCircuit(3)
qc_no.h([0,1,2])

print("=== WITH relative phase ===")
print(qc_phase.draw())
print("Z-basis:", measure(qc_phase, 'z'))
print("X-basis:", measure(qc_phase, 'x'))

print("\n=== WITHOUT relative phase ===")
print(qc_no.draw())
print("Z-basis:", measure(qc_no, 'z'))
print("X-basis:", measure(qc_no, 'x'))

=== WITH relative phase ===
     ┌───┐                                 
q_0: ┤ H ├─────■─────────────────────■─────
     ├───┤┌────┴────┐                │     
q_1: ┤ H ├┤ Rz(π/2) ├─────■──────────┼─────
     ├───┤└─────────┘┌────┴────┐┌────┴────┐
q_2: ┤ H ├───────────┤ Rz(π/3) ├┤ Rz(π/4) ├
     └───┘           └─────────┘└─────────┘
Z-basis: {'000': 0.129, '001': 0.124, '010': 0.125, '011': 0.124, '100': 0.125, '101': 0.125, '110': 0.125, '111': 0.125}
X-basis: {'000': 0.539, '001': 0.041, '010': 0.081, '011': 0.079, '100': 0.116, '101': 0.015, '110': 0.084, '111': 0.045}

=== WITHOUT relative phase ===
     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     └───┘
Z-basis: {'000': 0.132, '001': 0.124, '010': 0.121, '011': 0.128, '100': 0.121, '101': 0.127, '110': 0.123, '111': 0.124}
X-basis: {'000': 1.0}


In [18]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import pi

sim = AerSimulator()

def measure(circuit, basis):
    qc = circuit.copy()
    if basis == 'x':
        qc.h(range(qc.num_qubits))
    qc.measure_all()
    counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
    return {k: round(v/4096, 3) for k, v in sorted(counts.items())}

# Complex circuit: entanglement + rotations + relative phase
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.rx(pi/3, [0, 2])
qc.ry(pi/4, 1)
qc.cx(0, 2)
qc.rz(pi/2, 0)
qc.crz(pi/3, 1, 2)
qc.rx(pi/5, 1)

print("=== Circuit WITH relative phase (CRZ, RZ) ===")
print(qc.draw())
print("Z-basis:", measure(qc, 'z'))
print("X-basis:", measure(qc, 'x'))

# Same circuit without relative phase gates
qc_no = QuantumCircuit(3)
qc_no.h(0)
qc_no.cx(0, 1)
qc_no.cx(1, 2)
qc_no.rx(pi/3, [0, 2])
qc_no.ry(pi/4, 1)
qc_no.cx(0, 2)
qc_no.rx(pi/5, 1)

print("\n=== Same circuit WITHOUT relative phase ===")
print(qc_no.draw())
print("Z-basis:", measure(qc_no, 'z'))
print("X-basis:", measure(qc_no, 'x'))

=== Circuit WITH relative phase (CRZ, RZ) ===
     ┌───┐     ┌─────────┐                ┌─────────┐           
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──┤ Rz(π/2) ├───────────
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  └─────────┘┌─────────┐
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼───────■─────┤ Rx(π/5) ├
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐┌────┴────┐└─────────┘
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├┤ Rz(π/3) ├───────────
                  └───┘   └─────────┘└───┘└─────────┘           
Z-basis: {'000': 0.278, '001': 0.177, '010': 0.039, '011': 0.136, '100': 0.051, '101': 0.061, '110': 0.13, '111': 0.126}
X-basis: {'000': 0.297, '001': 0.07, '010': 0.07, '011': 0.225, '100': 0.015, '101': 0.125, '110': 0.113, '111': 0.086}

=== Same circuit WITHOUT relative phase ===
     ┌───┐     ┌─────────┐                           
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■─────────────
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  ┌─────────┐
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼──┤ Rx(π/5)

In [19]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RZGate, PhaseGate, SGate, TGate, ZGate
from math import pi
import numpy as np

sim = AerSimulator()

def strip_phase_gates(circuit):
    """Create copy with all phase gates replaced by identity"""
    new_qc = circuit.copy_empty_like()
    new_qc.metadata = {}  # Avoid DAG issues
    
    for inst in circuit.data:
        gate = inst.operation
        qubits = inst.qubits
        clbits = inst.clbits
        
        # Check if it's a phase gate
        if isinstance(gate, (RZGate, PhaseGate, SGate, TGate, ZGate)):
            continue  # Skip phase gates entirely
        elif hasattr(gate, 'params') and len(gate.params) > 0:
            # Check if it's a parameterized rotation around Z
            if gate.name in ['rz', 'u1', 'p']:
                continue
            elif gate.name in ['crz', 'cp', 'cu1']:
                continue
            else:
                new_qc.append(gate, qubits, clbits)
        else:
            new_qc.append(gate, qubits, clbits)
    
    return new_qc

def detect_phase(circuit, shots=4096, threshold=0.05):
    n = circuit.num_qubits
    
    # X-basis measurement of original
    qc1 = circuit.copy()
    qc1.h(range(n))
    qc1.measure_all()
    probs1 = sim.run(transpile(qc1, sim), shots=shots).result().get_counts()
    total = sum(probs1.values())
    probs1 = {k: v/total for k, v in sorted(probs1.items())}
    
    # X-basis measurement of phase-stripped version
    stripped = strip_phase_gates(circuit)
    qc2 = stripped
    qc2.h(range(n))
    qc2.measure_all()
    probs2 = sim.run(transpile(qc2, sim), shots=shots).result().get_counts()
    total = sum(probs2.values())
    probs2 = {k: v/total for k, v in sorted(probs2.items())}
    
    # Calculate distance
    all_states = set(probs1.keys()) | set(probs2.keys())
    dist = sum(abs(probs1.get(s, 0) - probs2.get(s, 0)) for s in all_states) / 2
    
    return dist > threshold, dist, probs1, probs2

# Test
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.rx(pi/3, [0, 2])
qc.ry(pi/4, 1)
qc.cx(0, 2)
qc.rz(pi/2, 0)  # phase gate
qc.crz(pi/3, 1, 2)  # phase gate

print("Original circuit:")
print(qc.draw())

has_phase, dist, p1, p2 = detect_phase(qc)
print(f"\nPhase detected: {has_phase} (distance: {dist:.3f})")

Original circuit:
     ┌───┐     ┌─────────┐                ┌─────────┐
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──┤ Rz(π/2) ├
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  └─────────┘
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼───────■─────
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐┌────┴────┐
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├┤ Rz(π/3) ├
                  └───┘   └─────────┘└───┘└─────────┘

Phase detected: True (distance: 0.335)


In [20]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import pi
import numpy as np

sim = AerSimulator()

def measure_basis(circuit, basis, shots=8192):
    """Measure in specified basis: 'z', 'x', or 'y'"""
    qc = circuit.copy()
    
    if basis == 'x':
        qc.h(range(qc.num_qubits))
    elif basis == 'y':
        qc.sdg(range(qc.num_qubits))
        qc.h(range(qc.num_qubits))
    
    qc.measure_all()
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    total = sum(counts.values())
    return {k: v/total for k, v in sorted(counts.items())}

def detect_phase_tomography(circuit, threshold=0.05):
    """Detect relative phase using X, Y, Z measurements"""
    probs_z = measure_basis(circuit, 'z')
    probs_x = measure_basis(circuit, 'x')
    probs_y = measure_basis(circuit, 'y')
    
    # For real amplitudes: <X>^2 + <Y>^2 should match specific constraints
    # Simplified check: compare X vs Y distributions
    all_states = set(probs_x.keys()) | set(probs_y.keys())
    x_y_distance = sum(abs(probs_x.get(s, 0) - probs_y.get(s, 0)) for s in all_states) / 2
    
    return {
        'has_relative_phase': x_y_distance > threshold,
        'x_y_distance': x_y_distance,
        'probs_z': probs_z,
        'probs_x': probs_x,
        'probs_y': probs_y
    }


# Test circuit WITH relative phase
qc_phase = QuantumCircuit(3)
qc_phase.h(0)
qc_phase.cx(0, 1)
qc_phase.cx(1, 2)
qc_phase.rx(pi/3, [0, 2])
qc_phase.ry(pi/4, 1)
qc_phase.cx(0, 2)
qc_phase.rz(pi/2, 0)      # phase gate
qc_phase.crz(pi/3, 1, 2)  # phase gate

print("=== Circuit WITH relative phase ===")
print(qc_phase.draw())

result = detect_phase_tomography(qc_phase)
print(f"\nRelative phase detected: {result['has_relative_phase']}")
print(f"X-Y distance: {result['x_y_distance']:.3f}")
print(f"Z: {result['probs_z']}")
print(f"X: {result['probs_x']}")
print(f"Y: {result['probs_y']}")


# Test circuit WITHOUT relative phase
qc_no = QuantumCircuit(3)
qc_no.h(0)
qc_no.cx(0, 1)
qc_no.cx(1, 2)
qc_no.rx(pi/3, [0, 2])
qc_no.ry(pi/4, 1)
qc_no.cx(0, 2)

print("\n=== Circuit WITHOUT relative phase ===")
print(qc_no.draw())

result2 = detect_phase_tomography(qc_no)
print(f"\nRelative phase detected: {result2['has_relative_phase']}")
print(f"X-Y distance: {result2['x_y_distance']:.3f}")
print(f"Z: {result2['probs_z']}")
print(f"X: {result2['probs_x']}")
print(f"Y: {result2['probs_y']}")

=== Circuit WITH relative phase ===
     ┌───┐     ┌─────────┐                ┌─────────┐
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──┤ Rz(π/2) ├
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  └─────────┘
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼───────■─────
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐┌────┴────┐
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├┤ Rz(π/3) ├
                  └───┘   └─────────┘└───┘└─────────┘

Relative phase detected: True
X-Y distance: 0.399
Z: {'000': 0.31396484375, '001': 0.13525390625, '010': 0.00146484375, '011': 0.172119140625, '100': 0.029052734375, '101': 0.0306396484375, '110': 0.155029296875, '111': 0.1624755859375}
X: {'000': 0.3079833984375, '001': 0.0670166015625, '010': 0.06787109375, '011': 0.2236328125, '100': 0.0108642578125, '101': 0.1153564453125, '110': 0.1187744140625, '111': 0.0885009765625}
Y: {'000': 0.086181640625, '001': 0.167724609375, '010': 0.0390625, '011': 0.084716796875, '100': 0.20751953125, '101': 0.173095703125, '110': 0.162231445312

In [24]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import pi

sim = AerSimulator()

def measure(circuit, basis, shots=8192):
    qc = circuit.copy()
    n = qc.num_qubits
    if basis == 'x':
        qc.h(range(n))
    elif basis == 'y':
        qc.sdg(range(n))
        qc.h(range(n))
    qc.measure_all()
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    total = sum(counts.values())
    probs = {k: counts.get(k, 0)/total for k in [format(i, f'0{n}b') for i in range(2**n)]}
    return probs

# WITH relative phase
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.rx(pi/3, [0, 2])
qc.ry(pi/4, 1)
qc.cx(0, 2)
qc.rz(pi/2, 0)
qc.crz(pi/3, 1, 2)

print("=== Circuit WITH relative phase ===")
print(qc.draw())

pz = measure(qc, 'z')
px = measure(qc, 'x')
py = measure(qc, 'y')

print(f"\nState  |  Z-basis  |  X-basis  |  Y-basis")
print("-" * 50)
for s in sorted(pz.keys()):
    print(f" {s}    |  {pz[s]:.5f}  |  {px[s]:.5f}  |  {py[s]:.5f}")

dist = sum(abs(px[s] - py[s]) for s in px) / 2
print(f"\nRelative phase detected: {dist > 0.1} (X-Y distance: {dist:.3f})")


# WITHOUT relative phase
qc2 = QuantumCircuit(3)
qc2.h(0)
qc2.cx(0, 1)
qc2.cx(1, 2)
qc2.rx(pi/3, [0, 2])
qc2.ry(pi/4, 1)
qc2.cx(0, 2)

print("\n=== Circuit WITHOUT relative phase ===")
print(qc2.draw())

pz2 = measure(qc2, 'z')
px2 = measure(qc2, 'x')
py2 = measure(qc2, 'y')

print(f"\nState  |  Z-basis  |  X-basis  |  Y-basis")
print("-" * 50)
for s in sorted(pz2.keys()):
    print(f" {s}    |  {pz2[s]:.5f}  |  {px2[s]:.5f}  |  {py2[s]:.5f}")

dist2 = sum(abs(px2[s] - py2[s]) for s in px2) / 2
print(f"\nRelative phase detected: {dist2 > 0.1} (X-Y distance: {dist2:.3f})")

=== Circuit WITH relative phase ===
     ┌───┐     ┌─────────┐                ┌─────────┐
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──┤ Rz(π/2) ├
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  └─────────┘
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼───────■─────
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐┌────┴────┐
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├┤ Rz(π/3) ├
                  └───┘   └─────────┘└───┘└─────────┘

State  |  Z-basis  |  X-basis  |  Y-basis
--------------------------------------------------
 000    |  0.30884  |  0.29468  |  0.08044
 001    |  0.13147  |  0.07178  |  0.16577
 010    |  0.00183  |  0.06445  |  0.03662
 011    |  0.17896  |  0.22766  |  0.08191
 100    |  0.02917  |  0.01160  |  0.21240
 101    |  0.02673  |  0.11816  |  0.17603
 110    |  0.15723  |  0.12012  |  0.16748
 111    |  0.16577  |  0.09155  |  0.07935

Relative phase detected: True (X-Y distance: 0.400)

=== Circuit WITHOUT relative phase ===
     ┌───┐     ┌─────────┐                
q_0: ┤ H ├──

In [26]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import RZGate, CRZGate
from math import pi
import numpy as np

def detect_rz_phase(circuit):
    """Detects Rz/CRz gates by comparing vs reference without them."""
    sv_actual = Statevector(circuit)
    
    # Build reference circuit without phase gates
    ref_circuit = circuit.copy_empty_like()
    for inst in circuit.data:
        gate = inst.operation
        qubits = inst.qubits
        
        if isinstance(gate, (RZGate, CRZGate)):
            continue
        ref_circuit.append(gate, qubits)
    
    sv_ref = Statevector(ref_circuit)
    distance = np.linalg.norm(sv_actual.data - sv_ref.data)
    
    return {
        'has_rz_phase': distance > 1e-6,
        'distance': distance,
        'reference_circuit': ref_circuit
    }

# TEST CASE 1: With relative phase
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.rx(pi/3, [0, 2])
qc.ry(pi/4, 1)
qc.cx(0, 2)
qc.rz(pi/2, 0)
qc.crz(pi/3, 1, 2)

print("CASE 1: Circuit WITH relative phase (Rz, CRz)")
print("-" * 50)
print("Original circuit:")
print(qc.draw())

result = detect_rz_phase(qc)
print(f"\nReference circuit (phase gates removed):")
print(result['reference_circuit'].draw())

print(f"\n--- Result ---")
print(f"Rz phase detected: {result['has_rz_phase']}")
print(f"Statevector distance: {result['distance']:.6f}")


# TEST CASE 2: Without relative phase
qc2 = QuantumCircuit(3)
qc2.h(0)
qc2.cx(0, 1)
qc2.cx(1, 2)
qc2.rx(pi/3, [0, 2])
qc2.ry(pi/4, 1)
qc2.cx(0, 2)

print("\n\nCASE 2: Circuit WITHOUT relative phase")
print("-" * 50)
print("Original circuit:")
print(qc2.draw())

result2 = detect_rz_phase(qc2)
print(f"\nReference circuit (phase gates removed):")
print(result2['reference_circuit'].draw())

print(f"\n--- Result ---")
print(f"Rz phase detected: {result2['has_rz_phase']}")
print(f"Statevector distance: {result2['distance']:.6f}")

CASE 1: Circuit WITH relative phase (Rz, CRz)
--------------------------------------------------
Original circuit:
     ┌───┐     ┌─────────┐                ┌─────────┐
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──┤ Rz(π/2) ├
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  └─────────┘
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼───────■─────
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐┌────┴────┐
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├┤ Rz(π/3) ├
                  └───┘   └─────────┘└───┘└─────────┘

Reference circuit (phase gates removed):
     ┌───┐     ┌─────────┐                
q_0: ┤ H ├──■──┤ Rx(π/3) ├─────────────■──
     └───┘┌─┴─┐└─────────┘┌─────────┐  │  
q_1: ─────┤ X ├─────■─────┤ Ry(π/4) ├──┼──
          └───┘   ┌─┴─┐   ├─────────┤┌─┴─┐
q_2: ─────────────┤ X ├───┤ Rx(π/3) ├┤ X ├
                  └───┘   └─────────┘└───┘

--- Result ---
Rz phase detected: True
Statevector distance: 0.745333


CASE 2: Circuit WITHOUT relative phase
--------------------------------------------------
O

In [27]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import SGate, TGate, PhaseGate
from math import pi
import numpy as np

def detect_phase_gates(circuit, phase_gate_types=(SGate, TGate, PhaseGate)):
    """Detects phase gates by comparing vs reference without them."""
    sv_actual = Statevector(circuit)
    
    # Build reference circuit without phase gates
    ref_circuit = circuit.copy_empty_like()
    for inst in circuit.data:
        gate = inst.operation
        qubits = inst.qubits
        
        if isinstance(gate, phase_gate_types):
            continue
        ref_circuit.append(gate, qubits)
    
    sv_ref = Statevector(ref_circuit)
    distance = np.linalg.norm(sv_actual.data - sv_ref.data)
    
    return {
        'has_phase': distance > 1e-6,
        'distance': distance,
        'reference_circuit': ref_circuit
    }

# TEST CASE 1: With S and T phase gates
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc.cx(0, 1)
qc.s(1)      # Phase gate: adds i to |1>
qc.cx(1, 2)
qc.t(2)      # Phase gate: adds e^(iπ/4) to |1>
qc.h(0)
qc.cx(2, 0)
qc.sdg(0)    # Phase gate: adds -i to |1>

print("CASE 1: Circuit WITH S and T phase gates")
print("-" * 50)
print("Original circuit:")
print(qc.draw())

result = detect_phase_gates(qc)
print(f"\nReference circuit (S, T gates removed):")
print(result['reference_circuit'].draw())

print(f"\n--- Result ---")
print(f"Phase detected: {result['has_phase']}")
print(f"Statevector distance: {result['distance']:.6f}")


# TEST CASE 2: Without phase gates (only H and CX)
qc2 = QuantumCircuit(3)
qc2.h([0, 1, 2])
qc2.cx(0, 1)
qc2.cx(1, 2)
qc2.h(0)
qc2.cx(2, 0)

print("\n\nCASE 2: Circuit WITHOUT phase gates (H and CX only)")
print("-" * 50)
print("Original circuit:")
print(qc2.draw())

result2 = detect_phase_gates(qc2)
print(f"\nReference circuit (no phase gates to remove):")
print(result2['reference_circuit'].draw())

print(f"\n--- Result ---")
print(f"Phase detected: {result2['has_phase']}")
print(f"Statevector distance: {result2['distance']:.6f}")

CASE 1: Circuit WITH S and T phase gates
--------------------------------------------------
Original circuit:
     ┌───┐     ┌───┐          ┌───┐┌─────┐
q_0: ┤ H ├──■──┤ H ├──────────┤ X ├┤ Sdg ├
     ├───┤┌─┴─┐├───┤          └─┬─┘└─────┘
q_1: ┤ H ├┤ X ├┤ S ├──■─────────┼─────────
     ├───┤└───┘└───┘┌─┴─┐┌───┐  │         
q_2: ┤ H ├──────────┤ X ├┤ T ├──■─────────
     └───┘          └───┘└───┘            

Reference circuit (S, T gates removed):
     ┌───┐     ┌───┐┌───┐┌─────┐
q_0: ┤ H ├──■──┤ H ├┤ X ├┤ Sdg ├
     ├───┤┌─┴─┐└───┘└─┬─┘└─────┘
q_1: ┤ H ├┤ X ├──■────┼─────────
     ├───┤└───┘┌─┴─┐  │         
q_2: ┤ H ├─────┤ X ├──■─────────
     └───┘     └───┘            

--- Result ---
Phase detected: True
Statevector distance: 1.224745


CASE 2: Circuit WITHOUT phase gates (H and CX only)
--------------------------------------------------
Original circuit:
     ┌───┐     ┌───┐┌───┐
q_0: ┤ H ├──■──┤ H ├┤ X ├
     ├───┤┌─┴─┐└───┘└─┬─┘
q_1: ┤ H ├┤ X ├──■────┼──
     ├───┤└───┘┌─┴─┐  

In [5]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import SGate
from math import pi
import numpy as np

def detect_phase_gates(circuit):
    sv_actual = Statevector(circuit)
    
    ref_circuit = circuit.copy_empty_like()
    for inst in circuit.data:
        gate = inst.operation
        qubits = inst.qubits
        if isinstance(gate, (SGate,)):
            continue
        ref_circuit.append(gate, qubits)
    
    sv_ref = Statevector(ref_circuit)
    distance = np.linalg.norm(sv_actual.data - sv_ref.data)
    
    return {
        'has_phase': distance > 1e-6,
        'distance': distance,
        'reference_circuit': ref_circuit
    }

# CASE 1: Bell state with relative phase
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.cx(0, 1)  # Bell entanglement
qc1.s(1)      # Relative phase on |11>

print("CASE 1: Bell state WITH relative phase (S gate)")
print("-" * 50)
print("Original circuit:")
print(qc1.draw())

result1 = detect_phase_gates(qc1)
print(f"\nReference circuit (S gate removed):")
print(result1['reference_circuit'].draw())
print(f"\n--- Result ---")
print(f"Phase detected: {result1['has_phase']}")
print(f"Statevector distance: {result1['distance']:.6f}")

# CASE 2: GHZ with relative phase
qc2 = QuantumCircuit(3)
qc2.h(0)
qc2.cx(0, 1)
qc2.cx(1, 2)  # GHZ entanglement
qc2.s(2)      # Relative phase on |111>

print("\nCASE 2: GHZ state WITH relative phase")
print("-" * 50)
print("Original circuit:")
print(qc2.draw())

result2 = detect_phase_gates(qc2)
print(f"\nReference circuit (S gate removed):")
print(result2['reference_circuit'].draw())
print(f"\n--- Result ---")
print(f"Phase detected: {result2['has_phase']}")
print(f"Statevector distance: {result2['distance']:.6f}")

# CASE 3: Pure Bell without phase
qc3 = QuantumCircuit(2)
qc3.h(0)
qc3.cx(0, 1)

print("\nCASE 3: Bell state WITHOUT phase")
print("-" * 50)
print("Original circuit:")
print(qc3.draw())

result3 = detect_phase_gates(qc3)
print(f"\nReference circuit (no phase gates to remove):")
print(result3['reference_circuit'].draw())
print(f"\n--- Result ---")
print(f"Phase detected: {result3['has_phase']}")
print(f"Statevector distance: {result3['distance']:.6f}")

CASE 1: Bell state WITH relative phase (S gate)
--------------------------------------------------
Original circuit:
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐┌───┐
q_1: ─────┤ X ├┤ S ├
          └───┘└───┘

Reference circuit (S gate removed):
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘

--- Result ---
Phase detected: True
Statevector distance: 1.000000

CASE 2: GHZ state WITH relative phase
--------------------------------------------------
Original circuit:
     ┌───┐               
q_0: ┤ H ├──■────────────
     └───┘┌─┴─┐          
q_1: ─────┤ X ├──■───────
          └───┘┌─┴─┐┌───┐
q_2: ──────────┤ X ├┤ S ├
               └───┘└───┘

Reference circuit (S gate removed):
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘

--- Result ---
Phase detected: True
Statevector distance: 1.000000

CASE 3: Bell state WITHOUT phase
-----------------

In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit (replace with your circuit) ─────────────────────────────
def base_circuit():
    qr = QuantumRegister(2, 'q')
    qc = QuantumCircuit(qr)
    qc.h(0)
    qc.cx(0, 1)   # Bell state |Φ+⟩
    return qc, qr

# ── 3. Measurement circuits ─────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.h(0); qc.h(1)
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw circuits ─────────────────────────────────────────────────────────
for name, qc in circuits.items():
    print(f"\n{'='*52}")
    print(f"  {name} Measurement Circuit")
    print('='*52)
    print(qc.draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*52}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*52}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*52}")


  Z-basis Measurement Circuit
        ┌───┐     ┌─┐   
   q_0: ┤ H ├──■──┤M├───
        └───┘┌─┴─┐└╥┘┌─┐
   q_1: ─────┤ X ├─╫─┤M├
             └───┘ ║ └╥┘
meas: 2/═══════════╩══╩═
                   0  1 

  X-basis Measurement Circuit
        ┌───┐     ┌───┐┌─┐   
   q_0: ┤ H ├──■──┤ H ├┤M├───
        └───┘┌─┴─┐├───┤└╥┘┌─┐
   q_1: ─────┤ X ├┤ H ├─╫─┤M├
             └───┘└───┘ ║ └╥┘
meas: 2/════════════════╩══╩═
                        0  1 

  Y-basis Measurement Circuit
        ┌───┐     ┌─────┐┌───┐┌─┐   
   q_0: ┤ H ├──■──┤ Sdg ├┤ H ├┤M├───
        └───┘┌─┴─┐├─────┤├───┤└╥┘┌─┐
   q_1: ─────┤ X ├┤ Sdg ├┤ H ├─╫─┤M├
             └───┘└─────┘└───┘ ║ └╥┘
meas: 2/═══════════════════════╩══╩═
                               0  1 

Bitstring         Z-basis      X-basis      Y-basis
----------------------------------------------------
00                 0.5137       0.4863       0.0020
01                 0.0068       0.0049       0.4854
10                 0.0039       0.0039       0.5059
1

In [5]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — Bell state + Rz (introduces relative phase) ───────────
def base_circuit():
    qr = QuantumRegister(2, 'q')
    qc = QuantumCircuit(qr)
    qc.h(0)
    qc.cx(0, 1)          # Bell state |Φ+⟩ = (|00⟩ + |11⟩)/√2
    qc.rz(np.pi / 4, 0)  # Adds relative phase e^{iπ/4} to |11⟩ component
    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.h(0); qc.h(1)
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw circuits ─────────────────────────────────────────────────────────
for name, qc in circuits.items():
    print(f"\n{'='*54}")
    print(f"  {name} Measurement Circuit")
    print('='*54)
    print(qc.draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*54}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*54}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*54}")


  Z-basis Measurement Circuit
        ┌───┐     ┌─────────┐┌─┐
   q_0: ┤ H ├──■──┤ Rz(π/4) ├┤M├
        └───┘┌─┴─┐└───┬─┬───┘└╥┘
   q_1: ─────┤ X ├────┤M├─────╫─
             └───┘    └╥┘     ║ 
meas: 2/═══════════════╩══════╩═
                       1      0 

  X-basis Measurement Circuit
        ┌───┐     ┌─────────┐┌───┐┌─┐
   q_0: ┤ H ├──■──┤ Rz(π/4) ├┤ H ├┤M├
        └───┘┌─┴─┐└──┬───┬──┘└┬─┬┘└╥┘
   q_1: ─────┤ X ├───┤ H ├────┤M├──╫─
             └───┘   └───┘    └╥┘  ║ 
meas: 2/═══════════════════════╩═══╩═
                               1   0 

  Y-basis Measurement Circuit
        ┌───┐     ┌─────────┐┌─────┐┌───┐┌─┐
   q_0: ┤ H ├──■──┤ Rz(π/4) ├┤ Sdg ├┤ H ├┤M├
        └───┘┌─┴─┐└─┬─────┬─┘└┬───┬┘└┬─┬┘└╥┘
   q_1: ─────┤ X ├──┤ Sdg ├───┤ H ├──┤M├──╫─
             └───┘  └─────┘   └───┘  └╥┘  ║ 
meas: 2/══════════════════════════════╩═══╩═
                                      1   0 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------

In [6]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — 3 qubits with Rx, Ry, Rz ──────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)
    qc.rx(np.pi / 3, 0)
    qc.ry(np.pi / 4, 1)
    qc.rz(np.pi / 5, 2)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()
    qc.rx(np.pi / 6, 2)
    qc.ry(np.pi / 3, 0)
    qc.rz(np.pi / 4, 1)
    qc.barrier()
    return qc, qr

# ── 3. Statevector (before measurement) ─────────────────────────────────────
qc_sv = base_circuit()[0]
sv    = Statevector(qc_sv)

print(f"\n{'='*60}")
print("  Statevector (before measurement)")
print('='*60)
print(f"{'State':<10} {'Amplitude':>28} {'Probability':>12}")
print(f"{'-'*60}")

basis_labels = [f'|{format(i, "03b")}⟩' for i in range(8)]
for label, amp in zip(basis_labels, sv.data):
    prob = abs(amp) ** 2
    print(f"{label:<10} {str(amp):>28} {prob:>12.4f}")

print(f"{'='*60}")

# ── 4. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 5. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 6. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 7. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Statevector (before measurement)
State                         Amplitude  Probability
------------------------------------------------------------
|000⟩      (0.5689553128220768-0.36440530109089275j)       0.4565
|001⟩      (0.18988522738710295-0.3726707421101626j)       0.1749
|010⟩      (0.06542340525038343-0.06936508860933888j)       0.0091
|011⟩      (-0.09985682439112524-0.050879593332974336j)       0.0126
|100⟩      (-0.09764210614492483-0.1524511166000722j)       0.0328
|101⟩      (-0.09985682439112525-0.05087959333297434j)       0.0126
|110⟩      (0.25887403496156997+0.24416347239859942j)       0.1266
|111⟩      (0.18988522738710295-0.37267074211016255j)       0.1749

  Z-basis Measurement Circuit
        ┌─────────┐ ░            ░ ┌─────────┐ ░  ░ ┌─┐      
   q_0: ┤ Rx(π/3) ├─░───■────────░─┤ Ry(π/3) ├─░──░─┤M├──────
        ├─────────┤ ░ ┌─┴─┐      ░ ├─────────┤ ░  ░ └╥┘┌─┐   
   q_1: ┤ Ry(π/4) ├─░─┤ X ├──■───░─┤ Rz(π/4) ├─░──░──╫─┤M├───
        ├─────────┤ ░ └───┘┌─┴─┐ ░

In [7]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — 3 qubits, Rx/Ry/Rz + S gates ─────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    # Layer 1 — local rotations
    qc.rx(np.pi / 3, 0)
    qc.ry(np.pi / 4, 1)
    qc.rz(np.pi / 5, 2)
    qc.barrier()

    # Layer 2 — entanglement
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # Layer 3 — S gates (arbitrary phase kicks)
    qc.s(0)
    qc.s(2)
    qc.barrier()

    # Layer 4 — mixed rotations
    qc.rx(np.pi / 6, 2)
    qc.ry(np.pi / 3, 0)
    qc.rz(np.pi / 4, 1)
    qc.barrier()

    # Layer 5 — S gate on q1
    qc.s(1)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Z-basis Measurement Circuit
        ┌─────────┐ ░            ░ ┌───┐ ░ ┌─────────┐ ░       ░  ░ ┌─┐      
   q_0: ┤ Rx(π/3) ├─░───■────────░─┤ S ├─░─┤ Ry(π/3) ├─░───────░──░─┤M├──────
        ├─────────┤ ░ ┌─┴─┐      ░ └───┘ ░ ├─────────┤ ░ ┌───┐ ░  ░ └╥┘┌─┐   
   q_1: ┤ Ry(π/4) ├─░─┤ X ├──■───░───────░─┤ Rz(π/4) ├─░─┤ S ├─░──░──╫─┤M├───
        ├─────────┤ ░ └───┘┌─┴─┐ ░ ┌───┐ ░ ├─────────┤ ░ └───┘ ░  ░  ║ └╥┘┌─┐
   q_2: ┤ Rz(π/5) ├─░──────┤ X ├─░─┤ S ├─░─┤ Rx(π/6) ├─░───────░──░──╫──╫─┤M├
        └─────────┘ ░      └───┘ ░ └───┘ ░ └─────────┘ ░       ░  ░  ║  ║ └╥┘
meas: 3/═════════════════════════════════════════════════════════════╩══╩══╩═
                                                                     0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.3242       0.0898       0.2021
001                0.3066       0.0371       0.0967
010                0.0000       0.3623       0.00

In [10]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — shallow, 3 qubits ──────────────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.ry(np.pi / 4, 1)
    qc.s(2)
    qc.barrier()

    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    qc.rz(np.pi / 3, 0)
    qc.t(1)
    qc.rx(np.pi / 4, 2)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Z-basis Measurement Circuit
           ┌───┐    ░            ░ ┌─────────┐ ░  ░ ┌─┐      
   q_0: ───┤ H ├────░───■────────░─┤ Rz(π/3) ├─░──░─┤M├──────
        ┌──┴───┴──┐ ░ ┌─┴─┐      ░ └──┬───┬──┘ ░  ░ └╥┘┌─┐   
   q_1: ┤ Ry(π/4) ├─░─┤ X ├──■───░────┤ T ├────░──░──╫─┤M├───
        └──┬───┬──┘ ░ └───┘┌─┴─┐ ░ ┌──┴───┴──┐ ░  ░  ║ └╥┘┌─┐
   q_2: ───┤ S ├────░──────┤ X ├─░─┤ Rx(π/4) ├─░──░──╫──╫─┤M├
           └───┘    ░      └───┘ ░ └─────────┘ ░  ░  ║  ║ └╥┘
meas: 3/═════════════════════════════════════════════╩══╩══╩═
                                                     0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.3701       0.2080       0.0811
001                0.0771       0.1523       0.0742
010                0.0049       0.1084       0.3115
011                0.0654       0.0146       0.0166
100                0.0635       0.1309       0.3193
101                0.0156       0.007

In [11]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — 3 qubits, no T gate ───────────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.ry(np.pi / 4, 1)
    qc.s(2)
    qc.barrier()

    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    qc.rz(np.pi / 3, 0)
    qc.rx(np.pi / 4, 2)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Z-basis Measurement Circuit
           ┌───┐    ░            ░ ┌─────────┐ ░  ░ ┌─┐      
   q_0: ───┤ H ├────░───■────────░─┤ Rz(π/3) ├─░──░─┤M├──────
        ┌──┴───┴──┐ ░ ┌─┴─┐      ░ └─────────┘ ░  ░ └╥┘┌─┐   
   q_1: ┤ Ry(π/4) ├─░─┤ X ├──■───░─────────────░──░──╫─┤M├───
        └──┬───┬──┘ ░ └───┘┌─┴─┐ ░ ┌─────────┐ ░  ░  ║ └╥┘┌─┐
   q_2: ───┤ S ├────░──────┤ X ├─░─┤ Rx(π/4) ├─░──░──╫──╫─┤M├
           └───┘    ░      └───┘ ░ └─────────┘ ░  ░  ║  ║ └╥┘
meas: 3/═════════════════════════════════════════════╩══╩══╩═
                                                     0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.3584       0.3145       0.0703
001                0.0684       0.0879       0.0684
010                0.0107       0.0244       0.3184
011                0.0684       0.0547       0.0381
100                0.0527       0.0225       0.3193
101                0.0088       0.062

In [12]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — 3 qubits + extra Rz ───────────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.ry(np.pi / 4, 1)
    qc.s(2)
    qc.barrier()

    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    qc.rz(np.pi / 3, 0)
    qc.rx(np.pi / 4, 2)
    qc.barrier()

    qc.rz(np.pi / 7, 1)   # arbitrary Rz on q1
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Z-basis Measurement Circuit
           ┌───┐    ░            ░ ┌─────────┐ ░             ░  ░ ┌─┐      
   q_0: ───┤ H ├────░───■────────░─┤ Rz(π/3) ├─░─────────────░──░─┤M├──────
        ┌──┴───┴──┐ ░ ┌─┴─┐      ░ └─────────┘ ░ ┌─────────┐ ░  ░ └╥┘┌─┐   
   q_1: ┤ Ry(π/4) ├─░─┤ X ├──■───░─────────────░─┤ Rz(π/7) ├─░──░──╫─┤M├───
        └──┬───┬──┘ ░ └───┘┌─┴─┐ ░ ┌─────────┐ ░ └─────────┘ ░  ░  ║ └╥┘┌─┐
   q_2: ───┤ S ├────░──────┤ X ├─░─┤ Rx(π/4) ├─░─────────────░──░──╫──╫─┤M├
           └───┘    ░      └───┘ ░ └─────────┘ ░             ░  ░  ║  ║ └╥┘
meas: 3/═══════════════════════════════════════════════════════════╩══╩══╩═
                                                                   0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.3525       0.2617       0.0596
001                0.0615       0.1338       0.0684
010                0.0098       0.0781       0.3359
011            

In [13]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — simple 3 qubit GHZ state ──────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")


  Z-basis Measurement Circuit
        ┌───┐ ░            ░  ░ ┌─┐      
   q_0: ┤ H ├─░───■────■───░──░─┤M├──────
        └───┘ ░ ┌─┴─┐  │   ░  ░ └╥┘┌─┐   
   q_1: ──────░─┤ X ├──┼───░──░──╫─┤M├───
              ░ └───┘┌─┴─┐ ░  ░  ║ └╥┘┌─┐
   q_2: ──────░──────┤ X ├─░──░──╫──╫─┤M├
              ░      └───┘ ░  ░  ║  ║ └╥┘
meas: 3/═════════════════════════╩══╩══╩═
                                 0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.4971       0.2295       0.1270
001                0.0049       0.0020       0.1211
010                0.0000       0.0029       0.1240
011                0.0020       0.2314       0.1240
100                0.0000       0.0039       0.1240
101                0.0000       0.2773       0.1240
110                0.0020       0.2520       0.1387
111                0.4941       0.0010       0.1172


In [14]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — simple 3 qubit GHZ state ──────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.rz(np.pi/3,0)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")



  Z-basis Measurement Circuit
        ┌───┐ ░           ┌─────────┐ ░  ░ ┌─┐      
   q_0: ┤ H ├─░───■────■──┤ Rz(π/3) ├─░──░─┤M├──────
        └───┘ ░ ┌─┴─┐  │  └─────────┘ ░  ░ └╥┘┌─┐   
   q_1: ──────░─┤ X ├──┼──────────────░──░──╫─┤M├───
              ░ └───┘┌─┴─┐            ░  ░  ║ └╥┘┌─┐
   q_2: ──────░──────┤ X ├────────────░──░──╫──╫─┤M├
              ░      └───┘            ░  ░  ║  ║ └╥┘
meas: 3/════════════════════════════════════╩══╩══╩═
                                            0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.5205       0.1914       0.0186
001                0.0059       0.0596       0.2539
010                0.0000       0.0664       0.2295
011                0.0000       0.1895       0.0303
100                0.0000       0.0576       0.2168
101                0.0029       0.1904       0.0205
110                0.0000       0.1875       0.0127
111           

In [15]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — simple 3 qubit GHZ state ──────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.rz(np.pi/3,1)
    qc.barrier()

    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")



  Z-basis Measurement Circuit
        ┌───┐ ░                       ░  ░ ┌─┐      
   q_0: ┤ H ├─░───■────■──────────────░──░─┤M├──────
        └───┘ ░ ┌─┴─┐  │  ┌─────────┐ ░  ░ └╥┘┌─┐   
   q_1: ──────░─┤ X ├──┼──┤ Rz(π/3) ├─░──░──╫─┤M├───
              ░ └───┘┌─┴─┐└─────────┘ ░  ░  ║ └╥┘┌─┐
   q_2: ──────░──────┤ X ├────────────░──░──╫──╫─┤M├
              ░      └───┘            ░  ░  ║  ║ └╥┘
meas: 3/════════════════════════════════════╩══╩══╩═
                                            0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.4785       0.1934       0.0205
001                0.0029       0.0596       0.2305
010                0.0000       0.0781       0.2324
011                0.0029       0.1885       0.0244
100                0.0000       0.0635       0.2178
101                0.0020       0.1777       0.0156
110                0.0059       0.1660       0.0234
111           

In [16]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — simple 3 qubit GHZ state ──────────────────────────────
def base_circuit():
    qr = QuantumRegister(3, 'q')
    qc = QuantumCircuit(qr)

    qc.h(0)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.rz(np.pi/3,1)
    qc.barrier()
    qc.rz(np.pi/2,1)
    qc.barrier()
    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(3, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.sdg(2); qc.h(2)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  Z-basis Measurement Circuit")
print('='*60)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*60}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*60}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*60}")



  Z-basis Measurement Circuit
        ┌───┐ ░                       ░             ░  ░ ┌─┐      
   q_0: ┤ H ├─░───■────■──────────────░─────────────░──░─┤M├──────
        └───┘ ░ ┌─┴─┐  │  ┌─────────┐ ░ ┌─────────┐ ░  ░ └╥┘┌─┐   
   q_1: ──────░─┤ X ├──┼──┤ Rz(π/3) ├─░─┤ Rz(π/2) ├─░──░──╫─┤M├───
              ░ └───┘┌─┴─┐└─────────┘ ░ └─────────┘ ░  ░  ║ └╥┘┌─┐
   q_2: ──────░──────┤ X ├────────────░─────────────░──░──╫──╫─┤M├
              ░      └───┘            ░             ░  ░  ║  ║ └╥┘
meas: 3/══════════════════════════════════════════════════╩══╩══╩═
                                                          0  1  2 

Bitstring         Z-basis      X-basis      Y-basis
------------------------------------------------------------
000                0.5010       0.0234       0.0537
001                0.0039       0.2246       0.1641
010                0.0000       0.2158       0.2031
011                0.0000       0.0117       0.0605
100                0.0010       0.2383      

In [19]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import GenericBackendV2

# ── 1. Noisy backend ────────────────────────────────────────────────────────
device      = GenericBackendV2(num_qubits=5, seed=42)
noise_model = NoiseModel.from_backend(device)
backend     = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=device.coupling_map
)

# ── 2. Base circuit — Bell state |Φ+⟩ ───────────────────────────────────────
def base_circuit():
    qr = QuantumRegister(2, 'q')
    qc = QuantumCircuit(qr)
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    qc.rz(np.pi/4,1)
    qc.barrier()
    return qc, qr

# ── 3. Measurement circuits ──────────────────────────────────────────────────
def make_z_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_x_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.h(0); qc.h(1)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

def make_y_circuit():
    qc, qr = base_circuit()
    cr = ClassicalRegister(2, 'meas')
    qc.add_register(cr)
    qc.barrier()
    qc.sdg(0); qc.h(0)
    qc.sdg(1); qc.h(1)
    qc.barrier()
    qc.measure(qr, cr)
    return qc

circuits = {
    'Z-basis': make_z_circuit(),
    'X-basis': make_x_circuit(),
    'Y-basis': make_y_circuit(),
}

# ── 4. Draw Z-basis circuit only ─────────────────────────────────────────────
print(f"\n{'='*52}")
print("  Z-basis Measurement Circuit")
print('='*52)
print(circuits['Z-basis'].draw('text'))

# ── 5. Transpile & run ───────────────────────────────────────────────────────
SHOTS = 1024
all_counts = {}

for name, qc in circuits.items():
    tqc    = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=SHOTS).result()
    all_counts[name] = result.get_counts()

# ── 6. Unified side-by-side probability table ────────────────────────────────
all_bitstrings = sorted(
    set(bs for counts in all_counts.values() for bs in counts.keys())
)

print(f"\n{'='*52}")
print(f"{'Bitstring':<12} {'Z-basis':>12} {'X-basis':>12} {'Y-basis':>12}")
print(f"{'-'*52}")

for bs in all_bitstrings:
    z_prob = all_counts['Z-basis'].get(bs, 0) / SHOTS
    x_prob = all_counts['X-basis'].get(bs, 0) / SHOTS
    y_prob = all_counts['Y-basis'].get(bs, 0) / SHOTS
    print(f"{bs:<12} {z_prob:>12.4f} {x_prob:>12.4f} {y_prob:>12.4f}")

print(f"{'='*52}")


  Z-basis Measurement Circuit
        ┌───┐      ░             ░  ░ ┌─┐   
   q_0: ┤ H ├──■───░─────────────░──░─┤M├───
        └───┘┌─┴─┐ ░ ┌─────────┐ ░  ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░─┤ Rz(π/4) ├─░──░──╫─┤M├
             └───┘ ░ └─────────┘ ░  ░  ║ └╥┘
meas: 2/═══════════════════════════════╩══╩═
                                       0  1 

Bitstring         Z-basis      X-basis      Y-basis
----------------------------------------------------
00                 0.4854       0.4355       0.0693
01                 0.0049       0.0713       0.3926
10                 0.0039       0.0850       0.4600
11                 0.5059       0.4082       0.0781


### Section 1: Grover search circuit with Bit flip error as bug, comparing their kl divergence in terms of ratio

In [15]:
from qiskit.circuit.library.standard_gates import CXGate, CZGate
from qiskit.circuit import Measure

def count_1q_2q_gates(qc):
    """
    Count the number of 1-qubit and 2-qubit gates in a circuit,
    ignoring measure operations.
    """
    n_1q = 0
    n_2q = 0
    for instr, qargs, cargs in qc.data:
        # Skip measurements
        if isinstance(instr, Measure):
            continue
        # Two-qubit gates: CX, CZ, etc.
        if len(qargs) == 2:
            n_2q += 1
        # One-qubit gates
        elif len(qargs) == 1:
            n_1q += 1
        # Ignore anything else for now (barriers, etc.)
    return n_1q, n_2q

In [16]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Measure
import random

# ---- helpers ----
def kldivergence(p, q, epsilon=1e-10):
    p = np.array(p, dtype=float) + epsilon
    q = np.array(q, dtype=float) + epsilon
    p /= np.sum(p)
    q /= np.sum(q)
    return float(np.sum(p * np.log(p / q)))

def counts_to_probs(counts, n_qubits):
    total = sum(counts.values())
    probs = np.zeros(2**n_qubits, dtype=float)
    for bitstring, count in counts.items():
        idx = int(bitstring, 2)
        probs[idx] = count / total
    return probs

def print_top_states(label, probs, n_qubits, top_k=4, min_prob=1e-3):
    items = []
    for idx, p in enumerate(probs):
        if p >= min_prob:
            bit = format(idx, f"0{n_qubits}b")
            items.append((bit, p))
    items.sort(key=lambda x: x[1], reverse=True)
    items = items[:top_k]
    print(label)
    for bit, p in items:
        print(f"  {bit}: {p:.4f}")
    print()

# ---- Grover circuits (2 qubits) ----
n_qubits = 2

grover_correct = QuantumCircuit(n_qubits, n_qubits)
grover_correct.h([0, 1])
grover_correct.cz(0, 1)
grover_correct.h([0, 1])
grover_correct.x([0, 1])
grover_correct.h(1)
grover_correct.cx(0, 1)
grover_correct.h(1)
grover_correct.x([0, 1])
grover_correct.h([0, 1])
grover_correct.measure([0, 1], [0, 1])

grover_buggy = QuantumCircuit(n_qubits, n_qubits)
grover_buggy.h([0, 1])
grover_buggy.cz(0, 1)
grover_buggy.h([0, 1])
grover_buggy.x([0, 1])
grover_buggy.h(1)
grover_buggy.cx(0, 1)
grover_buggy.h(1)
grover_buggy.x([0, 1])
grover_buggy.h([0, 1])
bug_qubit = random.choice([0, 1])
grover_buggy.x(bug_qubit)
grover_buggy.measure([0, 1], [0, 1])

print("Correct Grover circuit:")
print(grover_correct.draw())
print(f"\nBuggy Grover circuit (random X on q{bug_qubit}):")
print(grover_buggy.draw())

# ---- ideal distribution ----
t_ideal = transpile(grover_correct, simulator_ideal)
counts_ideal = simulator_ideal.run(t_ideal, shots=shots).result().get_counts()
probs_ideal = counts_to_probs(counts_ideal, n_qubits=n_qubits)

# ---- hardware run (uses your existing helper) ----
grover_hw = hardware_bug_detection(
    name="GroverRandomX",
    qc_correct=grover_correct,
    qc_buggy=grover_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=probs_ideal,
    shots=shots,
)

probs_hw_correct = grover_hw["probs_hw_correct"]
probs_hw_buggy  = grover_hw["probs_hw_buggy"]

kl_noise = kldivergence(probs_hw_correct, probs_ideal)
kl_bug   = kldivergence(probs_hw_buggy,  probs_ideal)
ratio    = kl_bug / kl_noise if kl_noise > 0 else np.inf

print("\n--- KL numbers (hardware) ---")
print(f"KL (noise vs ideal): {kl_noise:.4f}")
print(f"KL (bug vs ideal) : {kl_bug:.4f}")
print(f"Detection ratio   : {ratio:.2f}x")

# ---- show which states changed ----
print_top_states("Ideal (simulator):", probs_ideal, n_qubits)
print_top_states("HW correct:", probs_hw_correct, n_qubits)
print_top_states("HW buggy:", probs_hw_buggy, n_qubits)

Correct Grover circuit:
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy Grover circuit (random X on q0):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

GROVERRANDOMX – REAL HARDWARE (ibm_fez)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (noise vs ideal): 1.5978
KL (bug vs 

In [16]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Measure
import random

# ---- helpers ----
def kldivergence(p, q, epsilon=1e-10):
    p = np.array(p, dtype=float) + epsilon
    q = np.array(q, dtype=float) + epsilon
    p /= np.sum(p)
    q /= np.sum(q)
    return float(np.sum(p * np.log(p / q)))

def counts_to_probs(counts, n_qubits):
    total = sum(counts.values())
    probs = np.zeros(2**n_qubits, dtype=float)
    for bitstring, count in counts.items():
        idx = int(bitstring, 2)
        probs[idx] = count / total
    return probs

def count_1q_2q_gates(qc):
    n_1q = 0
    n_2q = 0
    for ci in qc.data:
        instr = ci.operation
        qargs = ci.qubits
        if isinstance(instr, Measure):
            continue
        if len(qargs) == 2:
            n_2q += 1
        elif len(qargs) == 1:
            n_1q += 1
    return n_1q, n_2q

def get_avg_error_rates(backend):
    """Return avg 1q gate error, avg 2q gate error, avg readout error."""
    props = backend.properties()

    # 1-qubit gate error: average over all 1q gates on all qubits
    one_q_errors = []
    two_q_errors = []
    readout_errors = []

    for gate in props.gates:
        if len(gate.qubits) == 1:
            # gate.parameters includes "gate_error" in modern Qiskit backends
            for p in gate.parameters:
                if p.name == "gate_error":
                    one_q_errors.append(p.value)
        elif len(gate.qubits) == 2:
            for p in gate.parameters:
                if p.name == "gate_error":
                    two_q_errors.append(p.value)

    for qubit_props in props.qubits:
        for p in qubit_props:
            if p.name in ("readout_error", "prob_meas0_prep1", "prob_meas1_prep0"):
                # simple approximation: use readout_error if present,
                # else average of misprep/mismeas probs (if that's what backend exposes)
                if p.name == "readout_error":
                    readout_errors.append(p.value)
                else:
                    readout_errors.append(p.value)

    e_1q = float(np.mean(one_q_errors)) if one_q_errors else 0.0
    e_2q = float(np.mean(two_q_errors)) if two_q_errors else 0.0
    e_ro = float(np.mean(readout_errors)) if readout_errors else 0.0

    return e_1q, e_2q, e_ro

# ---- Grover circuits (2 qubits) ----
n_qubits = 2

grover_correct = QuantumCircuit(n_qubits, n_qubits)
grover_correct.h([0, 1])
grover_correct.cz(0, 1)
grover_correct.h([0, 1])
grover_correct.x([0, 1])
grover_correct.h(1)
grover_correct.cx(0, 1)
grover_correct.h(1)
grover_correct.x([0, 1])
grover_correct.h([0, 1])
grover_correct.measure([0, 1], [0, 1])

grover_buggy = QuantumCircuit(n_qubits, n_qubits)
grover_buggy.h([0, 1])
grover_buggy.cz(0, 1)
grover_buggy.h([0, 1])
grover_buggy.x([0, 1])
grover_buggy.h(1)
grover_buggy.cx(0, 1)
grover_buggy.h(1)
grover_buggy.x([0, 1])
grover_buggy.h([0, 1])
bug_qubit = random.choice([0, 1])
grover_buggy.x(bug_qubit)
grover_buggy.measure([0, 1], [0, 1])

print("Correct Grover circuit:")
print(grover_correct.draw())
print(f"\nBuggy Grover circuit (random X on q{bug_qubit}):")
print(grover_buggy.draw())

# ---- ideal distribution (simulator) ----
t_ideal = transpile(grover_correct, simulator_ideal)
counts_ideal = simulator_ideal.run(t_ideal, shots=shots).result().get_counts()
probs_ideal = counts_to_probs(counts_ideal, n_qubits=n_qubits)

# ---- hardware run (existing helper) ----
grover_hw = hardware_bug_detection(
    name="GroverRandomX",
    qc_correct=grover_correct,
    qc_buggy=grover_buggy,
    n_qubits=n_qubits,
    backend=backend,
    pass_manager=pm,
    probs_ideal=probs_ideal,
    shots=shots,
)

probs_hw_correct = grover_hw["probs_hw_correct"]
probs_hw_buggy  = grover_hw["probs_hw_buggy"]

kl_noise = kldivergence(probs_hw_correct, probs_ideal)
kl_bug   = kldivergence(probs_hw_buggy,  probs_ideal)
ratio    = kl_bug / kl_noise if kl_noise > 0 else np.inf

print("\n--- KL numbers (hardware) ---")
print(f"KL (noise vs ideal): {kl_noise:.4f}")
print(f"KL (bug vs ideal) : {kl_bug:.4f}")
print(f"Detection ratio   : {ratio:.2f}x")

# ---- lightweight noise estimator using backend properties ----
n_1q, n_2q = count_1q_2q_gates(grover_correct)
e_1q, e_2q, e_ro = get_avg_error_rates(backend)

F_gates = (1.0 - e_1q)**n_1q * (1.0 - e_2q)**n_2q
F_ro    = (1.0 - e_ro)**n_qubits
F_total = F_gates * F_ro

C_factor = 10.0
estimated_kl_noise_max = C_factor * (1.0 - F_total)**2

print("\n--- Lightweight noise estimate (from backend.properties) ---")
print(f"1q gates            : {n_1q}")
print(f"2q gates            : {n_2q}")
print(f"avg e_1q            : {e_1q:.4e}")
print(f"avg e_2q            : {e_2q:.4e}")
print(f"avg e_ro            : {e_ro:.4e}")
print(f"F_gates (est.)      : {F_gates:.4f}")
print(f"F_readout (est.)    : {F_ro:.4f}")
print(f"F_total (est.)      : {F_total:.4f}")
print(f"KL_noise_max (est.) : {estimated_kl_noise_max:.4e}")
print(f"KL_noise (hw)       : {kl_noise:.4e}")
print(f"KL_bug   (hw)       : {kl_bug:.4e}")
print(f"ΔKL bug - noise     : {kl_bug - kl_noise:.4e}")

Correct Grover circuit:
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy Grover circuit (random X on q1):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐        
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├────────
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌───┐┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤ X ├┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └───┘└╥┘
c: 2/════════════════════════════════════════════╩═══════╩═
                                                 0       1 

GROVERRANDOMX – REAL HARDWARE (ibm_fez)
------------------------------------------------------------
Qubits : 2
Shots  : 8192
KL (

## Per-Qubit Hardware Error & Flip Rate vs KL Ratio

In [11]:
# Block A: Build 2-qubit Grover circuits
# Correct: standard Grover with CZ oracle (marks |11⟩)
# Buggy:   identical + one random X (bit-flip) after diffusion

import random
from qiskit import QuantumCircuit

N_QUBITS = 2
SHOTS_GR = 1024

def build_grover_2q(inject_x_on=None):
    qc = QuantumCircuit(N_QUBITS, N_QUBITS)
    qc.h([0, 1])
    qc.cz(0, 1)               # oracle: phase-flip |11⟩
    qc.h([0, 1])              # diffusion start
    qc.x([0, 1])
    qc.h(1); qc.cx(0, 1); qc.h(1)
    qc.x([0, 1])
    qc.h([0, 1])              # diffusion end
    if inject_x_on is not None:
        qc.x(inject_x_on)    # inject bit-flip bug
    qc.measure([0, 1], [0, 1])
    return qc

bug_qubit      = random.choice([0, 1])
grover_correct = build_grover_2q()
grover_buggy   = build_grover_2q(inject_x_on=bug_qubit)

print(grover_correct.draw(output="text", fold=-1))
print(grover_buggy.draw(output="text", fold=-1))
print(f"Bug injected on: q{bug_qubit}")

     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐        
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├────────
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌───┐┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤ X ├┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └───┘└╥┘
c: 2/════════════════════════════════════════════╩═══════╩═
                                                 0       1 
Bug injected on: q1


In [12]:
# Block B: Ideal distribution from noiseless simulator
# Depends on: grover_correct, N_QUBITS, SHOTS_GR
# Uses globals: simulator_ideal, counts_to_probs

from qiskit import transpile

t_ideal         = transpile(grover_correct, simulator_ideal)
counts_ideal_gr = simulator_ideal.run(t_ideal, shots=SHOTS_GR).result().get_counts()
probs_ideal_gr  = counts_to_probs(counts_ideal_gr, N_QUBITS)

print("Ideal distribution:")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_ideal_gr[idx] > 1e-4:
        print(f"  |{b}>  {probs_ideal_gr[idx]:.4f}")

Ideal distribution:
  |11>  1.0000


In [13]:
# Block C: Per-qubit hardware error rates from IBM calibration data
# For each physical qubit q:
#   gate_err[q]  = 1 - prod(1 - gate_error_i)  for ALL gates touching q
#   readout[q]   = measurement error
#   combined[q]  = 1 - (1 - gate_err[q]) * (1 - readout[q])
#
# Lower combined → safer qubit.  Higher combined → more likely to bit-flip.
# Uses globals: backend, pm

import numpy as np

def get_per_qubit_errors(backend):
    props = backend.properties()
    nq    = backend.num_qubits

    # readout error per qubit
    readout = {}
    for q, qprops in enumerate(props.qubits):
        ro = next((float(p.value) for p in qprops if p.name == "readout_error"), None)
        if ro is None:
            m01 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
            m10 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
            ro  = (m01 + m10) / 2.0 if (m01 is not None and m10 is not None) else 0.0
        readout[q] = ro

    # gate error product per qubit — log-sum avoids float underflow
    log_fid = {q: 0.0 for q in range(nq)}
    for gate in props.gates:
        err = next((float(p.value) for p in gate.parameters if p.name == "gate_error"), None)
        if err is None:
            continue
        lf = np.log(max(1.0 - err, 1e-12))
        for q in gate.qubits:
            if q in log_fid:
                log_fid[q] += lf

    gate_err = {q: 1.0 - np.exp(log_fid[q]) for q in range(nq)}
    combined = {q: 1.0 - (1.0 - gate_err[q]) * (1.0 - readout[q]) for q in range(nq)}
    return combined, gate_err, readout


def print_hw_error_table(combined, gate_err, readout, used_qubits):
    print("\nPer-Qubit IBM Hardware Error Rates")
    print("=" * 64)
    print(f"  {'Qubit':>5}  {'Gate Err':>10}  {'Readout':>10}  {'Combined':>10}  {'Risk':>5}")
    print("-" * 64)
    for q in sorted(used_qubits):
        risk = "HIGH" if combined[q] > 0.15 else ("MED" if combined[q] > 0.05 else "LOW")
        print(f"  q{q:<4d}  {gate_err[q]:>10.6f}  {readout[q]:>10.6f}  {combined[q]:>10.6f}  {risk}")
    safest   = min(used_qubits, key=lambda q: combined[q])
    riskiest = max(used_qubits, key=lambda q: combined[q])
    print("=" * 64)
    print(f"  Safest   q{safest}  {combined[safest]:.6f}  <- lower flip probability")
    print(f"  Riskiest q{riskiest}  {combined[riskiest]:.6f}  <- higher flip probability")


# resolve which physical qubits the transpiler actually used
t_correct_gr = pm.run(grover_correct)
try:
    layout_gr            = t_correct_gr.layout.final_layout
    used_physical_qubits = sorted(set(layout_gr.get_physical_bits().keys()))
except Exception:
    used_physical_qubits = list(range(N_QUBITS))

combined_err, gate_err, readout_err = get_per_qubit_errors(backend)

print(f"Physical qubits used: {used_physical_qubits}")
print_hw_error_table(combined_err, gate_err, readout_err, used_physical_qubits)

Physical qubits used: [0, 1]

Per-Qubit IBM Hardware Error Rates
  Qubit    Gate Err     Readout    Combined   Risk
----------------------------------------------------------------
  q0       0.016025    0.005615    0.021550  LOW
  q1       0.019964    0.005371    0.025228  LOW
  Safest   q0  0.021550  <- lower flip probability
  Riskiest q1  0.025228  <- higher flip probability


In [14]:
# Block D: Run both circuits on IBM real hardware
# Uses globals: hardware_bug_detection, kl_divergence

gr_hw = hardware_bug_detection(
    name         = "Grover2Q_BitFlip",
    qc_correct   = grover_correct,
    qc_buggy     = grover_buggy,
    n_qubits     = N_QUBITS,
    backend      = backend,
    pass_manager = pm,
    probs_ideal  = probs_ideal_gr,
    shots        = SHOTS_GR,
)

probs_hw_correct = gr_hw["probs_hw_correct"]
probs_hw_buggy   = gr_hw["probs_hw_buggy"]

kl_correct  = kl_divergence(probs_hw_correct, probs_ideal_gr)
kl_buggy    = kl_divergence(probs_hw_buggy,   probs_ideal_gr)
ratio_joint = kl_buggy / kl_correct if kl_correct > 1e-9 else float("inf")

print("\n--- Joint KL Divergence (full 2-qubit distribution) ---")
print(f"  KL(hw_correct vs ideal)  =  {kl_correct:.4f}   <- hardware noise baseline")
print(f"  KL(hw_buggy   vs ideal)  =  {kl_buggy:.4f}   <- noise + injected bit-flip")
print(f"  Ratio (buggy / correct)  =  {ratio_joint:.2f}x")
print(f"  Verdict: {'BUG DETECTED' if ratio_joint > 1.5 else 'INCONCLUSIVE'}")


GROVER2Q_BITFLIP – REAL HARDWARE (ibm_marrakesh)
------------------------------------------------------------
Qubits : 2
Shots  : 1024
KL (noise vs ideal): 0.6639
KL (bug vs ideal) : 22.5079
Detection ratio   : 33.90x
Threshold         : 1.50x
Verdict           : BUG DETECTED
Job usage (this call): 2.00 s
Remaining total: 511.00 s | Used total: 89.00 s

--- Joint KL Divergence (full 2-qubit distribution) ---
  KL(hw_correct vs ideal)  =  0.6639   <- hardware noise baseline
  KL(hw_buggy   vs ideal)  =  22.5079   <- noise + injected bit-flip
  Ratio (buggy / correct)  =  33.90x
  Verdict: BUG DETECTED


In [15]:
# Block E: Per-qubit observed flip rate and marginal KL divergence
#
# marginal(joint, q):
#   sum over all bitstrings where qubit q == '0'
#   Qiskit MSB-first: qubit q is at character index (n-1-q)
#
# flip_rate[q] = |P_hw(q=0) - P_ideal(q=0)|
# kl_binary[q] = KL of the Bernoulli marginal on that qubit

def marginal(joint_probs, nqubits, q):
    p0 = sum(
        joint_probs[idx]
        for idx in range(2 ** nqubits)
        if format(idx, f"0{nqubits}b")[nqubits - 1 - q] == "0"
    )
    return p0, 1.0 - p0


def per_qubit_flip_rates(probs_ideal, probs_hw, nqubits):
    result = {}
    for q in range(nqubits):
        p0i, _ = marginal(probs_ideal, nqubits, q)
        p0h, _ = marginal(probs_hw,    nqubits, q)
        result[q] = {"p0_ideal": p0i, "p0_hw": p0h, "flip_rate": abs(p0h - p0i)}
    return result


def kl_binary(p0_ref, p0_test, eps=1e-10):
    p = np.array([p0_test, 1.0 - p0_test]) + eps
    q = np.array([p0_ref,  1.0 - p0_ref])  + eps
    p /= p.sum(); q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


fr_correct = per_qubit_flip_rates(probs_ideal_gr, probs_hw_correct, N_QUBITS)
fr_buggy   = per_qubit_flip_rates(probs_ideal_gr, probs_hw_buggy,   N_QUBITS)

# flip rate table
print(f"\nPer-Qubit Flip Rates  ({SHOTS_GR} shots)")
print("=" * 72)
print(f"  {'Q':>2}  {'P0_ideal':>9}  {'P0_hw(cor)':>11}  {'flip(cor)':>10}  "
      f"{'P0_hw(bug)':>11}  {'flip(bug)':>10}")
print("-" * 72)
for q in range(N_QUBITS):
    fc, fb = fr_correct[q], fr_buggy[q]
    tag    = "  <- BUG" if q == bug_qubit else ""
    print(f"  q{q}  {fc['p0_ideal']:>9.4f}  {fc['p0_hw']:>11.4f}  {fc['flip_rate']:>10.4f}  "
          f"{fb['p0_hw']:>11.4f}  {fb['flip_rate']:>10.4f}{tag}")
print("=" * 72)

# marginal KL table
print("\nPer-Qubit Marginal KL  (ideal -> hardware, Bernoulli)")
print("=" * 54)
print(f"  {'Q':>2}  {'KL(correct)':>13}  {'KL(buggy)':>11}  {'ratio':>7}  verdict")
print("-" * 54)
for q in range(N_QUBITS):
    fc, fb = fr_correct[q], fr_buggy[q]
    kl_c   = kl_binary(fc["p0_ideal"], fc["p0_hw"])
    kl_b   = kl_binary(fb["p0_ideal"], fb["p0_hw"])
    ratio  = kl_b / kl_c if kl_c > 1e-6 else float("inf")
    verdict = "BUG SIGNAL" if ratio > 2.0 else "no signal"
    tag     = "  <-" if q == bug_qubit else ""
    print(f"  q{q}  {kl_c:>13.4f}  {kl_b:>11.4f}  {ratio:>7.2f}x  {verdict}{tag}")
print("=" * 54)
print(f"  Bug was injected on: q{bug_qubit}")


Per-Qubit Flip Rates  (1024 shots)
   Q   P0_ideal   P0_hw(cor)   flip(cor)   P0_hw(bug)   flip(bug)
------------------------------------------------------------------------
  q0     0.0000       0.0186      0.0186       0.0137      0.0137
  q1     0.0000       0.0205      0.0205       0.9824      0.9824  <- BUG

Per-Qubit Marginal KL  (ideal -> hardware, Bernoulli)
   Q    KL(correct)    KL(buggy)    ratio  verdict
------------------------------------------------------
  q0         0.3349       0.2425     0.72x  no signal
  q1         0.3722      22.5326    60.54x  BUG SIGNAL  <-
  Bug was injected on: q1


In [16]:
# Block F: Does the IBM error ranking match the observed flip ranking?
# virt_to_phys maps circuit qubit index -> physical qubit used by transpiler

virt_to_phys = {v: p for v, p in enumerate(sorted(used_physical_qubits))}
hw_mean      = np.mean([combined_err[p] for p in used_physical_qubits])
flip_mean    = np.mean([fr_correct[v]["flip_rate"] for v in range(N_QUBITS)])

print("\nIBM Combined Error  vs  Observed Flip Rate")
print("=" * 64)
print(f"  {'VQ':>2}  {'PQ':>3}  {'IBM combined':>14}  {'Obs flip rate':>14}  Consistent?")
print("-" * 64)
for vq in range(N_QUBITS):
    pq  = virt_to_phys[vq]
    he  = combined_err[pq]
    of  = fr_correct[vq]["flip_rate"]
    ok  = (he < hw_mean) == (of < flip_mean)
    tag = "yes" if ok else "no (cross-talk/drift)"
    print(f"  q{vq}  q{pq:>2}  {he:>14.6f}  {of:>14.6f}  {tag}")
print("=" * 64)


IBM Combined Error  vs  Observed Flip Rate
  VQ   PQ    IBM combined   Obs flip rate  Consistent?
----------------------------------------------------------------
  q0  q 0        0.021550        0.018555  yes
  q1  q 1        0.025228        0.020508  yes


In [17]:
# Block G: Full probability distribution table

def print_dist(label, probs, nqubits, top_k=4):
    items = [(format(i, f"0{nqubits}b"), p) for i, p in enumerate(probs) if p > 1e-4]
    items.sort(key=lambda x: x[1], reverse=True)
    print(f"\n{label}")
    print("-" * 20)
    for b, p in items[:top_k]:
        print(f"  |{b}>  {p:.4f}")

print_dist("Ideal",      probs_ideal_gr,   N_QUBITS)
print_dist("HW correct", probs_hw_correct, N_QUBITS)
print_dist("HW buggy",   probs_hw_buggy,   N_QUBITS)


Ideal
--------------------
  |11>  1.0000

HW correct
--------------------
  |11>  0.9629
  |01>  0.0186
  |10>  0.0166
  |00>  0.0020

HW buggy
--------------------
  |01>  0.9707
  |11>  0.0156
  |00>  0.0117
  |10>  0.0020


In [18]:
# Per-qubit error rate summary: IBM calibration vs observed flip rate

print(f"\nPer-Qubit Error Rate Summary  (bug injected on q{bug_qubit})")
print("=" * 70)
print(f"  {'Q':>2}  {'IBM gate err':>13}  {'IBM readout':>12}  "
      f"{'IBM combined':>13}  {'Observed flip':>14}")
print("-" * 70)
for vq in range(N_QUBITS):
    pq  = virt_to_phys[vq]
    ge  = gate_err[pq]
    ro  = readout_err[pq]
    cb  = combined_err[pq]
    fr  = fr_correct[vq]["flip_rate"]   # from correct circuit (pure hardware noise)
    tag = "  <- BUG QUBIT" if vq == bug_qubit else ""
    print(f"  q{vq}  {ge:>13.6f}  {ro:>12.6f}  {cb:>13.6f}  {fr:>14.6f}{tag}")
print("=" * 70)
print()
print("  IBM combined  = predicted error from calibration data")
print("  Observed flip = measured |P_hw(q=0) - P_ideal(q=0)| on correct circuit")
print()

# which qubit is safer?
safest = min(range(N_QUBITS), key=lambda q: combined_err[virt_to_phys[q]])
riskiest = max(range(N_QUBITS), key=lambda q: combined_err[virt_to_phys[q]])
print(f"  Safer   qubit → q{safest}  "
      f"(IBM combined {combined_err[virt_to_phys[safest]]:.6f}, "
      f"flip {fr_correct[safest]['flip_rate']:.6f})")
print(f"  Riskier qubit → q{riskiest}  "
      f"(IBM combined {combined_err[virt_to_phys[riskiest]]:.6f}, "
      f"flip {fr_correct[riskiest]['flip_rate']:.6f})")


Per-Qubit Error Rate Summary  (bug injected on q1)
   Q   IBM gate err   IBM readout   IBM combined   Observed flip
----------------------------------------------------------------------
  q0       0.016025      0.005615       0.021550        0.018555
  q1       0.019964      0.005371       0.025228        0.020508  <- BUG QUBIT

  IBM combined  = predicted error from calibration data
  Observed flip = measured |P_hw(q=0) - P_ideal(q=0)| on correct circuit

  Safer   qubit → q0  (IBM combined 0.021550, flip 0.018555)
  Riskier qubit → q1  (IBM combined 0.025228, flip 0.020508)


## Section 2

In [11]:
# Block 1 — Imports
import numpy as np
import random
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Measure
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    thermal_relaxation_error,
    ReadoutError,
)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_pass_managers import generate_preset_pass_manager
from kaggle_secrets import UserSecretsClient

ModuleNotFoundError: No module named 'qiskit.transpiler.preset_pass_managers'

In [ ]:
# Block 2 — IBM Backend connection
user_secrets = UserSecretsClient()
token        = user_secrets.get_secret("IBM_QUANTUM_TOKEN")
service      = QiskitRuntimeService(token=token, instance="open/open/open")

backend = service.least_busy(simulator=False, operational=True, min_num_qubits=3)
pm      = generate_preset_pass_manager(
    backend=backend, optimization_level=1, translation_method="translator"
)

print(f"  backend  : {backend.name}")
print(f"  Qubits   : {backend.num_qubits}")
print(f"  Pending  : {backend.status().pending_jobs} jobs")

In [ ]:
# Block 3 — Grover circuits (2 qubits, oracle marks |11>)
# Correct : standard Grover — CZ oracle + one diffusion step
# Buggy   : same + one random X (bit-flip) injected after diffusion

N_QUBITS = 2
SHOTS_GR = 8192

def build_grover_2q(inject_x_on=None):
    """Build 2-qubit Grover circuit; optionally inject a bit-flip bug."""
    qc = QuantumCircuit(N_QUBITS, N_QUBITS)
    qc.h([0, 1])           # uniform superposition
    qc.cz(0, 1)            # oracle: phase-flip |11>
    qc.h([0, 1])           # diffusion start
    qc.x([0, 1])
    qc.h(1); qc.cx(0, 1); qc.h(1)
    qc.x([0, 1])
    qc.h([0, 1])           # diffusion end
    if inject_x_on is not None:
        qc.x(inject_x_on)  # BUG: bit-flip error
    qc.measure([0, 1], [0, 1])
    return qc

bug_qubit      = random.choice([0, 1])
grover_correct = build_grover_2q()
grover_buggy   = build_grover_2q(inject_x_on=bug_qubit)

print("Correct Grover circuit:")
print(grover_correct.draw(output="text", fold=-1))
print(f"\nBuggy Grover circuit (X injected on q{bug_qubit}):")
print(grover_buggy.draw(output="text", fold=-1))

In [ ]:
# Block 4 — Helper utilities

def counts_to_probs(counts, nqubits):
    """Convert Qiskit counts dict → probability vector of length 2^nqubits."""
    total = sum(counts.values())
    probs = np.zeros(2 ** nqubits, dtype=float)
    for bitstring, count in counts.items():
        probs[int(bitstring, 2)] = count / total
    return probs

def sampler_result_to_counts(result_entry):
    """Extract bitstring counts from a SamplerV2 pub result entry."""
    try:
        return result_entry.data.c.get_counts()
    except AttributeError:
        pass
    try:
        reg_name = list(result_entry.data.keys())[0]
        return result_entry.data[reg_name].get_counts()
    except Exception:
        pass
    raise RuntimeError("Cannot extract counts from SamplerV2 result entry.")

def get_usage_seconds(service):
    """Return (used_seconds, remaining_seconds) for this service instance."""
    usage     = service.usage()
    used      = float(usage.get("usage_consumed_seconds", 0.0))
    remaining = usage.get("usage_remaining_seconds")
    if remaining is None:
        limit     = usage.get("usage_limit_seconds") or usage.get("usage_allocation_seconds")
        remaining = float(limit - used) if limit is not None else None
    return used, remaining

def count_1q_2q_gates(qc):
    """Count single- and two-qubit gates, ignoring measurements."""
    n_1q = n_2q = 0
    for ci in qc.data:
        if isinstance(ci.operation, Measure):
            continue
        if   len(ci.qubits) == 1: n_1q += 1
        elif len(ci.qubits) == 2: n_2q += 1
    return n_1q, n_2q

In [ ]:
# Block 5 — Ideal (noiseless) reference distribution

simulator_ideal = AerSimulator()

t_ideal         = transpile(grover_correct, simulator_ideal)
counts_ideal_gr = simulator_ideal.run(t_ideal, shots=SHOTS_GR).result().get_counts()
probs_ideal_gr  = counts_to_probs(counts_ideal_gr, N_QUBITS)

print("Ideal distribution:")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_ideal_gr[idx] > 1e-4:
        print(f"  |{b}>  {probs_ideal_gr[idx]:.4f}")

In [12]:
# Block 6 — Hardware-accurate noise estimator
#
# Inputs extracted dynamically from the live IBM backend:
#   T1, T2         (seconds) — relaxation / dephasing times per qubit
#   gate_time_1q   (seconds) — single-qubit (sx) gate duration per qubit
#   err_1q                   — single-qubit gate error rate per qubit
#   gate_time_2q   (seconds) — two-qubit (cx) gate duration per qubit pair
#   err_2q                   — two-qubit gate error rate per qubit pair
#   P(0|1), P(1|0)           — readout misassignment probabilities per qubit
#
# Noise channels:
#   thermal_relaxation_error(T1, T2, gate_time)  →  amplitude + phase damping
#   depolarizing_error(gate_err, n)              →  residual gate noise
#   ReadoutError([[...]])                        →  asymmetric measurement error
#
# Composite per gate:
#   1-qubit :  thermal_1q  ∘  depolarising_1q
#   2-qubit :  (thermal_q0 ⊗ thermal_q1)  ∘  depolarising_2q

def build_hardware_noise_model(backend, qubits_used):
    props       = backend.properties()
    noise_model = NoiseModel()
    calib_table = {}

    # ── single-qubit channels ──────────────────────────────────────────────
    for q in qubits_used:

        try:    t1 = props.t1(q)
        except: t1 = 100e-6
        try:    t2 = props.t2(q)
        except: t2 = 80e-6
        t2 = min(t2, 2 * t1)              # physical constraint T2 ≤ 2·T1

        try:    gate_time_1q = props.gate_length("sx", [q])
        except: gate_time_1q = 35e-9      # 35 ns fallback

        try:    err_1q = props.gate_error("sx", [q])
        except: err_1q = 1e-3

        try:
            ro    = props.readout_error(q)
            p1_g0 = ro * 0.8              # P(1|0) — slightly asymmetric
            p0_g1 = ro                    # P(0|1)
        except:
            p1_g0 = p0_g1 = 0.01

        calib_table[q] = {
            "T1":           t1,
            "T2":           t2,
            "gate_time_1q": gate_time_1q,
            "err_1q":       err_1q,
            "readout_err":  (p0_g1 + p1_g0) / 2,
            "P(0|1)":       p0_g1,
            "P(1|0)":       p1_g0,
        }

        therm_1q    = thermal_relaxation_error(t1=t1, t2=t2, time=gate_time_1q)
        depo_1q     = depolarizing_error(err_1q, 1)
        combined_1q = therm_1q.compose(depo_1q)

        for gate in ["sx", "x", "id"]:
            try: noise_model.add_quantum_error(combined_1q, gate, [q])
            except: pass

        ro_matrix = [
            [1 - p1_g0, p1_g0  ],         # prepared |0>: P(meas 0), P(meas 1)
            [p0_g1,  1 - p0_g1 ],          # prepared |1>: P(meas 0), P(meas 1)
        ]
        try: noise_model.add_readout_error(ReadoutError(ro_matrix), [q])
        except: pass

    # ── two-qubit channels ─────────────────────────────────────────────────
    for i in range(len(qubits_used)):
        for j in range(i + 1, len(qubits_used)):
            q0, q1 = qubits_used[i], qubits_used[j]

            try:    gate_time_2q = props.gate_length("cx", [q0, q1])
            except: gate_time_2q = 300e-9  # 300 ns fallback

            try:    err_2q = props.gate_error("cx", [q0, q1])
            except: err_2q = 5e-3

            therm_q0    = thermal_relaxation_error(
                t1=calib_table[q0]["T1"], t2=calib_table[q0]["T2"], time=gate_time_2q)
            therm_q1    = thermal_relaxation_error(
                t1=calib_table[q1]["T1"], t2=calib_table[q1]["T2"], time=gate_time_2q)
            therm_2q    = therm_q0.expand(therm_q1)
            depo_2q     = depolarizing_error(err_2q, 2)
            combined_2q = therm_2q.compose(depo_2q)

            for gate in ["cx", "cz"]:
                for pair in [[q0, q1], [q1, q0]]:
                    try: noise_model.add_quantum_error(combined_2q, gate, pair)
                    except: pass

    return noise_model, calib_table


def print_calibration_table(calib_table, backend_name=""):
    if backend_name:
        print(f"\nCalibration Data  ({backend_name})")
    print(f"  {'Qubit':>5}  {'T1 (µs)':>10}  {'T2 (µs)':>10}  "
          f"{'gate_t(ns)':>11}  {'err_1q':>10}  {'RO_err':>8}")
    print("  " + "-" * 62)
    for q, d in sorted(calib_table.items()):
        print(f"  q{q:<4}  {d['T1']*1e6:>10.2f}  {d['T2']*1e6:>10.2f}  "
              f"{d['gate_time_1q']*1e9:>11.2f}  {d['err_1q']:>10.4e}  "
              f"{d['readout_err']:>8.4f}")
    print()

In [13]:
# Block 7 — Identify physical qubits after transpilation
#           + build hardware-accurate noise model

tqc_hw = pm.run(grover_correct)
try:
    layout_gr            = tqc_hw.layout.final_layout
    used_physical_qubits = sorted({
        layout_gr[grover_correct.qubits[q]] for q in range(N_QUBITS)
    })
except Exception:
    used_physical_qubits = list(range(N_QUBITS))

print(f"Physical qubits used on {backend.name}: {used_physical_qubits}")

print("\nExtracting calibration data from IBM backend ...")
noise_model_hw, calib_table = build_hardware_noise_model(backend, used_physical_qubits)
print_calibration_table(calib_table, backend_name=backend.name)

simulator_est = AerSimulator(noise_model=noise_model_hw)

NameError: name 'grover_correct' is not defined

In [ ]:
# Block 8 — Run Grover circuit under the hardware noise estimator

tqc_est        = transpile(grover_correct, simulator_est)
counts_est_cor = simulator_est.run(tqc_est, shots=SHOTS_GR).result().get_counts()
probs_est_cor  = counts_to_probs(counts_est_cor, N_QUBITS)

tqc_est_bug    = transpile(grover_buggy, simulator_est)
counts_est_bug = simulator_est.run(tqc_est_bug, shots=SHOTS_GR).result().get_counts()
probs_est_bug  = counts_to_probs(counts_est_bug, N_QUBITS)

print("Noise-Estimator distribution (correct circuit):")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_est_cor[idx] > 1e-4:
        print(f"  |{b}>  {probs_est_cor[idx]:.4f}")

In [ ]:
# Block 9 — Run Grover circuit on real IBM hardware

print(f"Running on real hardware: {backend.name} ...")
used_before, _ = get_usage_seconds(service)

tqc_hw_bug = pm.run(grover_buggy)
sampler    = Sampler(mode=backend)
job        = sampler.run(
    [(tqc_hw, grover_correct), (tqc_hw_bug, grover_buggy)],
    shots=SHOTS_GR
)
result = job.result()

counts_hw_cor = sampler_result_to_counts(result[0])
counts_hw_bug = sampler_result_to_counts(result[1])
probs_hw_cor  = counts_to_probs(counts_hw_cor, N_QUBITS)
probs_hw_bug  = counts_to_probs(counts_hw_bug, N_QUBITS)

used_after, rem_after = get_usage_seconds(service)
job_secs = used_after - used_before
print(f"  Job time : {job_secs:.1f}s  |  Total used: {used_after:.1f}s", end="")
if rem_after is not None:
    print(f"  |  Remaining: {rem_after:.1f}s")
else:
    print()

print("\nReal Hardware distribution (correct circuit):")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_hw_cor[idx] > 1e-4:
        print(f"  |{b}>  {probs_hw_cor[idx]:.4f}")

In [ ]:
# Block 10 — Per-qubit bit-flip error rate
#
# For each qubit i (marginalised over all other qubits):
#   d_i = 0 if P_ideal(qubit_i = 1) < 0.5 else 1    ← ideal dominant bit
#   flip_rate_i = P_measured(qubit_i ≠ d_i)

def marginal_p1(probs, nqubits, q):
    """Return P(qubit q = 1) marginalised over all other qubits."""
    p1 = 0.0
    for idx in range(2 ** nqubits):
        if (idx >> q) & 1:
            p1 += probs[idx]
    return p1

def per_qubit_bitflip_rate(probs_measured, probs_ideal, nqubits):
    flip_rates = {}
    for q in range(nqubits):
        p1_ideal = marginal_p1(probs_ideal,    nqubits, q)
        p1_meas  = marginal_p1(probs_measured, nqubits, q)
        d = 1 if p1_ideal >= 0.5 else 0       # ideal dominant bit
        if d == 0:
            flip_rates[q] = float(p1_meas)         # excess 1s = flips
        else:
            flip_rates[q] = float(1.0 - p1_meas)   # excess 0s = flips
    return flip_rates

In [ ]:
# Block 11 — Compare per-qubit flip-rate ratios: estimator vs hardware
#
# ratio_q = est_flip_rate_q / hw_flip_rate_q
# |ratio_q - 1| <= 0.30  →  ✓ MATCH
# otherwise              →  ✗ MISMATCH (estimator needs refinement)

def compare_flip_rate_ratios(est_rates, hw_rates, nqubits, tolerance=0.30):
    print(f"\n  {'Qubit':>5}  {'Est. Flip':>12}  {'HW Flip':>12}  "
          f"{'Ratio (E/H)':>13}  {'|Ratio-1|':>11}  {'Status':>12}")
    print("  " + "-" * 72)
    all_match = True
    for q in range(nqubits):
        er = est_rates.get(q, 0.0)
        hr = hw_rates.get(q, 0.0)
        if hr > 1e-8:
            ratio = er / hr
        elif er > 1e-8:
            ratio = np.inf
        else:
            ratio = 1.0                    # both ~0 → perfect match
        deviation = abs(ratio - 1.0)
        match     = deviation <= tolerance and not np.isinf(ratio)
        if not match:
            all_match = False
        flag = "✓ MATCH" if match else "✗ MISMATCH"
        print(f"  q{q:<4}  {er:>12.6f}  {hr:>12.6f}  {ratio:>13.4f}  "
              f"{deviation:>11.4f}  {flag:>12}")
    print("  " + "-" * 72)
    if all_match:
        print(f"  OVERALL VERDICT : Noise estimator is ACCURATE")
        print(f"                    All per-qubit ratios within ±{tolerance*100:.0f}% of hardware")
    else:
        print(f"  OVERALL VERDICT : Noise estimator NEEDS REFINEMENT")
        print(f"                    Some per-qubit ratios exceed ±{tolerance*100:.0f}% deviation")
    print()
    return all_match

In [ ]:
# Block 12 — Compute per-qubit bit-flip error rates
#             noise estimator vs real hardware vs ideal

flip_est = per_qubit_bitflip_rate(probs_est_cor, probs_ideal_gr, N_QUBITS)
flip_hw  = per_qubit_bitflip_rate(probs_hw_cor,  probs_ideal_gr, N_QUBITS)

print("Per-Qubit Bit-Flip Error Rates")
print("=" * 50)
print(f"  {'Qubit':>5}  {'Noise Estimator':>16}  {'Real Hardware':>14}")
print("  " + "-" * 42)
for q in range(N_QUBITS):
    print(f"  q{q:<4}  {flip_est[q]:>16.6f}  {flip_hw[q]:>14.6f}")
print()

In [ ]:
# Block 13 — Noise estimator accuracy assessment

print("=" * 74)
print("  NOISE ESTIMATOR ACCURACY ASSESSMENT  (tolerance = ±30%)")
print("=" * 74)

estimator_accurate = compare_flip_rate_ratios(
    flip_est, flip_hw, nqubits=N_QUBITS, tolerance=0.30
)

In [ ]:
# Block 14 — Per-qubit IBM calibration error vs observed flip rate

def get_per_qubit_errors(backend):
    props   = backend.properties()
    nq      = backend.num_qubits
    readout = {}
    for q, qprops in enumerate(props.qubits):
        ro  = next((float(p.value) for p in qprops if p.name == "readout_error"), None)
        if ro is None:
            m01 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
            m10 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
            ro  = (m01 + m10) / 2.0 if (m01 and m10) else 0.0
        readout[q] = ro
    log_fid = {q: 0.0 for q in range(nq)}
    for gate in props.gates:
        err = next((float(p.value) for p in gate.parameters if p.name == "gate_error"), None)
        if err is None:
            continue
        lf = np.log(max(1.0 - err, 1e-12))
        for q in gate.qubits:
            if q in log_fid:
                log_fid[q] += lf
    gate_err = {q: 1.0 - np.exp(log_fid[q]) for q in range(nq)}
    combined = {q: 1.0 - (1.0 - gate_err[q]) * (1.0 - readout[q]) for q in range(nq)}
    return combined, gate_err, readout

combined_err, gate_err, readout_err = get_per_qubit_errors(backend)
virt_to_phys = {v: p for v, p in enumerate(sorted(used_physical_qubits))}

print(f"Per-Qubit Error Rate Summary  (bug injected on q{bug_qubit})")
print("=" * 74)
print(f"  {'VQ':>2}  {'PQ':>3}  {'IBM gate err':>13}  {'IBM readout':>12}  "
      f"{'IBM combined':>13}  {'Obs flip rate':>14}")
print("  " + "-" * 68)
for vq in range(N_QUBITS):
    pq  = virt_to_phys[vq]
    tag = "  <- BUG QUBIT" if vq == bug_qubit else ""
    print(f"  q{vq}  q{pq:>2}  {gate_err[pq]:>13.6f}  {readout_err[pq]:>12.6f}  "
          f"{combined_err[pq]:>13.6f}  {flip_hw[vq]:>14.6f}{tag}")
print("=" * 74)

safest   = min(range(N_QUBITS), key=lambda q: combined_err[virt_to_phys[q]])
riskiest = max(range(N_QUBITS), key=lambda q: combined_err[virt_to_phys[q]])
print(f"\n  Safest   qubit → q{safest}  "
      f"IBM combined {combined_err[virt_to_phys[safest]]:.6f},  "
      f"flip {flip_hw[safest]:.6f}")
print(f"  Riskiest qubit → q{riskiest}  "
      f"IBM combined {combined_err[virt_to_phys[riskiest]]:.6f},  "
      f"flip {flip_hw[riskiest]:.6f}")
print()
print("  IBM combined  = predicted error from live calibration data")
print("  Obs flip rate = measured |P_hw(qubit != dominant) - P_ideal(qubit != dominant)|")

In [ ]:
# Block 15 — Full probability distribution table
#             Ideal | Noise Estimator | HW correct | HW buggy

def print_dist_table(probs_ideal, probs_est, probs_hw_cor, probs_hw_bug, nqubits):
    print("\nGrover Circuit — Full Probability Distribution")
    print("=" * 74)
    print(f"  {'State':>6}  {'Ideal':>10}  {'Noise Est.':>12}  "
          f"{'HW correct':>12}  {'HW buggy':>10}  {'Est vs HW':>10}")
    print("  " + "-" * 68)
    for idx in range(2 ** nqubits):
        b     = format(idx, f"0{nqubits}b")
        delta = abs(probs_est[idx] - probs_hw_cor[idx])
        flag  = "  ←" if delta > 0.05 else ""
        print(f"  {b:>6}  {probs_ideal[idx]:>10.4f}  {probs_est[idx]:>12.4f}  "
              f"{probs_hw_cor[idx]:>12.4f}  {probs_hw_bug[idx]:>10.4f}  "
              f"{delta:>10.4f}{flag}")
    print("  " + "-" * 68)
    mad = np.mean([abs(probs_est[i] - probs_hw_cor[i]) for i in range(2**nqubits)])
    print(f"  Mean Absolute Deviation (Est vs HW correct): {mad:.4f}")
    if   mad <= 0.05: label = "GOOD           (MAD ≤ 0.05)"
    elif mad <= 0.10: label = "ACCEPTABLE     (MAD ≤ 0.10)"
    else:             label = "POOR           (MAD > 0.10 — refine the estimator)"
    print(f"  Distribution accuracy  : {label}")
    print()

print_dist_table(probs_ideal_gr, probs_est_cor, probs_hw_cor, probs_hw_bug, N_QUBITS)